**CUADERNO COMPLETO DE IDEALISTA Y FOTOCASA (MERCADO DE COMPRA DE INMUEBLES) LISTO PARA ENTREGAR**

# 0. Librerías y carga de archivos

Antes de empezar con la limpieza propiamente dicha, dejamos preparado en un solo sitio todo lo que hace falta para poder ejecutar el resto del cuaderno de un tirón: todas las librerías que se usan a lo largo de todo el análisis (Idealista, Fotocasa y Armonización), y la carga de los 3 archivos en bruto necesarios. El resto de secciones ya no vuelven a tocar `files.upload()` ni a repetir imports — simplemente continúan con las variables que se cargan aquí.

## 0.1 Librerías e instalación de paquetes

In [ ]:
# Todas las librerías usadas en el cuaderno completo (Idealista + Fotocasa + Armonización)
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import re
import unicodedata
from google.colab import files

# Desactiva la notación científica y muestra 2 decimales (se usa en varias partes del cuaderno)
pd.set_option('display.float_format', '{:.2f}'.format)

# Necesario para las líneas de tendencia de los scatter de Fotocasa (px.scatter con trendline='ols')
!pip install statsmodels


## 0.2 Cargar CSV de Idealista

In [ ]:
print("Sube aquí el CSV original de Idealista (Datos_csv_idealista_1.csv):")
uploaded = files.upload()
archivo_idealista = list(uploaded.keys())[0]

## 0.3 Cargar CSV de Fotocasa

In [ ]:
print("Sube aquí el CSV original de Fotocasa (datos_fotocasa_total.csv):")
uploaded = files.upload()
archivo_fotocasa = list(uploaded.keys())[0]

## 0.4 Cargar CSV de Inside AirBnB

In [ ]:
print("Sube aquí el CSV en bruto de Inside AirBnB (datos_inside_airbnb_limpio.csv):")
uploaded = files.upload()
archivo_inside_airbnb = list(uploaded.keys())[0]

# **IDEALISTA // IDEALISTA**
# **IDEALISTA // IDEALISTA**

# 1. Exploración inicial y primera limpieza — Idealista

## 1.1 Dataset Mejorado Idealista - Intro

### 1.2 Primera exploración del Dataset definitivo de **Idealista**

In [ ]:
# Cargamos el dataset original de Idealista (las librerías ya se cargaron en la Sección 0).
df_idealista_web_scraping = pd.read_csv(archivo_idealista)

In [ ]:
df_idealista_web_scraping.head(6)

Hemos añadido 2 nuevas columnas respecto al dataset original: **Garaje**, que nos permite tener en cuenta esta variable pero sobre todo nos corrige las entradas erroneas de Habitaciones, m2 y planta; y la columna **Ciudad** para diferenciar esta pues hemos combinado todo en un solo dataset.

In [ ]:
# Primer vistazo del df_idealista_web_scraping
print(df_idealista_web_scraping.info())
print(df_idealista_web_scraping.columns)
print(f"\nTamaño del dataset: {len(df_idealista_web_scraping)}")

*Resumen de columnas según las notas del análisis* (31.435 registros, 11 columnas de partida): `zona` (distritos, dato clave), `pagina` (poco relevante), `nombre` (dato clave, de aquí se extraerá el barrio), `precio` (dato clave), `garaje` (relevante), `habitaciones` (dato clave), `metros_cuadrados` (dato clave, 339 nulos a revisar), `planta` (relevante, 1.951 nulos a revisar), `descripcion` (aporta poco de forma estructurada), `link` (identifica cada anuncio y los duplicados) y `ciudad` (diferencia las 3 ciudades).

Podemos comprobar que los datos en cada una de las columnas están bien procesados. No hay números que Python esté entendiendo como objects/strings.
Tenemos bastantes nulos en planta y en m2, lo cual más tarde pasaremos a revisar y si procede a retirar de nuestro dataset.
Antes de proceder con nada más, procedo a revisar las filas (anuncios) **duplicadas** que tengamos para proceder a su retirada. Para ello crearé una columna extrayendo el "**ID anuncio**" de los links.

### 1.3 Evaluar duplicados

In [ ]:
# Extraigo el código dentro de los links para establecer un id de cada anuncio:
df_idealista_web_scraping["id_anuncio"] = df_idealista_web_scraping["link"].str.extract(r'/inmueble/(\d+)/')

In [ ]:
df_idealista_web_scraping.sample(5)

In [ ]:
# Voy a evaluar si hay repetidos en el ID anuncio:
ids_duplicados = df_idealista_web_scraping[df_idealista_web_scraping["id_anuncio"].duplicated()]["id_anuncio"].unique()
df_duplicados = df_idealista_web_scraping[df_idealista_web_scraping["id_anuncio"].isin(ids_duplicados)].sort_values("id_anuncio")

In [ ]:
df_duplicados

In [ ]:
# Comparamos el df_idealista_web_scraping original y los duplicados:
print(f"Tamaño del dataset original: {len(df_idealista_web_scraping)}")
print(f"\nTamaño del dataset de DUPLICADOS: {len(df_duplicados)}")

Primero sacamos las filas que tienen el ID anuncio duplicado en el df_idealista_web_scraping y a su vez definimos un nuevo df_idealista_web_scraping llamado ids_duplicados con los IDs que sean únicos. Así tenemos un df_idealista_web_scraping que contiene esos anuncios/filas duplicados. Para a continuación usar la función ".isin" creando otro df_idealista_web_scraping donde nos meta los anuncios que tienen un ID igual a alguno de los IDs qeu tenemos en nuestro df_idealista_web_scraping de duplicados. Finalmente lo ordenamos para poder tenerlos juntos al visualizarlos.
Es curioso que tenemos casi 1.500 anuncios que están duplicados, lo cual parece obedecer a que se encuentra el mismo anuncio en varias páginas.

Después de analizarlo, **procedemos a retirar las filas/anuncios que estén duplicados**.

In [ ]:
# Dado que hay muchos valores repetidos, voy a retirar esos duplicados con la función df_idealista_web_scraping.drop_duplicates
df_sin_duplicados = df_idealista_web_scraping.drop_duplicates(subset="id_anuncio")

In [ ]:
# Ahora comparamos el df_idealista_web_scraping original,los duplicados y el dataset sin duplicados
print(f"Tamaño del dataset original: {len(df_idealista_web_scraping)}")
print(f"Tamaño del dataset de duplicados: {len(df_duplicados)}")
print(f"Tamaño del dataset SIN DUPLICADOS: {len(df_sin_duplicados)}")
print(f"Diferencia con el original: {len(df_idealista_web_scraping) - len(df_sin_duplicados)}")

### 1.4 Recuento de valores y nulosVoy a hacer un recuento de los valores únicos que tenemos en cada columna y en algunos casos vamos a ver qué valores únicos encontramos. También miramos los valores nulos que encontramos.

In [ ]:
df_idealista_2 = df_sin_duplicados

In [ ]:
# Empezaremos mirando el número de valroes que encotnramos en cada columna:
print(f"Nº de Zonas: {df_idealista_2['zona'].nunique()}")
print(f"Nº de nombres: {df_idealista_2['nombre'].nunique()}")
print(f"Nº de precios: {df_idealista_2['precio'].nunique()}")
print(f"Nº de opciones en Garaje: {df_idealista_2['garaje'].nunique()}")
print(f"Nº de opciones en habitaciones: {df_idealista_2['habitaciones'].nunique()}")
print(f"Nº de opciones en m2: {df_idealista_2['metros_cuadrados'].nunique()}")
print(f"Nº de opciones en Planta: {df_idealista_2['planta'].nunique()}")
print(f"Nº de ID Anuncio: {df_idealista_2['id_anuncio'].nunique()}")
print(f"Nº de Ciudades: {df_idealista_2['ciudad'].nunique()}")

Me preocupan los elevados números de valores únicos en Garaje, Habitaciones y Planta pues podría significar que, al igual que en el primer análisis de Idealista, pueda haber errores.

In [ ]:
# Ahora a ver cuáles son esos valores únicos en las variables clave:
print(f"Opciones en Garaje: {df_idealista_2['garaje'].unique()}")
print(f"Opciones en habitaciones: {df_idealista_2['habitaciones'].unique()}")
print(f"Opciones en Planta: {df_idealista_2['planta'].unique()}")

*Según las notas del análisis*, al revisar en detalle esos valores únicos se observa: en **Garaje**, opciones como «No», «Garaje incluido» y múltiples «Garaje opc. X €» con importes muy dispares (desde 100 € hasta 100.000 €); en **Habitaciones**, valores razonables (1–10) mezclados con valores imposibles (148, 160, 212, 224, 290, 325, 593…); y en **Planta**, cadenas de texto tipo «Planta 7ª exterior con ascensor», «Bajo exterior sin ascensor», «exterior con ascensor», entre NaN y otras variantes.

Ahora ya veo que Garaje tiene muchas opciones porque hay anuncios con el garaje opcional por un precio que va variando. En Planta pasa lo mismo que ya sabía, que hay variables "exterior/interior" y "con/sin ascensor" que aumentan mucho el número de opciones. Finalmente, en **habitaciones**, sí que no parece que valores por encima de 30 o 40 sean reales. Habrá que analizar en profundidad.

Procedemos ahora a **analizar los nulos que teníamos en metros cuadrados y en planta**.

In [ ]:
# Defino un df_idealista_web_scraping para meter los nulos que hay en planta:
df_nulos_planta = df_nulos = df_idealista_2[df_idealista_2['planta'].isnull()]
df_nulos_m2 = df_nulos = df_idealista_2[df_idealista_2['metros_cuadrados'].isnull()]
# Contamos cuántos nulos tenemos:
print(f"Nº de nulos en Planta: {len(df_nulos_planta)}")
print(f"Nº de nulos en m2: {len(df_nulos_m2)}")

In [ ]:
df_nulos_planta.sample(10)

Tras hacer varias visualizaciones de los anuncios donde planta es NaN, parece que son casas, chalets, etc. Puede que aquí no sean errores como tal.

In [ ]:
df_nulos_m2.sample(10)

Sin embargo, cuando coinciden valores NaN en planta y m2 a la vez, sí que aparecen errores porque son todo Estudios donde aparentemente el número de habitaciones está mal recogido y parece que son los m2 en realidad.
Vamos a intentar cambiar el valor en habitaciones por 1 y poner este en m2, valorar y si a lugar agruparlos en una **Categoría Estudio**.

In [ ]:
# CORREGIR ESTUDIOS (NaN en m2 y planta)
# Identificamos las filas que tienen NaN en m2 y planta al mismo tiempo:
estudio_nan = (df_idealista_2['metros_cuadrados'].isna() & df_idealista_2['planta'].isna())

# Primero pasamos el valor de habitaciones (que en realidad son m2) a metros_cuadrados:
df_idealista_2.loc[estudio_nan, 'metros_cuadrados'] = df_idealista_2.loc[estudio_nan, 'habitaciones']

# Luego ponemos un 1 en habitaciones porque realmente es todo un mismo habitáculo:
df_idealista_2.loc[estudio_nan, 'habitaciones'] = 1

print(f"Filas corregidas: {estudio_nan.sum()}")


In [ ]:
df_idealista_2.loc[estudio_nan, ['nombre', 'habitaciones', 'metros_cuadrados', 'planta']].sample(10)

El resultado es que la mayoría tienen valores lógicos, pero encontramos algunas casas en estos 339 anuncios que hemos corregido. **Lo más útil será hacer una diferenciación por categorías de inmuebles.**


## 1.5 Categorías de inmueble

In [ ]:
# Vamos a CLASIFICAR EL TIPO DE INMUEBLE
# Voy a utilizar np.select porque es como un if/elif/else con una lista de condiciones y sus resultados.
# Añadimos la instrucción de que ignore si es mayúscula o minúscula con "case".

condiciones = [
    df_idealista_2['nombre'].str.contains('tico',        case=False, na=False),
    df_idealista_2['nombre'].str.contains('plex',        case=False, na=False),
    df_idealista_2['nombre'].str.contains('Estudio',     case=False, na=False),
    df_idealista_2['nombre'].str.contains('Bajo',        case=False, na=False),
    df_idealista_2['nombre'].str.contains('Casa|Chalet', case=False, na=False),
    df_idealista_2['nombre'].str.contains('Piso',        case=False, na=False),
    df_idealista_2['nombre'].str.contains('Finca',        case=False, na=False),
]
categorias = ['Ático', 'Dúplex', 'Estudio', 'Bajo', 'Casa', 'Piso', 'Finca']

df_idealista_2['tipo_inmueble'] = np.select(condiciones, categorias, default='Otro')


In [ ]:
print(df_idealista_2['tipo_inmueble'].value_counts())

In [ ]:
df_2_Otros = df_idealista_2[(df_idealista_2['tipo_inmueble'] == 'Otro')]
df_2_Otros.sample(3)

Hemos podido añadir la nueva columna con categoría o tipo de inmuebles que encontramos en los nombres de los anuncios. Tendremos que llevar a cabo un análisis en detalle de si esas categorías son correctas ya que tenemos valores un poco extraños como que solo haya 1 Bajo en todo el dataset.

De momento, ya parece que hemos

In [ ]:
# Una vez hecha la clasificación, quito los valores que nos distorsionan el análisis porque al ver los anuncios encontramos que tienen problemas.
# NOTA sobre esta celda: en el cuaderno original se hacía df_idealista_2.drop([16999,5967,5975]), usando el índice de fila.
# Ese índice depende del momento exacto en que se había guardado/recargado el CSV, así que aquí identificamos
# las mismas 3 filas de forma robusta por su id_anuncio (99269702, 87360857, 108919167) — las 3 que
# quedaron clasificadas como tipo 'Otro' y no aportan nada al análisis.
ids_otros_a_retirar = ['99269702', '87360857', '108919167']
df_sin_nulos_categorizado = df_idealista_2[~df_idealista_2['id_anuncio'].isin(ids_otros_a_retirar)].copy()

### 1.6 Recuento sin nulos y con categorías

Finalmente, tras haber quitado los duplicados y luego haber corregido una parte de los nulos, volvemos a hacer un recuento de todos los campos. Y ahora ya hacemos un **recuento de** cuántos **anuncios** de inmuebles tenemos **por cada barrio** en cada una de las tres ciudades

In [ ]:
df_idealista_3 = df_sin_nulos_categorizado

In [ ]:
print(f"Tamaño del dataset original: {len(df_idealista_web_scraping)}")
print(f"Tamaño del dataset tras primera limpieza: {len(df_idealista_3)}")
print(f"Nº de Zonas: {df_idealista_3['zona'].nunique()}")
print(f"Nº de nombres: {df_idealista_3['nombre'].nunique()}")
print(f"Nº de precios: {df_idealista_3['precio'].nunique()}")
print(f"Nº de opciones en Garaje: {df_idealista_3['garaje'].nunique()}")
print(f"Nº de opciones en habitaciones: {df_idealista_3['habitaciones'].nunique()}")
print(f"Nº de opciones en m2: {df_idealista_3['metros_cuadrados'].nunique()}")
print(f"Nº de opciones en Planta: {df_idealista_3['planta'].nunique()}")
print(f"Nº de ID Anuncio: {df_idealista_3['id_anuncio'].nunique()}")
print(f"Nº de Ciudades: {df_idealista_3['ciudad'].nunique()}")


In [ ]:
# Hago el recuento del nº de anuncios/inmuebles por cada DISTRITO, diferenciando por cada ciudad
for ciudad, grupo in df_idealista_3.groupby('ciudad'):
    print(f"\n{'--'*20}")
    print(f"  {ciudad.upper()}")
    print(f"{'--'*20}")
    print(grupo['zona'].value_counts().to_string())

In [ ]:
# Ahora hago el recuento sin diferenciar por ciudad:
print(f"Recuento Nº de zonas: {df_idealista_3['zona'].value_counts().sort_values(ascending=False)}")

Para tenerlo claro hasta ahora, voy a lista aquí los cambios hechos hasta ahora al df_idealista_web_scraping por números:

**df_idealista_web_scraping** → Es el df_idealista_web_scraping original, sin cambios.

**df_idealista_2** → Aquí lo único que he retirado es aquellos anuncios que aparecen duplicados en el df_idealista_web_scraping, diferenciando por el ID_anuncio que he creado a partir dl link.

**df_idealista_3** → Tras analizar los nulos en planta y m2, he establecido la relación de los Estudios y he corregido los m2 de estos. Finalmente, he añadido las categorías o tipos de inmuebles y he quitado los 3 en “Otros” pues no eran relevantes.

Pasamos ahora a ir analizando los valores que nos encontramos, pero antes: guardo csv para facilitar la gestión del NB → df_limpio

### 1.7 Guardar CSV (checkpoint 1)

In [ ]:
# Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_idealista_3.to_csv('datos_idealista_combinado_limpio.csv', index=False)

# 2. Evaluamos habitaciones

## 2.1 Evaluar habitaciones

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
print(f"Filas del df_idealista_3: {len(df_idealista_3)}")

In [ ]:
df_idealista_3.sample(6)

In [ ]:
sns.boxplot(data=df_idealista_3, x='habitaciones')
plt.title("Distribución del número de habitaciones (df_idealista_3)")
plt.show()

In [ ]:
# Ahora hacemos recuento del número de inmuebles por nº habitaciones.
# Usando .to_string() en la fórmula nos printea todos los resultados
print(f"Recuento Nº de habitaciones: {df_idealista_3['habitaciones'].value_counts().sort_index(ascending=True).to_string()}")

*Recuento exacto según las notas del análisis:* 1 hab. → 3.804 anuncios · 2 → 8.302 · 3 → 10.871 · 4 → 4.507 · 5 → 1.397 · 6 → 530 · 7 → 166 · 8 → 81 · 9 → 24 · 10 → 38 · y 396 anuncios por encima de 10 habitaciones (con una cola larga hasta 400, cada valor con muy pocos anuncios).

Voy a revisar los anuncios que están por encima de 10 habitaciones para ver qué casos encontramos. Aunque ya veo que el total de inmuebles **por encima de 10 habitaciones** son 396, por lo que **no representan más del 1.4%** de nuestro dataset.
Especialmente, voy a mirar esos grupos de inmuebles en números de habitaciones concretos: 21, 30, 33, 35, 37, 40, 41, 45, 48, 50, 55, 60 y 65.

In [ ]:
# Vamos a ver si deberíamos incluir los inmuebles con MÁS DE 10 HABITACIONES
df_habitaciones_muchas = df_idealista_3[(df_idealista_3['habitaciones']>10)].sort_values(by='habitaciones')
df_habitaciones_muchas

In [ ]:
# Ahora voy a revisar los inmuebles que tienen un número de habitaciones con agrupaciones de más de 10 (21, 30, 33, etc)
df_habitaciones_cluster = df_idealista_3[df_idealista_3['habitaciones'].isin([21, 30, 33])].sort_values(by='metros_cuadrados')
df_habitaciones_cluster.sample(5)

In [ ]:
# Parece que también puede haver casos por debajo de 21
df_habitaciones_cluster_2 = df_idealista_3[df_idealista_3['habitaciones'].isin([20, 18, 16])].sort_values(by='metros_cuadrados')
df_habitaciones_cluster_2.sample(5)

Al revisar los inmuebles que tienen un número de habitaciones con agrupaciones de más de 10 (21, 30, 33, etc) he descubierto que aquí también **está cogiendo mal el número de habitaciones y metros cuadrados**. Al mirar por debajo de 21 también se observa que se puede estar cometiendo ese error. Lo que me preocupa es que puedo tener esos **errores** en anuncios con menos de 10 habitaciones. Lo que se me ha ocurrido para filtrar estos errores: **evaluar los anuncios con Nº Hab > M2**


### 2.2 Errores Nº Hab > M2

In [ ]:
# Vamos a coger todos los casos en los que Nº Habitaciones > Metros Cuadrados:
df_habitaciones_mayor_m2 = df_idealista_3[((df_idealista_3['habitaciones'] > df_idealista_3['metros_cuadrados']))].sort_values(by='metros_cuadrados')
df_habitaciones_mayor_m2

Cuidado porque parece que en realidad los m2 no es que estén cambiados, es que esos valores son la planta. Parece que aquí ha pasado como con el Garaje en el dataset original (primer WS). El error en este caso podría haber sido que en los anuncios en las páginas de Idealista no tendrían el nº de habitaciones y por ende se han corrido los valores. Es decir, **¡exactamente el problema que hubo con Garaje en el primer análisis!**


In [ ]:
# Vamos a ver en concreto los anuncios que nos son estduios porque puede que haya errores distintos:
df_habitaciones_mayor_m2 = df_idealista_3[((df_idealista_3['habitaciones'] > df_idealista_3['metros_cuadrados']) & (df_idealista_3['tipo_inmueble'] != "Casa"))].sort_values(by='metros_cuadrados')
df_habitaciones_mayor_m2

En las casas/chalets, las habitaciones están bien pero los metros cuadrados son incorrectos porque es el nº de planta (porque no hay plantas, son casas).
Tendrías que eliminar estos casos para no elevar la complejidad de estos problemas. Cogeré las que sean "Casa" y los m2 sean 1 para hacer la eliminación.

In [ ]:
# Cogeré las que sean "Casa" y los m2 sean 1
df_habitaciones_mayor_m2_casa = df_idealista_3[((df_idealista_3['habitaciones'] > df_idealista_3['metros_cuadrados']) & (df_idealista_3['tipo_inmueble'] == "Casa"))].sort_values(by='metros_cuadrados')
df_habitaciones_mayor_m2_casa

In [ ]:
# Voy a eliminar esos casos con .copy() y creo un nuevo df_idealista_web_scraping:
hab_mas_m2_casas = ((df_idealista_3['habitaciones'] > df_idealista_3['metros_cuadrados']) & (df_idealista_3['tipo_inmueble'] == "Casa"))
df_idealista_3 = df_idealista_3[~hab_mas_m2_casas].copy()  # El ~ invierte la máscara: nos quedamos con las que NO cumplen la condición
print(f"Filas eliminadas: {hab_mas_m2_casas.sum()} | Filas restantes: {len(df_idealista_3)}")

Voy a eliminar estos casos porque no se han recogido los datos de M2  y son esenciales para nuestro análisis. El problema es que en Idealista no vienen las plantas y por ello el dato no se extrajo.
Nos quedamos con los casos de otro tipo de inmuebles donde sí que está claro que el nº de habitaciones son los m2 y los m2 son las plantas.

**Filas eliminadas: 44 | Filas restantes: 30.072**


**ERROR IMPORTANTE !!!!!** Al querer filtrar las filas que tienen nº hab = m2, he escrito mal el código y todo el dataset se me ha rescrito por esto:

Comparación (no modifica nada) ✅

hab_igual_m2 = df_idealista_3['habitaciones'] == df_idealista_3['metros_cuadrados']

Asignación (sobreescribe la columna entera) ❌

hab_igual_m2 = df_idealista_3['habitaciones'] = df_idealista_3['metros_cuadrados']

In [ ]:
# CORREGIR Nº HAB > M2 --> Mover habitaciones a m2 y mover m2 a planta de tosos los inmuebles QUE NO SEAN CASAS
# Identificamos las filas en las que Nº Habitaciones > Metros Cuadrados:
hab_mas_m2 = df_idealista_3['habitaciones'] > df_idealista_3['metros_cuadrados']

# Primero pasamos el valor de metros_cuadrados (que en realidad son el nº de planta) a planta... y añadimos "Planta xª":
df_idealista_3.loc[hab_mas_m2, 'planta'] = df_idealista_3.loc[hab_mas_m2, 'metros_cuadrados'].apply(lambda x: f"Planta {int(x)}ª")
# Luego ponemos el valor de habitaciones en metros_cuadrados:
df_idealista_3.loc[hab_mas_m2, 'metros_cuadrados'] = df_idealista_3.loc[hab_mas_m2, 'habitaciones']
# ME HE OLVIDADO ANTES DE ESTE PASO: finalmente, hago lo mismo que hicimos antes de poner un 1 en habitaciones:
df_idealista_3.loc[hab_mas_m2, 'habitaciones'] = 1

print(f"Filas corregidas: {hab_mas_m2.sum()}")


In [ ]:
df_idealista_3[hab_mas_m2].head(10)

### 2.3 ¿Hay duplicados todavía?

In [ ]:
# Columnas clave para detectar duplicados (excluimos link, id y descripcion que varían entre agencias)
cols_duplicado = ['nombre', 'precio', 'habitaciones', 'metros_cuadrados', 'planta', 'ciudad']

# Marcamos las filas duplicadas (keep=False marca TODAS las ocurrencias, no solo la segunda)
mascara_dup = df_idealista_3.duplicated(subset=cols_duplicado, keep=False)

df_duplicados_2 = df_idealista_3[mascara_dup].sort_values(by=cols_duplicado)

print(f"Filas duplicadas encontradas: {len(df_duplicados_2)}")
df_duplicados_2.head(20)

Al revisar la parte de los errores de scrapeo, he descubierto de casualidad que hay filas que tienen distinto id_anuncio (por el link ) pero que realmente están duplicadas. No sé de dónde puede haber salido este error, pero al hacer una revisión más exhaustiva **he encontrado 3.554 duplicados**. Hay poco que analizar aquí, así que directamente los vamos a **retirar** dejando solo 1 de cada.

In [ ]:
# Quedarse solo con la primera ocurrencia de cada duplicado
df_idealista_3 = df_idealista_3.drop_duplicates(subset=cols_duplicado, keep='first').copy()
print(f"Filas tras eliminar duplicados: {len(df_idealista_3)}")

Hasta este punto, en este apartado he hecho las siguientes acciones:
- He evaluado el númerode habitaciones -> he encontrado que la mayoría de anuncios con números de habitaciones altos en relaidad eran errores.
- He identificado estas filas con errores con **Nº habitaciones > m2**.
- De estos errores, los que eran Casas estaban mal scrapeados, por lo que he eliminado 44 filas -> nos quedan 30.072.
- Los que no eran casas, los he corregido poniendo un 1 en habitaciones, el valor en habitaciones lo he mvido a m2 y el valor que había en m2 a planta.
- He detectado que podía haber duplicados todavía, por lo que he vuelto a hacer una **limpieza de duplicados -> nos quedan 27.940.**

### 2.4 Guardar CSV (checkpoint 2)

In [ ]:
# (2do) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_idealista_3.to_csv('datos_idealista_combinado_limpio_2.csv', index=False)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_idealista_4 = df_idealista_3.copy()
print(f"Filas del df_idealista_4: {len(df_idealista_4)}")

## 2.5 Evaluar Habitaciones - 2ª parte

Una vez resueltas las inconsistencias de las habitaciones (mal scrapeadas con m2 y planta), procedo a rehacer el el análisis de los números de habitaciones en el dataset. Y ahora ya pasaré a ver qué outliers tenemos como tal, no siendo errores.

In [ ]:
sns.boxplot(data=df_idealista_4, x='habitaciones')
plt.title("Distribución del número de habitaciones (df_idealista_4)")
plt.show()

In [ ]:
# VUELVO a hacer el recuento del número de inmuebles por nº habitaciones.
print(f"Recuento Nº de habitaciones (2da parte): {df_idealista_4['habitaciones'].value_counts().sort_index(ascending=True).to_string()}")

In [ ]:
# Vamos a ver qué pasa con los inmuebles con MÁS DE 10 HABITACIONES otra vez, pero ahora ya como outliers:
df_habitaciones_outliers = df_idealista_4[(df_idealista_4['habitaciones']>10)].sort_values(by='habitaciones')
df_habitaciones_outliers

He revisado en detalle los anuncios de las filas 6391, 22929, 9677, 12240, 16113, 22736 o 15290 y no tendría sentido incluirlos porque los inmuebles son casos muy particulares de edificios históricos, oficinas, edificios enteros que requieren un tratamiento especial, y en general poco representativos para nuestro análisis o de especial complejidad. Como intuyo que nos van a ensuciar nuestra muestra, procedo a **retirar todos los que están por encima de 12 habitaciones**.

In [ ]:
# Filtro y me quedo con solo los anuncios por debajo de 12 habitaciones.
df_habitaciones_filtrado_12 = df_idealista_4[(df_idealista_4['habitaciones']<13)].sort_values(by='habitaciones')
print(f"Filas del df_habitaciones_filtrado_12: {len(df_habitaciones_filtrado_12)}")

Estoy viendo que convendría meter algún texto en los NaN de planta que tienen las filas porque son casas. Pero antes convendría revisarlos.

In [ ]:
# Vuelvo a mirar los de planta NaN por si acaso y para pasarlos a algo lógico.
df_planta_nulos_2 = df_habitaciones_filtrado_12[(df_habitaciones_filtrado_12['planta'].isnull())].sort_values(by='habitaciones')
df_planta_nulos_2

No parece que haya una manera efectiva de sacar información de otras columnas para rellenar el ***valor NaN de planta***, por lo tanto será ***mejor dejarlo como está***.

PAsamos ahora a hacer un primer vistazo de los estadísticos que encontramos en nuestro actual df_idealista_web_scraping (df_habitaciones_filtrado_12)

In [ ]:
df_habitaciones_filtrado_12.describe()

De momento no creo que saquemos mucha información de los estadísticos de nuestro df_idealista_web_scraping porque hemos limpiado bastante la información pero aún tenemos que entender bien los valores que tenemos y extraer o calcular otros que están en texto (planta, Precio/m2). Sí que podemos comprobar que el nº de habitaciones se acumula en torno a 3 habitaciones (voy a hacer la primera representación gráfica), que puede haber aún valores extraños en m2 (¿mínimo de 5?) o que hay demasiada variavilidad en m2 y precio. De ahí que la clave será calcular más adelante el Precio/m2.

In [ ]:
habitaciones = df_habitaciones_filtrado_12['habitaciones']
plt.figure(figsize=(6,4))
plt.hist(habitaciones, bins=40, edgecolor='black')
plt.title('Nº de habitaciones')
plt.xlabel('Tipo')
plt.ylabel('Número de inmuebles')
plt.show()

### 2.6 Primera Correlación

In [ ]:
columnas_correlacion = ['precio', 'habitaciones', 'metros_cuadrados']
matriz_correlacion = df_habitaciones_filtrado_12[columnas_correlacion].corr()
matriz_correlacion

### 2.7 Guardar CSV (checkpoint 3)

In [ ]:
# (3er) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_habitaciones_filtrado_12.to_csv('datos_idealista_combinado_limpio_3.csv', index=False)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_idealista_5 = df_habitaciones_filtrado_12.copy()
print(f"Filas del df_idealista_5: {len(df_idealista_5)}")

### 2.8 **(Recapitulación)** Hasta ahora, vemos lo siguiente:aquí los cambios hechos hasta ahora al df_idealista_web_scraping por números:

**df_idealista_web_scraping** → Es el df_idealista_web_scraping original, sin cambios.

**df_idealista_2** → Aquí lo único que he retirado es aquellos anuncios que aparecen duplicados en el df_idealista_web_scraping, diferenciando por el ID_anuncio que he creado a partir dl link.

**df_idealista_3** → Tras analizar los nulos en planta y m2, he establecido la relación de los Estudios y he corregido los m2 de estos. Finalmente, he añadido las categorías o tipos de inmuebles y he quitado los 3 en “Otros” pues no eran relevantes.

Pasamos ahora a ir analizando los valores que nos encontramos, pero antes: guardo csv para facilitar la gestión del NB → df_limpio

# 3. Evaluamos el número de planta, ascensor y exterior

Voy a empezar a mirar en detalle la distribución del número de plantas, sabiendo que todavía tengo las descripciones estructuradas pero muy numerosas.
Lo primero que veo es que tenemos ahora 137 opciones en la columna planta (131 al comenzar) porque he metido varias opciones que en los errores de scrapeo estaban en m2.

*Según las notas del análisis*, las combinaciones más frecuentes en la columna `planta` antes de la extracción son: Planta 1ª ext. con ascensor (3.190), Planta 2ª ext. con ascensor (2.661), Planta 3ª ext. con ascensor (2.478), Planta 4ª ext. con ascensor (2.246), Planta 5ª ext. con ascensor (1.524), Bajo ext. con ascensor (1.145), Planta 1ª ext. sin ascensor (1.098), 3ª ext. sin ascensor (733), 7ª ext. con ascensor (725), entre otras — de ahí la necesidad de separar esta columna en 3 variables (nº de planta, ascensor y exterior/interior).

In [ ]:
# Hasta aquí ya tenemos bastante información con la que trabajar.
# Voy a intentar sacar algo de la columna planta ya que hay descripciones comunes
print(f"Nº de planta: {df_idealista_5['planta'].nunique()}")
print(f"Nº de planta: {df_idealista_5['planta'].value_counts().to_string()}")

Como ya vimos, tenemos una estructura de Planta, Exterior/Interior y con/sin Ascensor. Procederemos a extraer y clasificar apropiadamente estas variables en columnas nuevas.

1.- Extraemos el número de planta: Buscamos un dígito (\d+) seguido del símbolo ª y lo metemos en una nueva columna, convirtiéndolo a integer. Si en la columna 'planta' pone 'Bajo', ponemos un 0 en 'num_planta'. Si pone "Entreplanta" lo convertiré en 0.5 para utilizarlo como un grado a la hora de comparar con otras variables.

2.- Extraemos el estado del ascensor: Buscamos literalmente una de las dos opciones. Luego lo convertiré en 0 o 1 para hacer la correlación.

3.- Extraemos el si es exterior o interior: exactamente igual que con la columna ascensor.

In [ ]:
# 1. Extraemos el número (ej: 2ª, 1ª)
df_idealista_5['num_planta'] = df_idealista_5['planta'].str.extract(r'(\d+)ª')
df_idealista_5['num_planta'] = pd.to_numeric(df_idealista_5['num_planta'])
df_idealista_5.loc[df_idealista_5['planta'].str.contains('Bajo', na=False), 'num_planta'] = 0
df_idealista_5.loc[df_idealista_5['planta'].str.contains('Entreplanta', na=False), 'num_planta'] = 0.5
# 2. Extraemos el estado del ascensor
df_idealista_5['ascensor'] = df_idealista_5['planta'].str.extract('(con ascensor|sin ascensor)')
# 3. Extraemos el si es exterior o interior
df_idealista_5['exterior'] = df_idealista_5['planta'].str.extract('(exterior|interior)')

# Comprobamos el resultado
df_idealista_5[['planta', 'num_planta', 'ascensor', 'exterior']].head(10)

In [ ]:
print(f"Compruebo los valores que se han movido a Nº de planta: {df_idealista_5['num_planta'].unique()}")
print(f"Compruebo los valores que se han movido a Ascensor: {df_idealista_5['ascensor'].unique()}")
print(f"Compruebo los valores que se han movido a Exterior: {df_idealista_5['exterior'].unique()}")

In [ ]:
df_planta_nulos_3 = df_idealista_5[(df_idealista_5['num_planta'].isnull() & df_idealista_5['planta'].notnull())].sort_values(by='planta')
df_planta_nulos_3

In [ ]:
df_planta_nulos_3 = df_idealista_5[(df_idealista_5['ascensor'].isnull() & df_idealista_5['planta'].notnull())].sort_values(by='planta')
df_planta_nulos_3

He querido comprobar si hay demasiados nulos, y veo que no me ha podido sacar algunos casos como "planta 1 -".
Además, al revisar estos casos, he econtrado que también hay... ¿sótanos? Una locura. **Me había saltado estas 2 opciones: Semi-sótano y Sótano**

In [ ]:
df_idealista_5.loc[df_idealista_5['planta'].str.contains('Semi-sótano', na=False), 'num_planta'] = -1
# Casi cometo el error de meter Sótano primero, lo cual me podría haber trastocado los Semi-sótanos
df_idealista_5.loc[df_idealista_5['planta'].str.contains('Sótano', na=False), 'num_planta'] = -2
df_idealista_5.loc[df_idealista_5['planta'].str.contains('-1', na=False), 'num_planta'] = -1
df_idealista_5.loc[df_idealista_5['planta'].str.contains('-2', na=False), 'num_planta'] = -2

In [ ]:
# Vuelvo a comprobar los valores en num_planta:
print(f"Compruebo de nuevo los valores que se han movido a Nº de planta: {df_idealista_5['num_planta'].unique()}")

In [ ]:
# Vuelvo a comprobar los nulos en num_planta:
df_planta_nulos_4 = df_idealista_5[(df_idealista_5['num_planta'].isnull() & df_idealista_5['planta'].notnull())].sort_values(by='planta')
df_planta_nulos_4

In [ ]:
# A ver los valores que han quedado ahora en la columna planta
print(f"Compruebo los valores que han quedado ahora en la columna planta: {df_planta_nulos_4['planta'].unique()}")

In [ ]:
print(f"Nº de inmueble por ascensor: {df_idealista_5['ascensor'].value_counts().to_string()}")
print(f"Nº de inmueble por orientación: {df_idealista_5['exterior'].value_counts().to_string()}")
print(f"Recuento Nº de inmueble por Nº planta: {df_idealista_5['num_planta'].nunique()}")
print(f"Nº de inmueble por Nº planta: {df_idealista_5['num_planta'].value_counts().sort_index().to_string()}")

In [ ]:
num_plantas = df_idealista_5['num_planta']
plt.figure(figsize=(20,4))
plt.hist(num_plantas, bins=40, edgecolor='black')
plt.title('Distribución de Nº de planta')
plt.xlabel('Nº de planta')
plt.ylabel('Número de inmuebles')
plt.show()

### 3.1 Guardar CSV (checkpoint 4)

In [ ]:
# (4to) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_idealista_5.to_csv('datos_idealista_combinado_limpio_4.csv', index=False)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_idealista_6 = df_idealista_5.copy()
print(f"Filas del df_idealista_6: {len(df_idealista_6)}")
print(f"Columnas del df_idealista_6: {df_idealista_6.columns}")

### 3.2 Outliers de Nº Planta

Vuelvo a sacar el recuento y a dibujar varios gráficos para analizar qué filas incluiré en caso de que los **outliers** nos estén ensuciando el dataset.

In [ ]:
# Hago el recuento de nuevo
print(f"Nº de inmueble por Nº planta: {df_idealista_6['num_planta'].value_counts().sort_index().to_string()}")

In [ ]:
# De nuevo, dibujo un histogrma para ver la distribución de los inmuebles entre el número de plantas.
num_plantas = df_idealista_6['num_planta']
plt.figure(figsize=(20,4))
plt.hist(num_plantas, bins=30, edgecolor='black')
plt.title('Distribución de Nº de planta')
plt.xlabel('Nº de planta')
plt.ylabel('Número de inmuebles')
plt.show()

In [ ]:
import plotly.express as px
fig = px.box(df_idealista_6, x='num_planta', title='Distribución por Nº de planta')
fig.show()

Ya puedo ver los valores del boxplot con la representación de plotly, pero voy a calcularlos para poder tener claros estos valores.

In [ ]:
#Calculamos cuartiles y límites para detección de outliers
Q1 = df_idealista_6["num_planta"].quantile(0.25)
Q2 = df_idealista_6["num_planta"].quantile(0.50)
Q3 = df_idealista_6["num_planta"].quantile(0.75)
IQR = Q3 - Q1 # Calculo el Rango intrecuartil
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print("Cuartiles del precio:")
print(f"  Q1 (25%): {Q1}º planta")
print(f"  Q2 (50%): {Q2}º planta")
print(f"  Q3 (75%): {Q3}º planta")
print(f"  IQR:      {IQR} plantas de rengo")
print(f"\nLímites para outliers:")
print(f"  Límite inferior: {limite_inferior}º planta")
print(f"  Límite superior: {limite_superior}º planta")

outliers_altos_planta = df_idealista_6[df_idealista_6["num_planta"] > limite_superior]
print(f"\nOutliers por planta altos (> {limite_superior}º planta): {len(outliers_altos_planta)}")


In [ ]:
# represento también con seaborn para trasladar a las notas
plt.figure(figsize=(10, 5))
sns.boxplot(x=df_idealista_6['num_planta'])
plt.title('Distribución por Nº de planta')
plt.xlabel('Nº de planta')
plt.show()

In [ ]:
outliers_altos_planta

En el boxplot veo que por encima del nº de planta 8.5 podríamos considerarlos outliers según el criterio del cálculo del IQR. Sin embargo, viendo que tenemos 825 resultados, habría que afinar un poco más la retirada para no perder valor de muestra.

Voy a revisar aquellos que están por ecnima de 20 y en concreto los 3 por encima de 24.

In [ ]:
# Diría que los únicos casos susceptibles de ser revisados y retirados son aquellos que están por encima de 25:
df_6_muy_altos = df_idealista_6[df_idealista_6['num_planta'] > 24].sort_values(by='num_planta')
df_6_muy_altos

Como ya comentaba, no merece la pena retirar todos los que son outliers según el cáculo del IQR. Especialmente porque, incluso evaluando los más altos de todos, veo que son viviendas de lujo en los pisos superirores de grandes bloques de pisos. Aunque voy a quitar el más alto de todos porque no parece realista.

In [ ]:
# Quito el de planta 30 filtrando:
df_6_filtrado = df_idealista_6[df_idealista_6['num_planta'] != 30].sort_values(by='num_planta')
print(f"Tamaño del df_idealista_6 quitando el piso 30: {len(df_6_filtrado)}")

### 3.3 Segunda Correlación - Variables binariasAhora vamos a repetir el cálculo de la matriz de correlaciones, pero primero voy a añadir columnas binarias (0, 1) para las de Exterior/Interior y Con/Sin ascensor. Así podemos ver si hay relaciones importantes a tener en cuenta.
Creo un nuevo df_idealista_web_scraping con los valores binarios y, si encontramos relación, extenderé el cambio para todo el df_idealista_web_scraping

In [ ]:
# Ahora evaluo si hay algo de correlación entre que un inmueble tenga o no ascenor, garaje y sea exterior con el precio.
df_6_binario = df_6_filtrado[['precio', 'ascensor', 'exterior', 'num_planta', 'garaje', 'habitaciones', 'metros_cuadrados']].copy()
# añado .copy() para que no den errores por ser una copia deldf original
df_6_binario['ascensor_corr'] = np.where(df_6_binario['ascensor'] == 'con ascensor', 1, 0)
df_6_binario['exterior_corr'] = np.where(df_6_binario['exterior'] == 'exterior', 1, 0)
df_6_binario['garaje_corr'] = np.where(df_6_binario['garaje'] == 'Garaje incluido', 1, 0)
df_6_binario

In [ ]:
# Evaluamos la correlación del df_binario
columnas_binarias = ['precio','ascensor_corr', 'exterior_corr', 'num_planta', 'garaje_corr', 'habitaciones', 'metros_cuadrados']
matrix_corr_binaria = df_6_binario[columnas_binarias].corr()
matrix_corr_binaria

**Evaluamos la correlación** entre el precio y las siguientes variables:

Garaje con Precio: **0.26** → correlación moderada. Tener garaje sube el precio de forma apreciable.

Nº planta y Ascensor con Precio: **0.20 y 0.19** → ambas bajas pero consistentes, vale la pena mantenerlas.

Exterior con Precio: **0.05** → prácticamente nula, así que no la tendremos muy en cuenta más tarde.

Por supuesto, la correlación entre m2 y habitaciones con el precio es clara como ya vimos anteriormente


In [ ]:
df_6_filtrado

### 3.4 **(Recapitulación)** Hasta ahora, vemos lo siguiente:aquí los cambios hechos hasta ahora al df_idealista_web_scraping por números:

**df_idealista_web_scraping** → Es el df_idealista_web_scraping original, sin cambios.

**df_idealista_2** → Aquí lo único que he retirado es aquellos anuncios que aparecen duplicados en el df_idealista_web_scraping, diferenciando por el ID_anuncio que he creado a partir dl link.

**df_idealista_3** → Tras analizar los nulos en planta y m2, he establecido la relación de los Estudios y he corregido los m2 de estos. Finalmente, he añadido las categorías o tipos de inmuebles y he quitado los 3 en “Otros” pues no eran relevantes.

Pasamos ahora a ir analizando los valores que nos encontramos, pero antes: guardo csv para facilitar la gestión del NB → df_limpio

# 4. Calculamos Precio/m2 y Precio/hab para evaluar las zonas
-Hasta ahora, parece que solo **m2** tiene relación significativa estadísticamente con el precio (Una **correlación de casi el 0.8**). Si este es el caso, **podríamos sacar una medida de Precio / m2** para que veamos la distribución en la ciudad por las zonas. Pero antes, voy a mirar bien los valores en el precio.
También valoraré hacer una variable tipo **Precio / Habitaciones**

## 4.1 Cálculo inicial de Precio/m2 y Precio/hab, y relación con m²

In [ ]:
# Para poder pintar un gráfico mejor que pueda mostrarme más información facilmente, usaré plotly e instalo "statsmodels"
import plotly.express as px
!pip install statsmodels

In [ ]:
fig = px.scatter(
    df_6_filtrado,
    x='metros_cuadrados',
    y='precio',
    trendline='ols',          # línea de tendencia por regresión lineal
    title='Relación entre m² y Precio',
    labels={'metros_cuadrados': 'Metros cuadrados', 'precio': 'Precio (€)'},
    opacity=0.4,              # transparencia para ver densidad donde se solapan puntos
)

fig.show()

In [ ]:
# Dado que he visto valores apartados de la tendencia, pero también
print(f"1er cuartil del precio - Q1 (25%): {df_6_filtrado['precio'].quantile(0.25)}")
print(f"2do cuartil del precio - Q2 (50%): {df_6_filtrado['precio'].quantile(0.50)}")
print(f"Valor promedio del precio: {df_6_filtrado['precio'].mean()}")
print(f"3er cuartil del precio - Q3 (75%): {df_6_filtrado['precio'].quantile(0.75)}")

print(f"\n1er cuartil de los m2 - Q1 (25%): {df_6_filtrado['metros_cuadrados'].quantile(0.25)}")
print(f"2do cuartil de los m2 - Q2 (50%): {df_6_filtrado['metros_cuadrados'].quantile(0.50)}")
print(f"Valor promedio de los m2: {df_6_filtrado['metros_cuadrados'].mean()}")
print(f"3er cuartil de los m2 - Q3 (75%): {df_6_filtrado['metros_cuadrados'].quantile(0.75)}")


In [ ]:
# Por si acaso voy a revisar si en el precio hay valores que puedan ser outliers, aunque ya he filtrado bastante
plt.figure(figsize=(15,5))
sns.boxplot(data=df_6_filtrado, x='precio')
plt.title("Distribución del precio")
plt.show()

In [ ]:
# Igual pero para metros cuadrados porque ya vi que podía haber valores muy extraños:
plt.figure(figsize=(15,5))
sns.boxplot(data=df_6_filtrado, x='metros_cuadrados')
plt.title("Distribución del metro cuadrado")
plt.show()

In [ ]:
# Pinto otro scatter para las notas:
plt.figure(figsize=(15,5))
plt.scatter(df_6_filtrado['precio'], df_6_filtrado['metros_cuadrados'])
plt.title("Scatter de la relación de Precio y m2")
plt.grid(True)
plt.show()

Calculamos las nuevas variables antes de ir a revisar los casos más extraños.

In [ ]:
# Vamos a calcular directamente las variables Precio/m2 y también Precio/habitaciones para después:
df_6_filtrado['Precio/m2'] = df_6_filtrado['precio'] / df_6_filtrado['metros_cuadrados']
df_6_filtrado['Precio/hab'] = df_6_filtrado['precio'] / df_6_filtrado['habitaciones']
df_6_filtrado

### 4.2 Retiramos errores de Precio y m2

In [ ]:
# Voy a revisar valores extraños que he visto en el scatter para estar seguro que no son errores
df_6_filtrado_m2 = df_6_filtrado[df_6_filtrado['metros_cuadrados'] < 10].sort_values(by='metros_cuadrados')
df_6_filtrado_m2

Después de revisar en detalle varios de los anuncios de inmuebles que tienen menos de 30 m2, he acotado los anuncios erroneos a los que están por debajo de 10m2 --> son directamente errores.
Filtro estos valores y paso a mirar casos similares con solo el precio.

In [ ]:
# Quito los anuncios de los inmuebles con menos de 10m2
df_6_filtrado_m2_corregido = df_6_filtrado[df_6_filtrado['metros_cuadrados'] > 10]
df_6_filtrado_m2_corregido

Ahora voy a hacer lo mismo con el precio solamente, a ver si hay algún error.

In [ ]:
# Voy a revisar valores extraños en precio
df_6_filtrado_precio_bajo = df_6_filtrado_m2_corregido[df_6_filtrado_m2_corregido['precio'] < 100000].sort_values(by='precio')
df_6_filtrado_precio_bajo

Mirando los anuncios por debajo de 50.000€ encuentro pisos ocupados, en ruinas, inmueble en copropiedad sin posesión, con el precio distinto en el anuncio (más alto, y que tiene más sentido), anuncios que ya no existen, etc.
He subido a 80.000€ y 100.000€ para ver dónde podría poner un límite para intentar retirar una parte donde pueda haber muchos problemas.

In [ ]:
# Voy a revisar valores extraños en precio y a la vez en Precio/m2:
df_6_filtrado_precio_bajo_2 = df_6_filtrado_m2_corregido[(df_6_filtrado_m2_corregido['precio'] < 80000) & (df_6_filtrado_m2_corregido['Precio/m2'] < 1500)].sort_values(by='precio')
df_6_filtrado_precio_bajo_2

He encontrado viendo en detalle, que filtrando precio < 80.000€ y Precio/m2 < 1.500€ puedo quitarme parte de los valores más extraños y posibles anuncios descartables del df_idealista_web_scraping (16 anuncios fuera) --> **Nos quedan 27.858**

**TENGO QUE ENCONTRAR UNA FORMA DE FILTRAR LOS INMUEBLES QUE TIENEN "TARAS".**

In [ ]:
# Filtro por los valores que acabo de describir:
df_6_filtrado_m2_precio_corregido = df_6_filtrado_m2_corregido[(df_6_filtrado_m2_corregido['precio'] >= 80000) & (df_6_filtrado_m2_corregido['Precio/m2'] >= 1500)]
df_6_filtrado_m2_precio_corregido

In [ ]:
len(df_6_filtrado_m2_precio_corregido)

He tenido que hacer un filtro más complejo porque me había desaparecido casi 300 filas al meter precio Y Precio/m2, porque faltan los que son precio OR Precio/m2.

In [ ]:
print(f"Nº de filas en el df_6_filtrado_m2_corregido: {len(df_6_filtrado_m2_corregido)}")

# Drop rows where precio < 80000 AND Precio/m2 < 1500
df_6_filtrado_m2_precio_corregido = df_6_filtrado_m2_corregido[~((df_6_filtrado_m2_corregido['precio'] < 80000) & (df_6_filtrado_m2_corregido['Precio/m2'] < 1500))].copy()

print(f"Nº de filas en el nuevo df_idealista_web_scraping filtrado (precio < 80000 AND Precio/m2 < 1500): {len(df_6_filtrado_m2_precio_corregido)}")

In [ ]:
len(df_6_filtrado_m2_precio_corregido)

In [ ]:
# df_idealista_7 = df_6_filtrado_m2_precio_corregido

### 4.3 Guardar CSV (checkpoint 5)

In [ ]:
# (5to) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_6_filtrado_m2_precio_corregido.to_csv('datos_idealista_combinado_limpio_5.csv', index=False)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_idealista_7 = df_6_filtrado_m2_precio_corregido.copy()
print(f"Filas del df_idealista_7: {len(df_idealista_7)}")
print(f"Columnas del df_idealista_7: {df_idealista_7.columns}")

# 5. Precio/m² y Precio/habitaciones: outliers

### 5.1 Analizamos ahora Precio/m2

**Outliers**

Voy a probar a utilizar el método IQR (Rango Intercuartílico) para detectar los valores atípicos (outliers) utilizamos:

Q1 (25%): el 25% de los datos están por debajo;
Q2 (50%): la mediana, valor central;
Q3 (75%): el 75% de los datos están por debajo;

IQR = Q3 - Q1

Los límites para outliers se calculan como:

Límite inferior = Q1 - 1.5 x IQR &
Límite superior = Q3 + 1.5 x IQR

In [ ]:
# Vamos a volver a hacer un boxplot pero para Precio/m2 y así buscar outliers
sns.boxplot(data=df_idealista_7, x='Precio/m2')
plt.title("Distribución del Precio/m2")
plt.show()

A simple vista, tenemos 2 claros outliers que seguramente sean conflictivos. Antes de volver a mirar los que se acumulan cerca de los bordes superiores (Q3 + 1.5*IQR), voy a retirar las filas 11577 y 19316 para a continuación pasar a dibujar el boxplot de nuevo y ver si hay que retirar más

In [ ]:
# NOTA: en el cuaderno original esto se hacía con df_idealista_7.drop([11577,19316]), por índice de fila.
# Ese índice depende de en qué momento se guardó/recargó el CSV, así que en esta versión reproducible
# identificamos los mismos 2 anuncios de forma robusta: son, con diferencia, los 2 valores de Precio/m2
# más altos de todo el dataset (158.333 €/m2 y 146.083 €/m2), id_anuncio 108042893 y 110510266.
ids_precio_m2_top2 = ['108042893', '110510266']
df_7_corregido_altos = df_idealista_7[~df_idealista_7['id_anuncio'].isin(ids_precio_m2_top2)].copy()

In [ ]:
len(df_7_corregido_altos)

In [ ]:
# Vamos a volver a hacer un boxplot de Precio/m2 y seguir buscando outliers
plt.figure(figsize=(15,5))
sns.boxplot(data=df_7_corregido_altos, x='Precio/m2')
plt.title("Distribución del Precio/m2 (sin 2 más altos)")
plt.show()

In [ ]:
#Calculamos cuartiles y límites para detección de outliers
Q1 = df_7_corregido_altos["Precio/m2"].quantile(0.25)
Q2 = df_7_corregido_altos["Precio/m2"].quantile(0.50)
Q3 = df_7_corregido_altos["Precio/m2"].quantile(0.75)
IQR = Q3 - Q1 # Calculo el Rango intrecuartil
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
print(f"Límite superior: {limite_superior}€")
print(f"  Q1 (25%): {Q1} €")
print(f"  Q2 (50%): {Q2} €")
print(f"  Q3 (75%): {Q3} €")
print(f"  IQR:      {IQR} € de rengo")

In [ ]:
df_7_outliers_precio_m2 = df_7_corregido_altos[(df_7_corregido_altos['Precio/m2'] > 13640)].sort_values(by='Precio/m2')
df_7_outliers_precio_m2.tail(10)

Calculando el borde superior (Q3 + 1.5*IQR = 13.640,18€ / m2), nos sale que hay **1.019 filas por encima**. Me parece demasiados valores para simplemente no tener en cuenta, por lo que tendríamos que ***quizás establecer 2 datasets: normal y lujo.***
Esto es algo que habrá que determinar más adelante. Por lo pronto **retiraré solo los que están muy por encima**, en la cola del 2do boxplot que he representado: 22147, 23324, 21994, 22090 y 169 (por **encima de 29.000€ / m2**)

In [ ]:
# NOTA: en el cuaderno original esto se hacía con .drop([22147, 23324, 21994, 22090, 169]), por índice.
# Identificamos estas 5 filas de forma robusta: son los 5 siguientes valores más altos de Precio/m2
# (todos por encima de 29.000 €/m2), con id_anuncio: 107144247, 110931054, 110699533, 111094629 y 107207962.
ids_precio_m2_siguientes5 = ['107144247', '110931054', '110699533', '111094629', '107207962']
df_7_corregido_altos_2 = df_7_corregido_altos[~df_7_corregido_altos['id_anuncio'].isin(ids_precio_m2_siguientes5)].copy()

In [ ]:
len(df_7_corregido_altos_2)

In [ ]:
# Hago un último boxplot de Precio/m2 por si acaso:
plt.figure(figsize=(15,5))
sns.boxplot(data=df_7_corregido_altos_2, x='Precio/m2')
plt.title("Distribución del Precio/m2 (sin 2+5 más altos)")
plt.show()

Ahora ya solo quedan unos outliers que están más próximos entre ellos, y retirarlos supondría eliminar un trozo clave del dataset.
De momento continuo con estos, pero quizás haya que valorar el retirarlos más tarde.

### 5.2 Analizamos ahora Precio/hab

In [ ]:
# Vamos a volver a hacer un boxplot pero para Precio/m2 y así buscar outliers
plt.figure(figsize=(15,5))
sns.boxplot(data=df_7_corregido_altos_2, x='Precio/hab')
plt.title("Distribución del Precio/Habitaciones")
plt.show()

A simple vista, tenemos 2 claros outliers que seguramente sean conflictivos. Antes de volver a mirar los que se acumulan cerca de los bordes superiores (Q3 + 1.5*IQR), voy a retirar las filas 11577 y 19316 para a continuación pasar a dibujar el boxplot de nuevo y ver si hay que retirar más

In [ ]:
#Calculamos cuartiles y límites para detección de outliers
Q1_h = df_7_corregido_altos_2["Precio/hab"].quantile(0.25)
Q2_h = df_7_corregido_altos_2["Precio/hab"].quantile(0.50)
Q3_h = df_7_corregido_altos_2["Precio/hab"].quantile(0.75)
IQR_h = Q3_h - Q1_h # Calculo el Rango intrecuartil
limite_inferior_h = Q1 - 1.5 * IQR_h
limite_superior_h = Q3 + 1.5 * IQR_h

print(f"  Q1 (25%): {Q1_h} €")
print(f"  Q2 (50%): {Q2_h} €")
print(f"  Q3 (75%): {Q3_h} €")
print(f"  IQR:      {IQR_h} € de rengo")
print(f"Límite inferior: {limite_inferior_h}€")
print(f"Límite superior: {limite_superior_h}€")

Para equilibrar, también voy a revisar y si corresponde eliminar los outliers más alejados del borde superior del boxplot.
Viendo los valores del IQR y los cuartiles, voy a ver si estos 3 valores alejados son lógicos.

**IQR del Precio / Habitaciones:**

Q1 (25%): 124.500,0 €

Q2 (50%): 210.000 €

Q3 (75%): 360.000 €

IQR:      235.500 € de rango

Límite superior: 361.027.78 €


In [ ]:
# Revisamos esos valores más alejados del borde superior de 361.027€:
df_7_outliers_hab = df_7_corregido_altos_2[(df_7_corregido_altos_2['Precio/hab'] > 380000)].sort_values(by='Precio/hab')
df_7_outliers_hab.tail(6)

Veo que esos 3 valores tienen saltos importantes respecto al resto de los valores más altos. **Es lógico retirar esos 3** por si acaso.

In [ ]:
# NOTA: en el cuaderno original esto se hacía con .drop([25306,25277,25305]), por índice.
# Identificamos estas 3 filas de forma robusta: son, con diferencia, los 3 valores de Precio/hab
# más altos que quedan (saltos claros respecto al resto), id_anuncio 110681587, 111383075 y 110816282
# — las 3 son 'Casa o chalet independiente en El Viso, Madrid' con 1 sola habitación registrada.
ids_precio_hab_top3 = ['110681587', '111383075', '110816282']
df_7_outliers_hab_corregidos = df_7_corregido_altos_2[~df_7_corregido_altos_2['id_anuncio'].isin(ids_precio_hab_top3)].copy()

In [ ]:
# Vamos a volver a hacer un boxplot de Precio/hab para revisar
plt.figure(figsize=(15,5))
sns.boxplot(data=df_7_outliers_hab_corregidos, x='Precio/hab')
plt.title("Distribución del Precio/Habitaciones")
plt.show()

In [ ]:
len(df_7_outliers_hab_corregidos)

Veo que si sigo haciendo este ejercicio, no voy a terminar de retirar casos como estos y no parecen especialmente sospechosos. Continuaré así.

### 5.3 Tercera CorrelaciónUna vez eliminados varios valores atípicos o distorsionadores, voy a calcular una nueva correlación para comparar con lo que había obtenido hasta ahora.

In [ ]:
# Evaluamos la correlación nuevamente
columnas_limpias = ['precio','habitaciones', 'metros_cuadrados', 'num_planta','Precio/m2', 'Precio/hab']
matrix_corr_limpia = df_7_outliers_hab_corregidos[columnas_limpias].corr()
matrix_corr_limpia

### 5.4 Guardar CSV (checkpoint 6)

In [ ]:
# (6to) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_7_outliers_hab_corregidos.to_csv('datos_idealista_combinado_limpio_6.csv', index=False)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_idealista_8 = df_7_outliers_hab_corregidos.copy()
print(f"Filas del df_idealista_8: {len(df_idealista_8)}")
print(f"Columnas del df_idealista_8: {df_idealista_8.columns}")

In [ ]:
# Para comprobar
# df_idealista_8 = df_7_outliers_hab_corregidos
# len(df_idealista_8)

# 6. Limpieza de casos especiales (Ocupados, Ruinas, etc.)

Como ya vimos anteriormente, podemos tener un número importante de casos donde los inmuebles tengan algún tipo de problema que dificulte su gestión. Tanto la toma de posesión en sí como gastos ocultos que acaban en quiebra de la inversión.

**Casos encontrados en el dataset gracias a Claude AI:**

Alquilado / con inquilino / renta antigua — 802 casos (el más frecuente, son pisos de inversión no disponibles para poner en el mercado del alquiler vacacional)

Para reformar — 577 casos (muchos son "para reformar" normales, no ruinas; pero dificilmente podremos diferenciar)

Piso ocupado — 348 casos (no podemos disponer del inmueble)

Sin posesión / entrega diferida — 113 casos (no podemos disponer del inmueble)

Nuda propiedad — 160 casos (no podemos disponer del inmueble)

Subasta — 69 casos (no podemos disponer del inmueble)

Herencia / procedimiento judicial / embargo — 49 casos (no podemos disponer del inmueble)

Ruinas / derribo — 3 casos

Local reconvertido — 5 casos

In [ ]:
# Primero listamos las palabras clave para identificar los anuncios que vamos a excluir del dataset final:
palabras_clave_excluir = [
    'ocupado', 'ocupada', 'ocupante',
    'nuda propiedad', 'nuda',
    'alquilado', 'arrendado', 'con inquilino', 'renta antigua',
    'subasta',
    'embargo', 'procedimiento judicial',
    'sin posesión', 'posesión diferida', 'entrega diferida',
    'ruina', 'derribo',
    'para reformar', 'reforma total', 'a reformar']

# Definimos el texto para que los busque tanto en el nombre como en la descripción por si acaso solo está en uno de los 2:
texto = df_idealista_8['nombre'].fillna('') + ' ' + df_idealista_8['descripcion'].fillna('')
texto = texto.str.lower()
# Y finalmente el patrón de
patron = '|'.join(palabras_clave_excluir)
variable_excluir = texto.str.contains(patron, regex=True, na=False)


In [ ]:
# HAcemos un df_idealista_web_scraping separado para analizarlo
df_excluidos = df_idealista_8[variable_excluir].copy()
print(f"Casos a excluir: {len(df_excluidos)}")

In [ ]:
df_excluidos.sample(10)

In [ ]:
# Miro las descripciones problemáticas que encontramos:
df_excluidos['descripcion'].unique()

In [ ]:
# ── DATASET LIMPIO ────────────────────────────────────────────
df_idealista_9 = df_idealista_8[~variable_excluir].copy()
print(f"Dataset limpio: {len(df_idealista_9)} filas")

Hemos creado un df_idealista_web_scraping separado para revisar lo que tenemos y, aunque puede que hayamos incluido muchos casos donde aparecen las palabra “reforma” o “reformar” sin que necesite un gasto significativo, es demasiado complejo de diferenciar sin estudiar los anuncios uno por uno. **Así que retiramos todos los casos.**

Casos encontrados → **2.260 filas**

Casos encontrados → **25.588 filas**

# 7. Analizamos distribución de Precio/m2 y estadísticos sin outliers

## 7.1 Distribución del Precio y del Precio/m2

In [ ]:
# Vamos a plotear el precio primero
# Configuramos el estilo
sns.set_theme(style="whitegrid")
# Creamos el histograma con ajuste previo de la escala (A MILES DE €)
plt.figure(figsize=(16, 8))
sns.histplot(df_idealista_9['precio'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios de Venta', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')

plt.show()

In [ ]:
# Y ahora, vamos a plotear el precio/m2 para comparar
sns.set_theme(style="whitegrid")
plt.figure(figsize=(20, 10))
sns.color_palette("mako", as_cmap=True)
sns.histplot(df_idealista_9['Precio/m2'], bins=60, kde=True)
plt.title('Distribución de Precios/m2 de Venta en Barcelona', fontsize=15)
plt.xlabel('Precio por metro cuadrado (en €)')
plt.ylabel('Número de Inmuebles')

plt.show()

In [ ]:
# Imprimo los principales estadísticos de todo el dataset:
print(f"Recuento de nuestro dataset: {df_idealista_9['Precio/m2'].count()}")
print(f"Promedio del Precio/m2 en nuestro dataset: {df_idealista_9['Precio/m2'].mean()}")
print(f"Valor Medio del Precio/m2 en nuestro dataset: {df_idealista_9['Precio/m2'].median()}")
print(f"Valor Modal del Precio/m2 en nuestro dataset: {df_idealista_9['Precio/m2'].mode()}")
print(f"Valor Mínimo del Precio/m2 en nuestro dataset: {df_idealista_9['Precio/m2'].min()}")
print(f"Valor Máximo del Precio/m2 en nuestro dataset: {df_idealista_9['Precio/m2'].max()}")
print(f"Desviación típica del Precio/m2 en nuestro dataset: {df_idealista_9['Precio/m2'].std()}")

In [ ]:
# Sacamos los estadísticos de CADA CIUDAD para tener la información completa:
df_idealista_9.groupby('ciudad')['Precio/m2'].agg(
    count='count',
    mean='mean',
    median='median',
    min='min',
    max='max',
    std='std'
).sort_values('mean', ascending=False).rename(columns={'count':'Recuento',
                                                       'mean':'Promedio P/m2','median':'Mediana P/m2','min':'Min P/m2','max':'Max P/m2','std':'DesvStd P/m2'})

In [ ]:
# Sacamos los estadísticos de CADA DISTRITO para tener la información completa:
df_idealista_9.groupby('zona')['Precio/m2'].agg(
    count='count',
    mean='mean',
    median='median',
    min='min',
    max='max',
    std='std'
).sort_values('mean', ascending=False).rename(columns={'count':'Recuento',
                                                       'mean':'Promedio P/m2','median':'Mediana P/m2','min':'Min P/m2','max':'Max P/m2','std':'DesvStd P/m2'})

In [ ]:
# Con esto veo media y mediana, pero es mejor hacerlo combinado con recuento y otros más abajo.
# df_precio_zonas_media = df_limpio_5.groupby('zona')['Precio/m2'].mean().reset_index().sort_values(by='Precio/m2',ascending=False).rename(columns={'Precio/m2':'Media P/m2'})
# df_precio_zonas_mediana = df_limpio_5.groupby('zona')['Precio/m2'].median().reset_index().sort_values(by='Precio/m2',ascending=False).rename(columns={'Precio/m2':'Mediana P/m2'})
# df_precio_zonas_media
# df_precio_zonas_mediana

In [ ]:
# Como las desviaciones típicas del Precio/m2 son muy altas, quiero graficar la zona con más desviación (eixample) y ver la distribución de precios
# df_eixample = df_final_barcelona[(df_final_barcelona['zona']=='eixample')].sort_values(by='Precio/m2').reset_index()
# df_eixample

### 7.2 Analizamos la excesiva variabilidad
Hemos visto en los estadísticos que, si bien la varibilidad entre zonas es lógica y esperable, tenemos **demasiada desviación típica** en general, en cada ciudad (sobre todo Madrid) y en muchos de los barrios. Vamos a plotear la densidad en cada valor del precio.

In [ ]:
# Calculo la meida y mediana para añadirlas:
mean_precio_m2   = df_idealista_9['Precio/m2'].mean()
median_precio_m2 = df_idealista_9['Precio/m2'].median()

plt.figure(figsize=(16, 6))

# Con histplot y kde=True, Python dibuja el histograma + la curva de densidad encima
sns.histplot(data=df_idealista_9, x='Precio/m2', kde=True, color='steelblue')
# Añado las linea verticales de Media y Mediana:
plt.axvline(mean_precio_m2,   color='red',   linestyle='--', label=f'Media P/m2 ({mean_precio_m2:.2f})')
plt.axvline(median_precio_m2, color='green', linestyle=':',  label=f'Mediana P/m2 ({median_precio_m2:.2f})')
# Titulando el gráfico:
plt.title('Distribución de Precio/m2 con Media y Mediana')
plt.xlabel('Precio por metro cuadrado (€)')
plt.ylabel('Número de inmuebles')
plt.legend()
plt.show()

### 7.3 Guardar CSV (checkpoint 7)

In [ ]:
len(df_idealista_9)

In [ ]:
# (7to) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_idealista_9.to_csv('datos_idealista_combinado_limpio_7.csv', index=False)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_idealista_10 = df_idealista_9.copy()
print(f"Filas del df_idealista_10: {len(df_idealista_10)}")
print(f"Columnas del df_idealista_10: {df_idealista_10.columns}")

# 8. Sacar Barrios del Dataset

### 8.1 Usado en el primer dataset

**2 opciones de extracción:**

Dado que no hay un criterio único para determinar el barrio, habría que ir por pasos:
- Primero: vamos a por los fáciles con df_idealista_web_scraping["barrio"] = df_idealista_web_scraping["nombre"].str.extract(r',\s*([^,]+),\s*Barcelona') para que me saque del texto si tienen una estructura de "... Barrio, Barcelona"
- Segundo: vamos a crear un diccionario de barrio:clave para luego buscar con una función y un bucle

vamos a crear un diccionario de barrio:clave para luego buscar con una función y un bucle

In [ ]:
# Creo un diccionario de barrios con palabras clave para que puedan ser buscadas:
dicc_barrios = {
    "El Gòtic": ["gotic", "gotico", "gòtic", "gótico"],
    "El Raval": ["raval"],
    "La Barceloneta": ["barceloneta"],
    "Sant Pere - Santa Caterina i la Ribera": [
        "sant pere", "santa caterina", "ribera", "born"
    ],
    "La Dreta de l'Eixample": [
        "dreta de l'eixample", "dreta eixample"
    ],
    "L'Antiga Esquerra de l'Eixample": [
        "antiga esquerra de l'eixample", "antiga esquerra"
    ],
    "La Nova Esquerra de l'Eixample": [
        "nova esquerra de l'eixample", "nova esquerra"
    ],
    "La Sagrada Família": [
        "sagrada familia", "sagrada família"
    ],
    "Sant Antoni": ["sant antoni"],
    "El Fort Pienc": ["fort pienc"],
    "Vila de Gràcia": ["vila de gracia", "gracia"],
    "El Camp d'en Grassot i Gràcia Nova": [
        "grassot", "gracia nova"
    ],
    "La Salut": ["salut"],
    "El Coll": ["coll"],
    "Vallcarca i els Penitents": [
        "vallcarca", "penitents"
    ],
    "El Poblenou": ["poblenou"],
    "Diagonal Mar i el Front Marítim del Poblenou": [
        "diagonal mar", "front maritim"
    ],
    "La Vila Olímpica del Poblenou": [
        "vila olimpica"
    ],
    "El Clot": ["clot"],
    "El Camp de l'Arpa del Clot": [
        "camp de l'arpa"
    ],
    "Sant Martí de Provençals": [
        "sant marti de provencals"
    ],
    "La Verneda i la Pau": [
        "verneda", "la pau"
    ],
    "El Besòs": ["besos"],
    "Sants": ["sants"],
    "Sants - Badal": ["badal"],
    "Hostafrancs": ["hostafrancs"],
    "La Bordeta": ["bordeta"],
    "El Poble Sec - Parc de Montjuïc": [
        "poble sec", "montjuic"
    ],
    "La Font de la Guatlla": ["guatlla"],
    "La Marina del Port": ["marina del port"],
    "La Marina del Prat Vermell": ["prat vermell"],
    "Zona Franca - Port": ["zona franca"]
}

### 8.2 Segunda opción de extracción: la buena

In [ ]:
# Diccionario para las 3 CIUDADES:
dicc_barrios = {

    # BARRIOS BARCELONA ───────────────────────────────────────────────────────
    "El Gòtic": ["gotic", "gotico", "gòtic", "gótico"],
    "El Raval": ["raval"],
    "La Barceloneta": ["barceloneta"],
    "Sant Pere - Santa Caterina i la Ribera": ["sant pere", "santa caterina", "ribera", "born"],
    "La Dreta de l'Eixample": ["dreta de l'eixample", "dreta eixample"],
    "L'Antiga Esquerra de l'Eixample": ["antiga esquerra de l'eixample", "antiga esquerra"],
    "La Nova Esquerra de l'Eixample": ["nova esquerra de l'eixample", "nova esquerra"],
    "La Sagrada Família": ["sagrada familia", "sagrada família"],
    "Sant Antoni": ["sant antoni"],
    "El Fort Pienc": ["fort pienc"],
    "Vila de Gràcia": ["vila de gracia", "gracia"],
    "El Camp d'en Grassot i Gràcia Nova": ["grassot", "gracia nova"],
    "La Salut": ["salut"],
    "El Coll": ["coll"],
    "Vallcarca i els Penitents": ["vallcarca", "penitents"],
    "El Poblenou": ["poblenou"],
    "Diagonal Mar i el Front Marítim del Poblenou": ["diagonal mar", "front maritim"],
    "La Vila Olímpica del Poblenou": ["vila olimpica"],
    "El Clot": ["clot"],
    "El Camp de l'Arpa del Clot": ["camp de l'arpa"],
    "Sant Martí de Provençals": ["sant marti de provencals"],
    "La Verneda i la Pau": ["verneda", "la pau"],
    "El Besòs": ["besos"],
    "Sants": ["sants"],
    "Sants - Badal": ["badal"],
    "Hostafrancs": ["hostafrancs"],
    "La Bordeta": ["bordeta"],
    "El Poble Sec - Parc de Montjuïc": ["poble sec", "montjuic"],
    "La Font de la Guatlla": ["guatlla"],
    "La Marina del Port": ["marina del port"],
    "La Marina del Prat Vermell": ["prat vermell"],
    "Zona Franca - Port": ["zona franca"],

    # BARRIOS MADRID ───────────────────────────────────────────────────────
    # Salamanca
    "Recoletos": ["recoletos"],
    "Goya": ["goya"],
    "Fuente del Berro": ["fuente del berro"],
    "Guindalera": ["guindalera"],
    "Lista": ["lista"],
    "Castellana": ["castellana"],
    # Chamberí
    "Almagro": ["almagro"],
    "Trafalgar": ["trafalgar"],
    "Ríos Rosas": ["rios rosas"],
    "Arapiles": ["arapiles"],
    "Gaztambide": ["gaztambide"],
    "Vallehermoso": ["vallehermoso"],
    # Retiro
    "Jerónimos": ["jeronimos", "jerónimos"],
    "Ibiza": ["ibiza"],
    "Niño Jesús": ["nino jesus"],
    "Pacífico": ["pacifico"],
    "Adelfas": ["adelfas"],
    "Estrella": ["estrella"],
    # Chamartín
    "El Viso": ["el viso"],
    "Prosperidad": ["prosperidad"],
    "Ciudad Jardín": ["ciudad jardin"],
    "Hispanoamérica": ["hispanoamerica"],
    "Nueva España": ["nueva españa"],
    "Castilla": ["castilla"],
    # Centro
    "Malasaña - Universidad": ["malasana", "universidad"],
    "Chueca - Justicia": ["chueca", "justicia"],
    "Sol": ["sol", "puerta del sol"],
    "Embajadores": ["embajadores", "lavapies", "lavapiés"],
    "Palacio": ["palacio"],
    "Cortes": ["cortes"],
    # Tetuán
    "Bellas Vistas": ["bellas vistas"],
    "Cuatro Caminos": ["cuatro caminos"],
    "Castillejos": ["castillejos"],
    "Almenara": ["almenara"],
    "Valdeacederas": ["valdeacederas"],
    "Berruguete": ["berruguete"],
    # Arganzuela
    "Imperial": ["imperial"],
    "Acacias": ["acacias"],
    "Chopera": ["chopera"],
    "Legazpi": ["legazpi"],
    "Delicias": ["delicias"],
    "Palos de Moguer": ["palos de moguer"],
    # Fuencarral
    "El Pardo": ["el pardo"],
    "Fuentelarreina": ["fuentelarreina"],
    "Peñagrande": ["penagrande", "peñagrande"],
    "El Pilar": ["el pilar"],
    "La Paz": ["la paz"],
    "Valverde": ["valverde"],
    "Mirasierra": ["mirasierra"],
    "Las Tablas": ["las tablas"],
    # Hortaleza
    "Palomas": ["palomas"],
    "Piovera": ["piovera"],
    "Canillas": ["canillas"],
    "Pinar del Rey": ["pinar del rey"],
    "Apóstol Santiago": ["apostol santiago"],
    "Valdefuentes": ["valdefuentes"],
    # Ciudad Lineal
    "Ventas": ["ventas"],
    "Pueblo Nuevo": ["pueblo nuevo"],
    "Quintana": ["quintana"],
    "Concepción": ["concepcion"],
    "San Pascual": ["san pascual"],
    "San Juan Bautista": ["san juan bautista"],
    "Colina": ["colina"],
    "Atalaya": ["atalaya"],
    "Costillares": ["costillares"],
    # San Blas
    "Simancas": ["simancas"],
    "Hellín": ["hellin"],
    "Amposta": ["amposta"],
    "Arcos": ["arcos"],
    "Rosas": ["rosas"],
    "Rejas": ["rejas"],
    "Canillejas": ["canillejas"],
    "Salvador": ["salvador"],
    # Usera
    "Pradolongo": ["pradolongo"],
    "Moscardó": ["moscardo"],
    "Orcasitas": ["orcasitas"],
    "Pradillo": ["pradillo"],
    "Almendrales": ["almendrales"],
    "Zofío": ["zofio"],
    # Puente de Vallecas
    "Entrevías": ["entrevias"],
    "San Diego": ["san diego"],
    "Palomeras Bajas": ["palomeras bajas"],
    "Palomeras Sureste": ["palomeras sureste"],
    "Portazgo": ["portazgo"],
    "Numancia": ["numancia"],
    # Villaverde
    "San Andrés": ["san andres"],
    "San Cristóbal": ["san cristobal"],
    "Butarque": ["butarque"],
    "Los Rosales": ["los rosales"],
    "Los Ángeles": ["los angeles"],
    # Villa de Vallecas
    "Casco Histórico de Vallecas": ["vallecas", "casco historico de vallecas"],
    "Santa Eugenia": ["santa eugenia"],
    # Vicálvaro
    "Casco Histórico de Vicálvaro": ["vicalvaro"],
    "Valdebernardo": ["valdebernardo"],
    # Moratalaz
    "Pavones": ["pavones"],
    "Horcajo": ["horcajo"],
    "Marroquina": ["marroquina"],
    "Media Legua": ["media legua"],
    "Fontarrón": ["fontarron"],
    "Vinateros": ["vinateros"],
    # Barajas
    "Alameda de Osuna": ["alameda de osuna"],
    "Aeropuerto": ["aeropuerto"],
    "Casco Histórico de Barajas": ["barajas"],
    "Timón": ["timon"],
    "Corralejos": ["corralejos"],
    # Nou Barris (Madrid - Hortaleza zona norte)
    "Hortaleza": ["hortaleza"],

    # BARRIOS VALENCIA ─────────────────────────────────────────────────────
    # Ciutat Vella
    "La Seu": ["la seu"],
    "La Xerea": ["xerea"],
    "El Carmen": ["carmen", "el carme"],
    "El Pilar (Valencia)": ["el pilar valencia"],
    "La Merced": ["merced"],
    "San Francesc": ["san francesc"],
    # L'Eixample (Valencia)
    "Russafa": ["russafa", "ruzafa"],
    "El Pla del Remei": ["pla del remei"],
    "Gran Vía": ["gran via"],
    # Extramurs
    "El Botànic": ["botanic", "botànic"],
    "La Roqueta": ["roqueta"],
    "La Petxina": ["petxina"],
    "Arrancapins": ["arrancapins"],
    # Campanar
    "Campanar": ["campanar"],
    "Les Tendetes": ["tendetes"],
    "El Calvari": ["calvari"],
    "Sant Pau": ["sant pau"],
    # La Saïdia
    "Marxalenes": ["marxalenes"],
    "Morvedre": ["morvedre"],
    "Trinitat": ["trinitat"],
    "Tormos": ["tormos"],
    "Sant Antoni (Valencia)": ["sant antoni valencia"],
    # El Pla del Real
    "Exposició": ["exposicio"],
    "Mestalla": ["mestalla"],
    "Jaume Roig": ["jaume roig"],
    "Creu del Grau": ["creu del grau"],
    # Algirós
    "L'Illa Perduda": ["illa perduda"],
    "City of Arts": ["ciutat de les arts", "city of arts"],
    "L'Amistat": ["amistat"],
    "La Bega Baixa": ["bega baixa"],
    "La Carrasca": ["carrasca"],
    # Benimaclet
    "Benimaclet": ["benimaclet"],
    "Camí de Vera": ["cami de vera"],
    # Rascanya
    "Orriols": ["orriols"],
    "Torrefiel": ["torrefiel"],
    "Sant Llorenç": ["sant llorenc"],
    # Benicalap
    "Benicalap": ["benicalap"],
    "Ciutat Fallera": ["ciutat fallera"],
    # Pobles del Nord
    "Benimamet": ["benimamet"],
    "Beniferri": ["beniferri"],
    # Quatre Carreres
    "Monteolivete": ["monteolivete"],
    "En Corts": ["en corts"],
    "Malilla": ["malilla"],
    "La Fonteta": ["fonteta"],
    "Na Rovella": ["na rovella"],
    "La Punta": ["la punta"],
    # Poblats Marítims
    "El Cabanyal": ["cabanyal"],
    "El Canyamelar": ["canyamelar"],
    "El Grau": ["el grau"],
    "La Marina (Valencia)": ["la marina valencia"],
    "Nazaret": ["nazaret"],
    # Camins al Grau
    "Aiora": ["aiora"],
    "Albors": ["albors"],
    "La Creu del Grau": ["la creu del grau"],
    "Camí Fondo": ["cami fondo"],
    "Penya-roja": ["penya-roja", "penya roja"],
    # Patraix
    "Patraix": ["patraix"],
    "Sant Isidre": ["sant isidre"],
    "Vara de Quart": ["vara de quart"],
    "Safranar": ["safranar"],
    "Favara": ["favara"],
    # Jesús
    "La Raiosa": ["raiosa"],
    "L'Hort de Senabre": ["hort de senabre"],
    "La Creu Coberta": ["creu coberta"],
    "Sant Marcel·lí": ["sant marcelli"],
    "Camí Real": ["cami real"],
    # L'Olivereta
    "Sant Isidre (Olivereta)": ["olivereta"],
    "Nou Moles": ["nou moles"],
    "Soternes": ["soternes"],
    "Tres Forques": ["tres forques"],
    "La Luz": ["la luz"],
    # Pobles del Sud
    "Pinedo": ["pinedo"],
    "El Saler": ["el saler"],
    "El Palmar": ["el palmar"],
    "El Perellonet": ["perellonet"],
    "La Torre": ["la torre"],
    "Faitanar": ["faitanar"],
}

Los solapamientos posibles en el diccionario son estos:

- "clot" aparece en "El Clot" pero "El Camp de l'Arpa del Clot" también contiene la palabra "clot" en el nombre del anuncio.

- "gracia" aparece en "Vila de Gràcia" pero anuncios de "El Camp d'en Grassot i Gràcia Nova" también pueden contener "gracia".

- "sant pere" y "ribera"/"born" podrían solaparse con otros barrios que mencionen esas palabras de pasada.

- "la paz", "rosas", "estrella", "salvador", "arcos" en Madrid son nombres tan genéricos que pueden aparecer en descripciones de calles sin ser el barrio

Vamos ahora a definir las funciones que usaremos para:

- Primero: limpiar el texto que encontramos en "nombre" para quitar tildes y así asegurar un match con la clave del diccionario.

- Segundo: mapear el barrio sacando el texto clave de la columna "nombre" y poder asignar una de las claves del diccionario.

- Finalmente aplicamos la función al df_idealista_web_scraping para crear la nueva columna.

In [ ]:
# Vamos a crear 2 funciones para hacer las 2 acciones necesarias: limpiar el texto que encontramos en "nombre"
# y luego mapear los barrios en base a ese nombre:

import unicodedata
def limpiar_texto(texto):
    if pd.isna(texto):          # Devuelve un string vacío si el nombre es NaN para que la función no falle
        return ""

    texto = texto.lower()       # Pasamos todo el texto objetivo a minúsculas para facilitar el match con el valor en el dicionario

                                # Usamos las funciones unicode data para quitar tildes y no tener que poner 2 opciones (con y sin tildo) en el diccionario.
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)  # Descomponemos las letras con tilde para retirarlas.
        if unicodedata.category(c) != 'Mn')         # Resultado: "Gràcia" → "gracia", "Ríos" → "rios"

    return texto

def mapear_barrio(nombre):
    texto = limpiar_texto(nombre)   # Empezamos limpiando el nombre con la función anterior

    for barrio_canonico, variantes in dicc_barrios.items():     # Recorre el diccionario para sacar el "barrio canónico" -> el nombre oficial del barrio
        for variante in variantes:
            if variante in texto:
                return barrio_canonico                          # Cuando se encuentra una variante del nombre del barrio, nos lo devuelve

    return "Otros"

In [ ]:
# Ya tengo la función que mapea los barrios, ahora la aplico a la columna "nombre" para crear la columna "barrio"
df_idealista_10["barrio"] = df_idealista_10["nombre"].apply(mapear_barrio)

In [ ]:
df_idealista_10["barrio"].value_counts()

In [ ]:
print(f"Lista de barrios: {df_idealista_10['barrio'].nunique()}")
print(f"Lista de barrios: {pd.Series(df_idealista_10['barrio'].value_counts()).to_string()}")

He conseguido sacar 188 barrios de todo el dataset.

Hay un total de **21.436 anuncios** con el barrio **correctamente mapeados**. Y tenemos **4.152** anuncios mapeados como "**otros**".
Habría que revisar esos otros por si acaso se pudieran editar.

In [ ]:
# Voy a revisar los que tienen como barrios "Otros"
df_barrios_otros = df_idealista_10[(df_idealista_10['barrio'] == "Otros")]
df_barrios_otros

In [ ]:
# Descargo el df_idealista_web_scraping con "Otros" en barrios porque veo que sí tenemos el barrio en el nombre, pero me ha podido fallar la clasificación del diccionario
df_barrios_otros.to_csv('datos_idealista_barrios_otros.csv', index=False)

### 8.3 Segunda pasada para encontrar barrios

Voy a volver a pasar código para corregir aquellos barrios que no han sido correctamente capturados con el diccionario y las funciones anteriores.

In [ ]:
# Defino un diccionario con los barrios que faltaban en el primero (revisando todos los que salen en Wikipedia) usando .update():
dicc_barrios.update({

    # ── BARCELONA (barrios que faltaban) ────────────────────
    "Sant Gervasi - Galvany":               ["sant gervasi - galvany", "galvany"],
    "Sant Gervasi - La Bonanova":           ["sant gervasi - la bonanova", "bonanova"],
    "El Putxet i el Farró":                 ["putxet", "farro", "farró"],
    "El Guinardó":                          ["guinardo", "el guinardo"],
    "El Baix Guinardó":                     ["baix guinardo"],
    "Can Baró":                             ["can baro"],
    "Les Roquetes":                         ["roquetes"],
    "Porta":                                ["porta"],
    "Horta":                                ["horta"],
    "Vallvidrera - El Tibidabo i les Planes":["vallvidrera", "tibidabo"],
    "Can Peguera - El Turó de la Peira":    ["can peguera", "turo de la peira"],
    "Les Tres Torres":                      ["les tres torres", "tres torres"],
    "La Font d'En Fargues":                 ["font d'en fargues", "fargues"],
    "La Prosperitat":                       ["prosperitat"],
    "Navas":                                ["navas"],
    "Ciutat Meridiana - Torre Baró - Vallbona": ["ciutat meridiana", "torre baro", "vallbona"],
    "La Teixonera":                         ["teixonera"],
    "La Sagrera":                           ["sagrera"],
    "Verdun":                               ["verdun"],
    "Sant Genís Dels Agudells - Montbau":   ["sant genis", "montbau"],
    "El Congrés i els Indians":             ["congres", "indians"],
    "El Bon Pastor":                        ["bon pastor"],
    "Pedralbes":                            ["pedralbes"],
    "Les Corts (barrio)":                   ["les corts"],
    "La Maternitat i Sant Ramon":           ["maternitat", "sant ramon"],
    "Sant Andreu (barrio)":                 ["sant andreu"],
    "Sarrià":                               ["sarria"],
    "Ciutat Jardí":                         ["ciutat jardi"],

    # ── MADRID (barrios que faltaban) ───────────────────────
    "Villaverde Alto":                      ["villaverde alto"],
    "Palos de la Frontera":                 ["palos de la frontera"],
    "Sanchinarro":                          ["sanchinarro"],
    "Los Ahijones":                         ["ahijones"],
    "Los Berrocales":                       ["berrocales"],
    "Virgen del Cortijo - Manoteras":       ["manoteras", "virgen del cortijo"],
    "El Cañaveral":                         ["cañaveral", "canaveral"],
    "12 de Octubre - Orcasur":              ["12 de octubre", "orcasur"],
    "Los Cerros":                           ["los cerros"],
    "Valdecarros":                          ["valdecarros"],
    "Ambroz":                               ["ambroz"],
    "Montecarmelo":                         ["montecarmelo"],
    "San Fermín":                           ["san fermin"],

    # ── VALENCIA (barrios que faltaban) ─────────────────────
    "Playa de la Malvarrosa":               ["malvarrosa"],
    "Mont-Olivet":                          ["mont-olivet", "montolivet"],
    "El Mercat":                            ["el mercat", "mercat"],
    "Natzaret":                             ["natzaret"],
    "Beteró":                               ["betero"],
    "La Vega Baixa":                        ["vega baixa"],
    "Camí Reial":                           ["cami reial"],
    "El Castellar - L'Oliveral":            ["castellar", "oliveral"],
})

In [ ]:
# Al haber creado ya parte del código, ahora solo tenemos que hacer un update para actualizar la columna "barrio" solo donde tenemos "Otros"
barrios_otros = df_idealista_10["barrio"] == "Otros"
df_idealista_10.loc[barrios_otros, "barrio"] = (df_idealista_10.loc[barrios_otros, "nombre"].apply(mapear_barrio))


In [ ]:
# Comprobación
print(f"Siguen como Otros: {(df_idealista_10['barrio'] == 'Otros').sum()}")
print(f"Lista de barrios: {df_idealista_10['barrio'].nunique()}")
print(f"Lista de barrios: {pd.Series(df_idealista_10['barrio'].value_counts()).to_string()}")

In [ ]:
df_idealista_10.info()

### 8.4 Conseguido - Analizo Barrios

In [ ]:
# Parece que lo hemos conseguido, los barrios extraidos son correctos :)
df_idealista_10.sample(20)

In [ ]:
#Estadísticas del precio/m2 por barrio
df_idealista_10.groupby('barrio')['Precio/m2'].agg(
    count='count',
    mean='mean',
    median='median',
    min='min',
    max='max',
    std='std').sort_values('mean', ascending=False).rename(columns={'count':'Recuento',
    'mean':'Promedio P/m2','median':'Mediana P/m2','min':'Min P/m2',
      'max':'Max P/m2','std':'DesvStd P/m2'}).reset_index().rename(columns={'barrio': 'Barrio'})

In [ ]:
# Saco la TABLA CON € PARA MOSTRAR, pero solo formateando esta tabla pues el € convierte el dato a string.
resultado_barrios = df_idealista_10.groupby('barrio')['Precio/m2'].agg(
    count='count',
    mean='mean',
    median='median',
    min='min',
    max='max',
    std='std').sort_values('mean', ascending=False).rename(columns={'count':'Recuento',
    'mean':'Promedio P/m2','median':'Mediana P/m2','min':'Min P/m2',
      'max':'Max P/m2','std':'DesvStd P/m2'}).reset_index().rename(columns={'barrio': 'Barrio'})
resultado_barrios.style.format({
    'Promedio P/m2': '{:,.0f} €',
    'Mediana P/m2': '{:,.0f} €',
    'Min P/m2': '{:,.0f} €',
    'Max P/m2': '{:,.0f} €',
    'DesvStd P/m2': '{:,.0f} €'
})

In [ ]:
# Acabo con un recuento de Barrios por Zona (distrito). Uso el display.max para que me muestre toda la tabla,
# y después lo elimino para que no me lo haga en todas partes.
# NOTA: en el cuaderno original esta celda usaba 'df_idealista_11', pero en ese punto del notebook original df_idealista_11 todavía
# no existía (se crea más adelante, al recargar df_idealista_10 con otro nombre); aquí usamos directamente df_idealista_10, que
# tiene exactamente el mismo contenido.
recuento_barrios = df_idealista_10.groupby(['ciudad', 'zona', 'barrio']).size().reset_index(name='Recuento').sort_values('Recuento', ascending=False)

pd.set_option('display.max_rows', None)
display(recuento_barrios)
pd.reset_option('display.max_rows')  # volvemos al comportamiento normal después

### 8.5 Guardar CSV (checkpoint 8)

In [ ]:
len(df_idealista_10)

In [ ]:
# (8to) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_idealista_10.to_csv('datos_idealista_limpios_finales.csv', index=False)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_idealista_11 = df_idealista_10.copy()
print(f"Filas del df_idealista_11: {len(df_idealista_11)}")
print(f"Columnas del df_idealista_11: {df_idealista_11.columns}")

In [ ]:
# ¿¿??Quito Página, link (ya tengo el id_anuncio) y Descripción porque no me sirven mucho. Usaré nombre para el barrio
# columnas_a_quitar = ['pagina', 'link', 'descripcion',]
# df_limpio? = df_limpio_2.drop(columnas_a_quitar, axis=1)

# 9. Caso Besós (Top Desviación Típica)

In [ ]:
df_idealista_11

He mantenido el nombre del análisis original que hice solo con Barcelona, pero voy a incluir los barrios con mayor desviación típica: Besós,

In [ ]:
df_desviacion = df_idealista_11[(df_idealista_11['barrio'] == 'El Besòs') | (df_idealista_11['barrio'] == 'Cortes') | (df_idealista_11['barrio'] == 'Jerónimos')]
df_desviacion.sort_values(by='precio', ascending=True)

In [ ]:
df_desviacion.sort_values(by='Precio/m2', ascending=False)

*Nota de las notas del análisis:* además de El Besòs, Cortes y Jerónimos, el barrio **Atalaya** también destaca por su variabilidad (20 anuncios, promedio 5.412 €/m², desviación típica 4.155 €/m²) aunque no se incluyó en la revisión manual original por tener muy pocos anuncios. Se deja constancia aquí para que quede reflejado en la memoria.

In [ ]:
# Hago un histograma del precio de estos barrios para ver la variabilidad
plt.figure(figsize= (20,4))
plt.hist(df_desviacion['precio']/1000,
         edgecolor='black',
         facecolor='red',
         bins=60)
plt.title('Distribución del precio inmuebles en Barrios (alta DT)')
plt.xlabel('Precio en miles de €')
plt.ylabel('Distribución')
plt.show;

In [ ]:
# Igual pero con P/m2 porque la desviación típica extraña es con P/m2
plt.figure(figsize= (20,4))
plt.hist(df_desviacion['Precio/m2'],
         edgecolor='black',
         facecolor='blue',
         bins=60)
plt.title('Distribución del PRECIO/M2 inmuebles en Barrios (alta DT)')
plt.xlabel('Precio en € por m2')
plt.ylabel('Distribución')
plt.show;

In [ ]:
# Voy a hacer un gráfico de regresión para comprobar que está todo bien
precio_miles = df_desviacion['precio']/1000
sns.regplot(x= precio_miles, y='metros_cuadrados', data=df_desviacion)
plt.title('Relación del m2 y Precio con Línea de Tendencia en Barrios (alta DT)')
plt.xlabel("Precio en miles de €", fontsize=12)
plt.ylabel("Metros cuadrados inmueble", fontsize=10)
plt.show()

In [ ]:
# A ver si con el Boxplot los sacamos:
sns.boxplot(data=df_desviacion, y='Precio/m2')
plt.title("Distribución del precio")
plt.show()

Dado que la regresión es lógica y que no hay más que unos pocos outliers en estas zonas solas, es el **dividir el dataset en 3 en función del precio** lo que nos puede racionalizar más las estadísticas.

# 10. Dividir y Guardar el DataFrame final de Idealista

Voy a extraer varios dfs que ya son finales pero con distintos conjuntos de información:

- df_final_limpio -> tenemos todas las filas (25.588) y todas las columnas que han sido limpiadas en varias fases a lo largo de este cuaderno (tanto df_idealista_10 como df_idealista_11 tienen el mismo contenido).
- df_final_limpio_simple -> igual que el anterior pero retirando las columnas que ya no nos aportan ninguna información o que no le vamos a dar ningún uso: ??
- df_estadisticos_precio_m2 -> guardaré la propia tabla con todos los barrios y los estadísticos calculados del Precio / m2.
- df_precio (x3) -> por si queremos mirar la distribución en mejor medida y hacer comparaciones más fiables entre el precio esperado por m2 en un barrio y la rentabilidad esperada en el mismo, he hecho una **división del df_idealista_web_scraping en 3 bandas de precios** para que se ajuste al mercado.

### 10.1 Dividir el df_idealista_web_scraping por precio

In [ ]:
df_idealista_11['precio'].mean()

In [ ]:
df_precio_1 = df_idealista_11[(df_idealista_11['precio'] < 500000)]
df_precio_2 = df_idealista_11[(df_idealista_11['precio'] >= 500000)]

In [ ]:
# Primero grafico la PRIMERA mitad (con las mismas características que antes)
sns.set_theme(style="whitegrid")
# Creamos el histograma con ajuste previo de la escala (A MILES DE €)
plt.figure(figsize=(16, 8))
sns.histplot(df_precio_1['precio'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios de Venta (debajo de 500K€)', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')
plt.show()

In [ ]:
# Y ahora grafico la SEGUNDA mitad (con las mismas características que antes)
sns.set_theme(style="whitegrid")
# Creamos el histograma con ajuste previo de la escala (A MILES DE €)
plt.figure(figsize=(16, 8))
sns.histplot(df_precio_2['precio'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios de Venta (encima de 500K€)', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')
plt.show()

Empiezo **dividiendo el df_idealista_web_scraping en 2 con el valor de la mediana**, pues al graficar la distribución del precio con todos los inmuebles no puedo observar apenas los que están por encima de 2.500.000 €. La **mediana son casi 500.000 €**, mientras que la media son casi 850.000 €. Pruebo a graficar con la mediana y veo que lo que queda por debajo se convierte en una distribución más uniforme, mientras que lo que queda por encima es parecida a la general, donde se agolpan los inmuebles cerca de la mediana. Estimo que **debería hacer la división con los valores del IQR**

- Por debajo del Q1 -> Precio de Entrada.
- Entre el Q1 y el Q3 -> Precio medio.
- Por encima del Q3 -> Precio de Lujo


In [ ]:
# Calculamos los valores del IQR del precio en el df_idealista_11:
Q1_precio = df_idealista_11["precio"].quantile(0.25)
Q2_precio = df_idealista_11["precio"].quantile(0.50)
Q3_precio = df_idealista_11["precio"].quantile(0.75)
IQR_precio = Q3_precio - Q1_precio # Calculo el Rango intrecuartil
limite_inferior_precio = Q1_precio - 1.5 * IQR_precio
limite_superior_precio = Q3_precio + 1.5 * IQR_precio
print(f"Límite inferior: {limite_inferior_precio}€")
print(f"  Q1 (25%): {Q1_precio} €")
print(f"  Q2 (50%): {Q2_precio} €")
print(f"  Q3 (75%): {Q3_precio} €")
print(f"Límite superior: {limite_superior_precio}€")
print(f"  IQR:      {IQR_precio} € de rengo")

**División con los valores del IQR**

- Por debajo del Q2 -> Precio de Entrada (Precio <= 500.000 €).
- Entre el Q2 y el Q3 -> Precio medio (500.000 € < Precio < 950.000 €).
- Por encima del Q3 -> Precio de Lujo (Precio <= 9500.000 €)

In [ ]:
# Filtro el df_idealista_web_scraping por el precio:
df_precio_entrada = df_idealista_11[(df_idealista_11['precio'] <= 500000)]
df_precio_medio = df_idealista_11[((df_idealista_11['precio'] > 500000) & (df_idealista_11['precio'] < 950000))]
df_precio_lujo = df_idealista_11[(df_idealista_11['precio'] >= 950000)]

In [ ]:
print(f" Tamaño del df_precio_entrada: {len(df_precio_entrada)}")
print(f" Tamaño del df_precio_medio: {len(df_precio_medio)}")
print(f" Tamaño del df_precio_lujo: {len(df_precio_lujo)}")
print(f" Tamaño total: {len(df_precio_entrada) + len(df_precio_medio) + len(df_precio_lujo)}")

In [ ]:
# Vamos ahora a representar los 3 gráficos con estas divisiones.
# Empezamos con el Precio entrada:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 8))
sns.histplot(df_precio_entrada['precio'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios de Venta (debajo de 500K€)', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')
plt.show()

In [ ]:
# Seguimos con el Precio medio:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 8))
sns.histplot(df_precio_medio['precio'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios de Venta (encima de 500K€ y debajo de 950K€)', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')
plt.show()

In [ ]:
# Acabamos con el Precio de lujo:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 8))
sns.histplot(df_precio_lujo['precio'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios de Venta (encima de 950K€)', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')
plt.show()

### 10.2 Guardar todos los dfs

Primero guardo los dfs divididos por precio que acabo de crear

In [ ]:
# Guardo los 3 dfs limpios y divididos por precio:
df_precio_entrada.to_csv('datos_idealista_limpios_precio_entrada.csv', index=False)
df_precio_medio.to_csv('datos_idealista_limpios_precio_medio.csv', index=False)
df_precio_lujo.to_csv('datos_idealista_limpios_precio_lujo.csv', index=False)

Guardo ahora el df_idealista_web_scraping con los valores estadístico del Precio/m2 por barrios

In [ ]:
#Estadísticas del precio/m2 por barrio para guardar
df_estadisticos_precio_m2 = df_idealista_11.groupby('barrio')['Precio/m2'].agg(
    count='count',
    mean='mean',
    median='median',
    min='min',
    max='max',
    std='std').sort_values('mean', ascending=False).rename(columns={'count':'Recuento',
    'mean':'Promedio P/m2','median':'Mediana P/m2','min':'Min P/m2',
      'max':'Max P/m2','std':'DesvStd P/m2'}).reset_index().rename(columns={'barrio': 'Barrio'})

In [ ]:
df_estadisticos_precio_m2

In [ ]:
# Guardamos ahora el df_idealista_web_scraping con los estadísticos de Precio/m2 por barrios
df_estadisticos_precio_m2.to_csv('datos_idealista_limpios_estadisticos_precio_m2.csv', index=False)

Finalmente, guardamos ahora el df_idealista_web_scraping limpio y con las columnas inutiles eliminadas

In [ ]:
# Voy a quitar 'pagina', 'planta', 'descripcion', 'link'
columnas_a_quitar = ['pagina', 'planta', 'descripcion', 'link']
df_idealista_limpio_columnas_simple = df_idealista_11.drop(columnas_a_quitar, axis=1)

In [ ]:
# Guardamos ahora el df_idealista_web_scraping limpio y con las columnas inutiles eliminadas:
df_idealista_limpio_columnas_simple.to_csv('datos_idealista_limpios_finales_simple (check final Claude).csv', index=False)

# **FOTOCASA // FOTOCASA**
# **FOTOCASA // FOTOCASA**

# 11. Exploración inicial y primera limpieza — Fotocasa

#Dataset FOTOCASA - Intro


## 11.2 Primera exploración del Dataset de **Fotocasa**

Mi preferencia de trabajo para añadir datos al cuaderno es usar la opción de ***files.upload()*** porque así no tengo que estar conectando con Google Drive todo el rato. Mis compañeros utilizan otra opción, así que habrá que adaptarlo antes de entregarlo. También utilizo esta opción con df_fotocasa_web_scraping.to_csv() para ir guardando mi progreso (descargo y luego vuelvo a importar en cada sección de "Guardar CSV") y así no tener que ejecutar todo el cuaderno.

In [ ]:
# Cargamos el dataset original de Fotocasa (las librerías ya se cargaron en la Sección 0).
df_fotocasa_web_scraping = pd.read_csv(archivo_fotocasa)

**Control de archivos:**

Aquí estoy utilizando el csv "***datos_fotocasa_total.csv***" que tiene la misma información que el archivo en nuestro Drive llamado "*Datos csv_fotocasa_con_duplicados.csv*".

Haré aquí la retirada de duplicados para que quede todo bien registrado.

In [ ]:
df_fotocasa_web_scraping.head(10)

*Resumen de columnas según las notas del análisis* (24.826 registros, 13 columnas de partida; la columna `descripcion` viene completamente vacía y no se usa): `precio` (dato clave), `publicacion` (antigüedad del anuncio), `nombre` (no hace falta sacar el barrio de aquí, pero sí el tipo de inmueble), `link` (incluye un ID útil para detectar duplicados), `barrio` (dato clave), `habitaciones` y `baños` (datos clave de tamaño/capacidad), `metros` (dato clave), `planta` (útil para calidad), `extras` (también útil para calidad, aunque hay pocos disponibles) y `detalles_crudo` (sirve para contrastar con el resto).

A priori parece que el dataset de **Fotocasa frente al de Idealista**, nos aporta mucha **más información** de cada anuncio de inmuebles para poder evaluar sus detalles y calidad. Tenemos ya el barrio bien indicado, aunque aquí lo que nos faltará es tener el distrito (lo añadiremos); pero además tenemos baños e incluso algunos extras como Terraza, Balcón, Aire Acondicionado, etc.

## 11.3 Evaluar duplicados

In [ ]:
# Reviso ejemplos de cómo están escritos los IDs dentro de los links
df_fotocasa_web_scraping['link'].loc[12432]

Algunos ejemplos del ID dentro del link:
1.- https://www.fotocasa.es/es/comprar/vivienda/barcelona-capital/aire-acondicionado-calefaccion/189129808/d
2.- https://www.fotocasa.es/es/comprar/vivienda/barcelona-capital/sant-pere,-sta-caterina-i-la-ribera/188141464/d
3.- https://www.fotocasa.es/es/comprar/vivienda/madrid-capital/aire-acondicionado-parking-terraza-trastero-ascensor-piscina-no-amueblado/185559371/d

Parece que el criterio es **... / ID / d**

*(En el cuaderno original, aquí había un primer intento de extraer el id_anuncio que no funcionaba del todo bien —quedaban nulos—, así que se hacía un segundo intento con otra expresión regular y se combinaban ambos. En esta versión vamos directos con la expresión regular que sí funciona para todos los casos.)*

In [ ]:
# Extraigo el id_anuncio directamente con las expresiones regulares que funcionan, sin el paso
# intermedio de crear una columna fallida y luego hacer drop+rename:
# 1) La mayoría de links siguen el patrón .../ID/d
df_fotocasa_web_scraping["id_anuncio"] = df_fotocasa_web_scraping["link"].str.extract(r'/(\d+)/d').astype(object)
# 2) Un grupo más pequeño de links no sigue exactamente ese patrón; para esos, cogemos el último
#    número de 8 o 9 dígitos al final de la URL, ignorando el '/d' si lo hay:
nulos_id = df_fotocasa_web_scraping['id_anuncio'].isnull()
extra = df_fotocasa_web_scraping.loc[nulos_id, 'link'].str.extract(r'(\d{8,9})(?:/d)?$')[0]
df_fotocasa_web_scraping.loc[nulos_id, 'id_anuncio'] = extra.astype(object)
df_fotocasa_web_scraping['id_anuncio'] = pd.to_numeric(df_fotocasa_web_scraping['id_anuncio'], errors='coerce')
print(f"Nulos en id_anuncio: {df_fotocasa_web_scraping['id_anuncio'].isnull().sum()}")

In [ ]:
# Voy a evaluar si hay repetidos en el ID anuncio:
ids_duplicados = df_fotocasa_web_scraping[df_fotocasa_web_scraping["id_anuncio"].duplicated()]["id_anuncio"].unique()
df_duplicados = df_fotocasa_web_scraping[df_fotocasa_web_scraping["id_anuncio"].isin(ids_duplicados)].sort_values("id_anuncio")

In [ ]:
df_duplicados

Definimos un nuevo df_fotocasa_web_scraping llamado ids_duplicados con los IDs que sean únicos. Así tenemos un df_fotocasa_web_scraping que contiene esos anuncios/filas duplicados. Para a continuación usar la función ".isin" creando otro df_fotocasa_web_scraping donde nos meta los anuncios que tienen un ID igual a alguno de los IDs qeu tenemos en nuestro df_fotocasa_web_scraping de duplicados. Finalmente lo ordenamos para poder tenerlos juntos al visualizarlos.
Es curioso que tenemos más de 12.600 anuncios que están duplicados, lo cual parece obedecer a que se encuentra el mismo anuncio en varias páginas.

Después de analizarlo, **procedemos a retirar las filas/anuncios que estén duplicados**.

In [ ]:
# Comparamos el df_fotocasa_web_scraping original y los duplicados:
print(f"Tamaño del dataset original: {len(df_fotocasa_web_scraping)}")
print(f"\nTamaño del dataset de DUPLICADOS: {len(df_duplicados)}")

In [ ]:
# Dado que hay muchos valores repetidos, voy a retirar esos duplicados con la función df_fotocasa_web_scraping.drop_duplicates
df_sin_duplicados = df_fotocasa_web_scraping.drop_duplicates(subset="id_anuncio")

In [ ]:
# Ahora comparamos el df_fotocasa_web_scraping original,los duplicados y el dataset sin duplicados
print(f"Tamaño del dataset original: {len(df_fotocasa_web_scraping)}")
print(f"Tamaño del dataset de duplicados: {len(df_duplicados)}")
print(f"Tamaño del dataset SIN DUPLICADOS: {len(df_sin_duplicados)}")
print(f"Diferencia con el original: {len(df_fotocasa_web_scraping) - len(df_sin_duplicados)}")

In [ ]:
df_sin_duplicados.to_csv('df_sin_duplicados_fotocasa_google_colab.csv', index=False)

Una vez quitados los **duplicados** usando el "link" para diferenciarlos, nos **quedan 17.291 filas**.

## 11.4 Recuento de valores

Voy a hacer un recuento de los valores únicos que tenemos en cada columna y en algunos casos vamos a ver qué valores únicos encontramos. También miramos los valores nulos que encontramos.

In [ ]:
df_fotocasa_2 = df_sin_duplicados
df_fotocasa_2

In [ ]:
df_fotocasa_2.info()

In [ ]:
# Empezaremos mirando el número de valroes que encotnramos en cada columna:
print(f"Nº de Opciones de Precios: {df_fotocasa_2['precio'].nunique()}")
print(f"Nº de Fechas de publicación: {df_fotocasa_2['publicacion'].nunique()}")
print(f"Nº de Nombres: {df_fotocasa_2['nombre'].nunique()}")
print(f"Nº de barrios: {df_fotocasa_2['barrio'].nunique()}")
print(f"Nº de Opciones de  habitaciones: {df_fotocasa_2['habitaciones'].nunique()}")
print(f"Nº de Opciones de  Baños: {df_fotocasa_2['banos'].nunique()}")
print(f"Nº de Opciones de m2: {df_fotocasa_2['metros'].nunique()}")
print(f"Nº de Opciones de Planta: {df_fotocasa_2['planta'].nunique()}")
print(f"Nº de Opciones de Extras: {df_fotocasa_2['extras'].nunique()}")
print(f"Nº de Opciones de Detalles en crudo: {df_fotocasa_2['detalles_crudo'].nunique()}")
print(f"Nº de Ciudades: {df_fotocasa_2['ciudad'].nunique()}")
print(f"Nº de ID Anuncio: {df_fotocasa_2['id_anuncio'].nunique()}")


Vemos que hay un par de variables que parecen más simplificadas que en Idealista: habitaciones, baños tienen pocos valores únicos, por lo que puede que no haya descuadres entre las columnas. Sin embargo, podría ser el caso en la columna de metros cuadrados por haber casi 1000 valores distintos. Pero **puede que** no y en general **este dataset extraído de Fotocasa no tenga tantos errores** de datos mal extraídos en distintas columnas **como tuvimos en Idealista**.
El número de nombres es casi la mitad que el tamaño del dataset, pero eso puede ser simplemente que haya algunos repetidos por tener descripciones tan simples como “Piso en el Raval, Barcelona Capital”.
El resto parece correcto


In [ ]:
# Ahora a ver cuáles son esos valores únicos en las variables clave mencionadas:
print(f"Opciones de  habitaciones: {df_fotocasa_2['habitaciones'].unique()}")
print(f"Opciones de  Baños: {df_fotocasa_2['banos'].unique()}")
# Quito m2 porque hay muchos y parece que están bien - print(f"Opciones de m2: {df_fotocasa_2['metros'].unique()}")
print(f"Opciones de Planta: {df_fotocasa_2['planta'].unique()}")

Vemos que la variabilidad en habitaciones y baños se debe a que no solo hay números únicos (2 habs., 1 baño) sin oque también hay valores combinados: 2-3 habs. o 4-5 baños.
Vamos a tener que ordenar bien estos valores para poder trabajar con ellos, y seguramente haya que estimar un valor para substituir esos que están combinados.


Procedemos ahora a **analizar los nulos que tenemos en varias variables**.

## 11.5 Evaluamos nulos

In [ ]:
# Defino un df_fotocasa_web_scraping para meter los nulos:
df_nulos_habitaciones = df_fotocasa_2[df_fotocasa_2['habitaciones'].isnull()]
df_nulos_banos = df_fotocasa_2[df_fotocasa_2['banos'].isnull()]
df_nulos_m2 = df_fotocasa_2[df_fotocasa_2['metros'].isnull()]
df_nulos_plantas = df_fotocasa_2[df_fotocasa_2['planta'].isnull()]
# Contamos cuántos nulos tenemos:
print(f"Nº de nulos en Habitaciones: {len(df_nulos_habitaciones)}")
print(f"Nº de nulos en Baños: {len(df_nulos_banos)}")
print(f"Nº de nulos en m2: {len(df_nulos_m2)}")
print(f"Nº de nulos en Planta: {len(df_nulos_plantas)}")

In [ ]:
df_nulos_habitaciones.sample(10)

En **habitaciones**, no encontramos ningún patrón como veíamos en Idealista porque eran Estudios y no tienen habitaciones (o errores).

Los dejamos de momento porque podríamos estimar el número de habitaciones con otras variables como m2 o detalles

In [ ]:
df_nulos_banos.sample(10)

En baños, no podemos evaluar facilmente si son casos normales o tenemos que entender algo en particular como pasa con habitaciones. De momento los dejo pero **seguramente retire aquellos anuncios que tienen NaN en habitaciones, baños y planta**.

In [ ]:
df_nulos_m2.sample(10)

Aquí pasa un poco lo mismo, que tenemos otros datos pero justo faltan los m2. Aquí habrá que eliminar estos 106 porque sería muy complejo estimar para solo un número pequeño de errores.

In [ ]:
df_nulos_plantas.sample(10)

Al igual que pasaba con habitaciones, no hay un patrón observable para que tenga sentido que falten las plantas (casas, chalets). Así que **tendremos que simplemente dejar los valores** porque en nuestro modelo general, no tendremos muy en cuenta la planta en la que se encuentra.

Los que sí voy a **retirar**, para no añadir demasiada complejidad, es aquellas filas que **no tienen ni habitaciones ni baños**, y luego aparte los **nulos de m2**.

In [ ]:
# Retiramos los que tienen NaN en habitaciones y en baños a la vez, y los que son nulos en m2:
nulos_hab_baño = (df_fotocasa_2['habitaciones'].isna() & df_fotocasa_2['banos'].isna())
nulos_m2 = df_fotocasa_2['metros'].isna()
# Combinamos en otra variable ambas condiciones con un | a modo de OR
nulos_elimiar = nulos_hab_baño | nulos_m2
# En vez de usar drop, parece que es mejor copiar en un df_fotocasa_web_scraping nuevo (.copy()) aquellos varoles que NO cumplan la condición, con ~
df_sin_nulos = df_fotocasa_2[~nulos_elimiar].copy()

In [ ]:
print(f"Tamaño del dataset sin duplicados: {len(df_fotocasa_2)}")
print(f"Tamaño del dataset SIN NULOS: {len(df_sin_nulos)}")
print(f"Diferencia: {len(df_fotocasa_2) - len(df_sin_nulos)}")

## 11.6 Retirar Duplicados (2ª pasada)

Una vez retirados los nulos, voy a **volver a hacer una revisión de los duplicados** pero usando otro método en el qeu python busca directamente si hay filas en las que hay varias columnas que coinciden con otras filas.


In [ ]:
# Segunda pasada: eliminar los que comparten precio+publicacion+nombre+barrio+metros
# por si algún anuncio se scrapeó con distintos links porque se repiten entre las páginas o algún error parecido
df_limpio_duplicados_2 = df_sin_nulos[df_sin_nulos.duplicated(subset=['precio', 'publicacion', 'nombre', 'barrio', 'metros'], keep=False)].copy().sort_values(by='nombre')
df_limpio_duplicados_2

In [ ]:
# Los retiramos
df_limpio_nulos_duplicados = df_sin_nulos.drop_duplicates(
    subset=['precio', 'publicacion', 'nombre', 'barrio', 'metros'],
    keep='first'
).copy()

print(f"Antes: {len(df_sin_nulos)} | Después: {len(df_limpio_nulos_duplicados)} | Retirados: {len(df_sin_nulos) - len(df_limpio_nulos_duplicados)}")

Para tenerlo claro hasta ahora, voy a lista aquí los cambios hechos hasta ahora al df_fotocasa_web_scraping por números:

- **df_fotocasa_web_scraping** → Es el df_fotocasa_web_scraping original, sin cambios.
- **df_fotocasa_2** → Aquí lo único que he retirado es aquellos anuncios que aparecen duplicados en el df_fotocasa_web_scraping, diferenciando por el ID_anuncio que he creado a partir dl link.
- **df_fotocasa_3** → Tras analizar los nulos en habitaciones, baños, planta y m2, he decidido prescindir de aquellos que eran a la vez nulos en habitaciones y en baños; y luego los que eran nulos en m2. Y además he hecho otra pasado de retirar duplicados usando otro método (valores duplicados en varias columnas a la vez)

Pasamos ahora a ir analizando los valores que nos encontramos, pero antes: guardo csv para facilitar la gestión del NB → **datos_fotocasa_limpio_1.csv**

## 11.7 Guardar CSV (checkpoint 1)

In [ ]:
len(df_limpio_nulos_duplicados)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_fotocasa_3 = df_limpio_nulos_duplicados.copy()
print(f"Filas del df_fotocasa_3: {len(df_fotocasa_3)}")

## 11.8 Categorizar los inmuebles

Igual que hicimos en Idealista, para consolidar datasets, vamos a establecer una categoría de inmuebles con lo sdistintos tipos que encontremos: Piso, Casa,Estudio, etc.

In [ ]:
print(df_fotocasa_3['nombre'].unique())

Observamos que hay estas 10 categorías:

- Piso
- Apartamento
- Ático
- Dúplex
- Estudio
- Loft
- Planta baja
- Casa o chalet
- Casa adosada
- Finca rústica

In [ ]:
# Vamos a CLASIFICAR EL TIPO DE INMUEBLE
# Voy a utilizar np.select porque es como un if/elif/else con una lista de condiciones y sus resultados si se cunmplen.
# Añadimos la instrucción de que ignore si es mayúscula o minúscula con "case" y si hay nulos con na=False.

condiciones = [
    df_fotocasa_3['nombre'].str.contains('tico',        case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('partamento',        case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('plex',        case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('Estudio',     case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('Loft',        case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('Planta baja',        case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('Casa|Chalet', case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('Casa adosada', case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('Piso',        case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('Finca',        case=False, na=False),
    df_fotocasa_3['nombre'].str.contains('Obra nueva',        case=False, na=False),
]
categorias = ['Ático', 'Apartamento', 'Dúplex', 'Estudio', 'Loft', 'Planta Baja', 'Casa','Casa adosada', 'Piso', 'Finca', 'Obra nueva']

df_fotocasa_3['tipo_inmueble'] = np.select(condiciones, categorias, default='Otro')


In [ ]:
print(df_fotocasa_3['tipo_inmueble'].value_counts())

In [ ]:
# NOTA: en el cuaderno original esta celda decía 'Otros' (con 's'), pero la categoría real que
# crea np.select es 'Otro' (sin 's'). Además, comprobamos que a estas alturas 'Otro' ya está
# vacío: como 'Obra nueva' quedó definida como categoría propia, no queda ningún anuncio sin
# clasificar. Lo mostramos de forma seguro (sin que .sample() falle si no hay filas):
df_3_Otros = df_fotocasa_3[(df_fotocasa_3['tipo_inmueble'] == 'Otro')]
print(f"Anuncios en 'Otro': {len(df_3_Otros)}")
df_3_Otros.sample(5) if len(df_3_Otros) > 0 else df_3_Otros

Hemos podido añadir la nueva columna con categoría o tipo de inmuebles que encontramos en los nombres de los anuncios con mucho éxito. Hemos clasificados casi todos loa anuncios y los que están con **"Otros" resultan ser "obra nueva"**, porque parece que los anunciantes ponen eso sin indicar qué tipo de inmueble es. De momento los pondremos en otra categoría, pero seguramente los tendremos que desechar más tarde. Vuelvo atras para corregir esto.

Parece que es lógico el recuento viendo cuantos anuncios tenemos en cada categoría.

**Retiro Fincas rústicas** porque no nos vale en nuestro proyecto.

In [ ]:
df_3_fincas = df_fotocasa_3[(df_fotocasa_3['tipo_inmueble'] == 'Finca')]
df_3_fincas.sample(5)

In [ ]:
# Retiramos Fincas:
df_3_fincas = df_fotocasa_3['tipo_inmueble'] == 'Finca'
df_3_categorizado = df_fotocasa_3[~df_3_fincas].copy()
print(f"Tamaño del dataset hasta ahora: {len(df_fotocasa_3)}")
print(f"Tamaño del dataset SIN FINCAS: {len(df_3_categorizado)}")

## 11.9 2º Recuento - sin nulos, duplicados y con categorías

Finalmente, tras haber quitado los duplicados, haber corregido una parte de los nulos y haber categorizado los pisos, volvemos a hacer un recuento de todos los campos.

In [ ]:
# Recuento del número de valroes que encotnramos en cada columna:
print(f"Nº de Opciones de Precios: {df_3_categorizado['precio'].nunique()}")
print(f"Nº de Fechas de publicación: {df_3_categorizado['publicacion'].nunique()}")
print(f"Nº de Nombres: {df_3_categorizado['nombre'].nunique()}")
print(f"Nº de barrios: {df_3_categorizado['barrio'].nunique()}")
print(f"Nº de Opciones de  habitaciones: {df_3_categorizado['habitaciones'].nunique()}")
print(f"Nº de Opciones de  Baños: {df_3_categorizado['banos'].nunique()}")
print(f"Nº de Opciones de m2: {df_3_categorizado['metros'].nunique()}")
print(f"Nº de Opciones de Planta: {df_3_categorizado['planta'].nunique()}")
print(f"Nº de Opciones de Extras: {df_3_categorizado['extras'].nunique()}")
print(f"Nº de Opciones de Detalles en crudo: {df_3_categorizado['detalles_crudo'].nunique()}")
print(f"Nº de Ciudades: {df_3_categorizado['ciudad'].nunique()}")
print(f"Nº de ID Anuncio: {df_3_categorizado['id_anuncio'].nunique()}")
print(f"Nº de tipos de inmuebles: {df_3_categorizado['tipo_inmueble'].nunique()}")

Vamos a pasar ahora a ir transformando los valores que tenemos en columnas clave en números y extrayendo la información que estos tienen

# 12. Evaluar Habitaciones y Baños

Usaré esto para llevar recuento

In [ ]:
df_fotocasa_4 = df_3_categorizado

In [ ]:
# Ahora a ver cuáles son esos valores únicos en estas variables:
print(f"Opciones de  habitaciones: {df_fotocasa_4['habitaciones'].unique()}")
print(f"Opciones de  Baños: {df_fotocasa_4['banos'].unique()}")

*Recuento exacto según las notas del análisis:* 1 hab. → 1.822 · 2 → 4.343 · 3 → 5.937 · 4 → 2.297 · 5 → 671 · 6 → 270 · 7 → 91 · 8 → 44 · 9 → 20 · 10 → 21 · 11 → 2 · 12 → 3 · 13 → 1 · 15 → 1 · 16 → 2 · 26 → 1 (seguramente erróneo). Valores combinados: 1-2 → 11 · 1-3 → 17 · 1-4 → 7 · 2-3 → 34 · 2-4 → 16 · 2-5 → 1 · 3-4 → 16 · 3-5 → 1 · 4-5 → 2.

In [ ]:
# Se verán mejor con un recuento:
print(f"Recuento Nº de habitaciones: {df_fotocasa_4['habitaciones'].value_counts().sort_index(ascending=True).to_string()}")

In [ ]:
# Filtro por los casos combinados:
df_combinados_hab = df_fotocasa_4[df_fotocasa_4['habitaciones'] == '2 - 3 habs·']
df_combinados_hab

Veo que, afortunadamente, los casos caombinados de habitaciones son pocos.
Vemos que tenemos la mayoría de casos están en valores correctos (no combinados) y además lógicos y frecuentes. Tenemos:

- Inmuebles con menos de 10 habitaciones → **15.495 inmuebles**
- Inmuebles con 10 o más de 10 habitaciones → **31 inmuebles**
- Inmuebles con valor combinado en habitaciones → **106 inmuebles**

 Voy a echar un vistazo a los baños antes de determinar lo que hacer con los casos "problemáticos" (sea corregirlos de manera simple con el valor intermedio, o retirarlos cuando corresponda), porque en algunos casos veo un patrón de combinados en habitaciones, en baños y además son Obra nueva.

*Recuento exacto según las notas del análisis:* 1 baño → 7.210 · 2 → 5.583 · 3 → 1.755 · 4 → 666 · 5 → 245 · 6 → 107 · 7 → 46 · 8 → 31 · 9 → 8 · 10 → 8 · 11 → 2 · 13 → 1 · 16 → 2. Valores combinados: 1-2 → 32 · 1-3 → 7 · 1-4 → 2 · 1-5 → 1 · 2-3 → 22 · 2-4 → 2 · 3-4 → 5 · 3-5 → 1 · 4-5 → 2.

In [ ]:
# Voy a revisar baños ahora
print(f"Recuento Nº de baños: {df_fotocasa_4['banos'].value_counts().sort_index(ascending=True).to_string()}")

In [ ]:
# Filtro por los casos combinados en baños:
df_combinados_bañ = df_fotocasa_4[df_fotocasa_4['banos'] == '1 - 2 baños·']
df_combinados_bañ

Vemos que en baños también tenemos la mayoría de casos están en valores correctos (no combinados) y además lógicos y frecuentes. Tenemos:

- Inmuebles con menos de 10 baños → **15.651 inmuebles**
- Inmuebles con 10 o más de 10 baños → **13 inmuebles**
- Inmuebles con valor combinado en baños → **74 inmuebles**

Como imaginaba, hay coincidencias entre los valores combinados de habitaciones y de baños, y todo dentro de Obra nueva.
Veo que **lo mejor es retirar** directamente estos:
- Los que tengan un combinado en habitaciones
- Los que tengan un combinado en baños si no han sido ya eliminados con lo de habitaciones.

In [ ]:
# Retiramos los que tienen COMBINADOS en habitaciones, y si han quedado baños también
# Primero aislamos aquellos que tienen un valor combinado
habitaciones_combinados = df_fotocasa_4['habitaciones'].str.contains(' - ', na=False)
baños_combinados = df_fotocasa_4['banos'].str.contains(' - ', na=False)

# Verificación previa
print(df_fotocasa_4[habitaciones_combinados]['habitaciones'].value_counts())
print(df_fotocasa_4[baños_combinados]['banos'].value_counts())


También quito el caso del que tiene 26 habitaciones porque es un error:

In [ ]:
error_hab = df_fotocasa_4['habitaciones'] == '26 habs·'

In [ ]:
# Eliminamos
combinados_a_eliminar = habitaciones_combinados | baños_combinados | error_hab
df_4_sin_combinados = df_fotocasa_4[~combinados_a_eliminar].copy()
print(f"Filas eliminadas: {combinados_a_eliminar.sum()} | Filas restantes: {len(df_4_sin_combinados)}")

In [ ]:
# Vuelvo a hacer recuentos para comprobar:
print(f"Recuento Nº de habitaciones: {df_4_sin_combinados['habitaciones'].value_counts().sort_index(ascending=True).to_string()}")

In [ ]:
# Voy a revisar baños ahora
print(f"Recuento Nº de baños: {df_4_sin_combinados['banos'].value_counts().sort_index(ascending=True).to_string()}")

Ya hemos hecho la primera limpieza de Habitaciones y Baños. Ahora vamos a pasar a números los valores (almacenando el texto en detalles) y así poder ahcer un análisis estadístico

## 12.1 Sacamos el valor numérico de habitaciones y baños

In [ ]:
df_4_sin_combinados

In [ ]:
# SAcamos con un regex el número:
df_4_sin_combinados['num_habitaciones'] = df_4_sin_combinados['habitaciones'].str.extract(r'(\d+)').astype(float)
df_4_sin_combinados['num_banos']        = df_4_sin_combinados['banos'].str.extract(r'(\d+)').astype(float)

# Comprobación
print(df_4_sin_combinados['num_habitaciones'].value_counts().sort_index())
print(df_4_sin_combinados['num_banos'].value_counts().sort_index())

In [ ]:
df_4_sin_combinados

Voy a **quitar las columnas de habitaciones y banos** que ya tengo en formato numérico porque no me hacen falta.. Y además voy a corregir el nombre de num_banos y a **ordenar las columnas** porque así trabajaremos mejor con ellas

## 12.2 Reorganizamos las columnas

In [ ]:
df_4_sin_combinados.columns

In [ ]:
df_fotocasa_5 = df_4_sin_combinados.drop(['habitaciones','banos'], axis=1)

In [ ]:
df_fotocasa_5 = df_fotocasa_5.rename(columns={'num_banos': 'num_baños'})

In [ ]:
# Ahora introduzco el nuevo orden:
nuevo_orden = [
    'id_anuncio', 'precio', 'publicacion', 'num_habitaciones', 'num_baños',
    'metros', 'planta', 'nombre', 'tipo_inmueble', 'ciudad', 'barrio',
    'extras', 'detalles_crudo', 'link', 'descripcion']

df_fotocasa_5 = df_fotocasa_5[nuevo_orden].copy()

In [ ]:
df_fotocasa_5

Para tenerlo claro hasta ahora, voy a lista aquí los cambios hechos hasta ahora al df_fotocasa_web_scraping por números:

- **df_fotocasa_web_scraping** → Es el df_fotocasa_web_scraping original, sin cambios.
- **df_fotocasa_2** → Aquí lo único que he retirado es aquellos anuncios que aparecen duplicados en el df_fotocasa_web_scraping, diferenciando por el ID_anuncio que he creado a partir dl link.
- **df_fotocasa_3** → Tras analizar los nulos en habitaciones, baños, planta y m2, he decidido prescindir de aquellos que eran a la vez nulos en habitaciones y en baños; y luego los que eran nulos en m2. Y además he hecho otra pasado de retirar duplicados usando otro método (valores duplicados en varias columnas a la vez).
- **df_fotocasa_4** → Tras analizar los recuentos de habitaciones y baños, he encontrado los valores combinados de ambos y al ver que la mayoría eran poco útiles los he retirado. Finalmente he convertido los valores a números.
- **df_fotocasa_5** → Aquí simplemente he eliminado las columnas de habitaciones y de baños porque tenemos las que contienen los números y además he reorganizado las columnas para trabajar mejor con ellas.

## 12.3 Outliers de habitaciones y baños

In [ ]:
print(f"Recuento Nº de habitaciones: {df_fotocasa_5['num_habitaciones'].value_counts().sort_index(ascending=True).to_string()}")

In [ ]:
sns.boxplot(data=df_fotocasa_5, x='num_habitaciones')
plt.title("Distribución del número de habitaciones (df_fotocasa_5)")
plt.show()

In [ ]:
# Vamos a ver si deberíamos incluir los inmuebles con MÁS DE 10 HABITACIONES
df_habitaciones_muchas = df_fotocasa_5[(df_fotocasa_5['num_habitaciones']>10)].sort_values(by='num_habitaciones')
df_habitaciones_muchas

He revisado en detalle varios anuncios con habitaciones por encima de 10 habitaciones y **no tendría sentido incluirlos** porque los inmuebles son casos muy particulares. Además, estos incluyen varios de los casos con muchos baños.

Así que voy a **primero a mirar los baños** por si acaso.

In [ ]:
print(f"Recuento Nº de baños: {df_fotocasa_5['num_baños'].value_counts().sort_index(ascending=True).to_string()}")

In [ ]:
sns.boxplot(data=df_fotocasa_5, x='num_baños')
plt.title("Distribución del número de baños (df_fotocasa_5)")
plt.show()

Los boxplots nos indican que habría que retirar los anuncios que estén:
- Por encima de 4 habitaciones
- Por encima de 3 baños

Sin embargo, tampoco conviene perder esta parte de nuestro dataset porque son relevantes y además que tampoco tenemos tantos inmuebles para nuestra muestra.

Por ello, voy a poner la lupa en más de 10 habitaciones y ahora en más de 8 baños.

In [ ]:
df_baños_muchos = df_fotocasa_5[(df_fotocasa_5['num_baños']>8)].sort_values(by='num_habitaciones')
df_baños_muchos

Como intuyo que nos van a ensuciar nuestra muestra, procedo a filtrar y **quedarme solo con los que están por debajo de 11 habitaciones y por debajo de 9 baños** → **nos quedan 15.768 inmuebles**.

In [ ]:
# Voy a eliminar esos casos con .copy() y creo un nuevo df_fotocasa_web_scraping:
muchos_baños = df_fotocasa_5['num_baños']>9
muchas_habitaciones = df_fotocasa_5['num_habitaciones']>10
hab_bañ_muchos_eliminar = muchos_baños | muchas_habitaciones

df_fotocasa_6 = df_fotocasa_5[~hab_bañ_muchos_eliminar].copy()
print(f"Filas eliminadas: {hab_bañ_muchos_eliminar.sum()} | Filas restantes: {len(df_fotocasa_6)}")


Para tenerlo claro hasta ahora, voy a lista aquí los cambios hechos hasta ahora al df_fotocasa_web_scraping por números:

- **df_fotocasa_web_scraping** → Es el df_fotocasa_web_scraping original, sin cambios.
- **df_fotocasa_2** → Aquí lo único que he retirado es aquellos anuncios que aparecen duplicados en el df_fotocasa_web_scraping, diferenciando por el ID_anuncio que he creado a partir dl link.
- **df_fotocasa_3** → Tras analizar los nulos en habitaciones, baños, planta y m2, he decidido prescindir de aquellos que eran a la vez nulos en habitaciones y en baños; y luego los que eran nulos en m2. Y además he hecho otra pasado de retirar duplicados usando otro método (valores duplicados en varias columnas a la vez).
- **df_fotocasa_4** → Tras analizar los recuentos de habitaciones y baños, he encontrado los valores combinados de ambos y al ver que la mayoría eran poco útiles los he retirado. Finalmente he convertido los valores a números.
- **df_fotocasa_5** → Aquí simplemente he eliminado las columnas de habitaciones y de baños porque tenemos las que contienen los números y además he reorganizado las columnas para trabajar mejor con ellas.
- **df_fotocasa_6** → Aquí simplemente he eliminado los outliers de habitaciones y baños. He decidido quitar los que están por encima de 10 habitaciones y 8 baños.

Y guardo csv para facilitar la gestión del NB → **datos_fotocasa_limpio_2.csv**

## 12.4 Guardar CSV (checkpoint 2)

In [ ]:
len(df_fotocasa_6)

In [ ]:
# (2do) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_fotocasa_6.to_csv('datos_fotocasa_limpio_2.csv', index=False)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_fotocasa_6 = df_fotocasa_6.copy()
print(f"Filas del df_fotocasa_6: {len(df_fotocasa_6)}")

# 13. Evaluar Fecha Publicación

Lo siguiente que vamos a hacer es revisar los valores que tenemos en fecha de publicación, donde encontraremos distintas medidas: días y meses.
Haré un recuento para ver lo que es relevante y luego transformaremos los valores.

In [ ]:
# Vamos a ver cuáles son esos valores únicos en estas variables:
print(f"Opciones de  Fecha Publicación: {df_fotocasa_6['publicacion'].unique()}")

In [ ]:
# Se verán mejor con un recuento:
print(f"Recuento Nº de Fechas de Publicación: {df_fotocasa_6['publicacion'].value_counts().sort_index(ascending=True).to_string()}")

Veo que tenemos demasiadas opciones de fecha (o momento) de publicación. Así que lo más simple será pasarlo todo a la misma medida, que por la distribución sería número de días que el anuncio lleva publicado.

In [ ]:
# Defino una función que vaya convirtiendo en valores númericos que correspondan a días
import re
def publicacion_a_dias(texto):
    if pd.isna(texto):              return None
    texto = texto.lower().strip()
    if texto == 'ayer':             return 1
    if 'más de 3 meses' in texto:  return 92
    if 'minuto' in texto:           return 0
    if 'hora' in texto:             return 0

    match = re.search(r'(\d+)', texto)
    if match:
        n = int(match.group(1))
        if 'día' in texto:  return n
        if 'mes' in texto:  return n * 30
    return None
# Aplicamos la función a nuestra columna publicación para crear la nueva columna:
df_fotocasa_6['dias_publicado'] = df_fotocasa_6['publicacion'].apply(publicacion_a_dias)


In [ ]:
# Comprobamos el resultado:
print(df_fotocasa_6['dias_publicado'].value_counts().sort_index())

In [ ]:
df_fotocasa_6

# 14. Evaluamos el número de planta

Voy a empezar a mirar en detalle la distribución del número de plantas, sabiendo que las descripciones parecen estructuradas y, al contrario de lo que teníamos en Idealista, aquí solo tenemos **26 valores únicos** en la columna Planta.

*Según las notas del análisis*, las opciones observadas en la columna `planta` (26 valores únicos, bastante menos variabilidad que en Idealista) son: 2ª, 3ª, 5ª, +15, 4ª, 1ª, Entresuelo, Bajos, 6ª, 8ª, 7ª, 9ª y 10ª, con variantes de formato (algunas repetidas con un punto al final).

In [ ]:
# Vamos a hacer un recuento de los valores que tenemos en planta
print(f"Nº de planta: {df_fotocasa_6['planta'].nunique()}")
print(f"Recuento de opciones de planta: {df_fotocasa_6['planta'].value_counts().to_string()}")

Vemos que aquí tenemos bastante menos variabilidad que en Idealista, aunque tenemos opciones repetidas con un punto al final. Procederemos a extraer y clasificar apropiadamente estas variables en columnas nuevas con valores numéricos para poder hacer análisis estadísticos:

1.- Extraemos el número de planta: Buscamos un dígito (\d+) seguido del símbolo ª y lo metemos en una nueva columna, convirtiéndolo a integer. Si en la columna 'planta' pone 'Bajo', ponemos un 0 en 'num_planta'. Si pone "Entresuelo" lo convertiré en 0.5 para utilizarlo como un grado a la hora de comparar con otras variables.

2.- En Idealista evaluamos ascensor y si el piso es exterior o interior en este mismo punto porque estaban en la misma columna. Pero aquí lo tenemos todo en la columna Extra, así que lo haremos más adelante.

In [ ]:
# 1. Extraemos el número (ej: 2ª, 1ª)
df_fotocasa_6['num_planta'] = df_fotocasa_6['planta'].str.extract(r'(\d+)ª')
df_fotocasa_6['num_planta'] = pd.to_numeric(df_fotocasa_6['num_planta'])
# 2. Vamos a extraer aquí los casos especiales
df_fotocasa_6.loc[df_fotocasa_6['planta'].str.contains('Bajo', na=False), 'num_planta'] = 0
df_fotocasa_6.loc[df_fotocasa_6['planta'].str.contains('Entresuelo', na=False), 'num_planta'] = 0.5
df_fotocasa_6.loc[df_fotocasa_6['planta'].str.contains('\+15',       na=False), 'num_planta'] = 15


# 3. Comprobamos el resultado
df_fotocasa_6[['planta', 'num_planta']].head(10)

In [ ]:
print(f"Nº de planta: {df_fotocasa_6['num_planta'].value_counts().to_string()}")

In [ ]:
df_fotocasa_6['num_planta'].mean()

Compruebo si hay valores no convertidos evaluando un df_fotocasa_web_scraping donde haya nulos en num_planta que no haya en planta

In [ ]:
df_planta_nulos = df_fotocasa_6[(df_fotocasa_6['num_planta'].isnull() & df_fotocasa_6['planta'].notnull())].sort_values(by='planta')
df_planta_nulos

Sale que no hay ninguno. Reviso los nulos de planta de nuevo por si acaso

In [ ]:
df_planta_nulos = df_fotocasa_6[(df_fotocasa_6['num_planta'].isnull())].sort_values(by='planta')
df_planta_nulos

Aquí es donde tenemos el problema, que hay 4.775 filas con NaN en planta y por tanto nos faltará información en el análisis paralelo (es decir, que para el núcleo no, pero sí si queremos hacer distintas visualizaciones).

In [ ]:
num_plantas = df_fotocasa_6['num_planta']
plt.figure(figsize=(8,4))
plt.hist(num_plantas, bins=40, edgecolor='black')
plt.title('Distribución de Nº de planta')
plt.xlabel('Nº de planta')
plt.ylabel('Número de inmuebles')
plt.show()

Tenemos una distribución bastante particular, aunque principalmente por esos valores que están aislados en +15 plantas. Podrías ser considerados outliers automaticamente, pero a diferencia de la amplia distribución de valores en Idealista que nos hacía tener que corregir, aquí no estimamos necesario retirar nada de momento. Mejor no seguir reduciendo nuestra muestra en este punto y proceder a analizar otras variables antes de tomar deecisiones sobre estos casos.

# 15. Extraemos los Metros Cuadrados

Vamos a sacar el valor numérico de la variable **metros cuadrados (m2)** para poder hacer cálculos con ella.

In [ ]:
# Vamos a hacer un recuento de los valores que tenemos en m2
print(f"Nº de Opciones m2: {df_fotocasa_6['metros'].nunique()}")
print(f"Recuento de m2: {df_fotocasa_6['metros'].value_counts().to_string()}")

Parece que tenmos una cantidad inusitada de opciones de m2 en nuestra muestra, con una distribución acumulada en torno a los 100 primeros valores pero muy heterogénea en general. Tendremos que primero transformar en valor numérico antes de valorar una limpieza.

In [ ]:
# Extraemos los valores numéricos
df_fotocasa_6['num_m2_2'] = df_fotocasa_6['metros'].str.extract(r'(\d+)m')
df_fotocasa_6['num_m2_2'] = pd.to_numeric(df_fotocasa_6['num_m2_2'])

In [ ]:
df_fotocasa_6.sample(5)

No me ha funcionado bien ninguno de los 2 intentos porque hay un espacio antes de la m y el símbolo ² después. El regex necesita contemplar ese espacio.

In [ ]:
# NOTA: igual que con id_anuncio en Fotocasa, el cuaderno original asumía una columna 'num_m2'
# de un primer intento fallido previo que no existe en este flujo (solo existe 'num_m2_2', creada
# arriba). Vamos directos al grano: quitamos solo 'num_m2_2' y extraemos bien 'num_m2'.

# Extraer correctamente. El \s* captura cero o más espacios entre el número y la m,
# así funciona tanto para "72 m²·" como para "72m²" si hubiera algún caso sin espacio.
df_fotocasa_6['num_m2'] = df_fotocasa_6['metros'].str.extract(r'(\d+)\s*m')
df_fotocasa_6['num_m2'] = pd.to_numeric(df_fotocasa_6['num_m2'])


In [ ]:
# Comprobamos el resultado
print(f"Valores en nueva columna: {df_fotocasa_6['num_m2'].count()}")
print(f"NaN en 'metros': {df_fotocasa_6['metros'].isna().sum()}")
print(f"NaN en 'num_m2': {df_fotocasa_6['num_m2'].isna().sum()}")

In [ ]:
df_fotocasa_6['num_m2'].sample(6)

Ya hemos conseguido sacar los valores correctamente. Voy ahora a hacer una representación gráfica para ver lo que hemos obtenido.

In [ ]:
num_m2 = df_fotocasa_6['num_m2']
plt.figure(figsize=(16,4))
plt.hist(num_m2, bins=40, edgecolor='black')
plt.title('Distribución de Nº de m2')
plt.xlabel('Nº de m2')
plt.ylabel('Número de inmuebles')
plt.show()

Hay valores tan altos que al intentar hacer un gráfico, se me acumula todo al principio sin distribuirse porque los hay lo suficientemente altos para que la leyenda se nos vaya por encima de los 50.000

In [ ]:
print("\nValores por encima de 500 m2:", (df_fotocasa_6['num_m2'] > 500).sum())
print(df_fotocasa_6[df_fotocasa_6['num_m2'] > 500]['num_m2'].value_counts().head(10))

In [ ]:
print("\nValores por encima de 1.000 m2:", (df_fotocasa_6['num_m2'] > 1000).sum())
print(df_fotocasa_6[df_fotocasa_6['num_m2'] > 1000]['num_m2'].value_counts().head(10))

In [ ]:
# Vuelvo a hacer un recuento de los valores que tenemos en m2
print(f"Recuento de m2: {df_fotocasa_6['num_m2'].value_counts().to_string()}")

In [ ]:
# Vamos a hacer un boxplot para evaluar outliers
plt.figure(figsize=(15,5))
sns.boxplot(data=df_fotocasa_6, x='num_m2')
plt.title("Distribución de los metros cuadrados")
plt.show()

In [ ]:
# Veo que hay 2 valores por encima de 150.000 por lo que voy a
print("\nValores por encima de 150.000 m2:", (df_fotocasa_6['num_m2'] > 150000).sum())
print(df_fotocasa_6[df_fotocasa_6['num_m2'] > 150000]['num_m2'].value_counts().head(10))

In [ ]:
df_6_m2_extremos = df_fotocasa_6[df_fotocasa_6['num_m2'] > 150000]
df_6_m2_extremos

Como me temía, al mirar en detalle esos valores extremos de 375.000 m2 y 171.000 m2, uno es directamente un error y el otro está mal cogido por la página y en realidad son 171 m2.
Con estos valores, va a ser complicado evaluar en detalle qué valores son correctos y cuáles no.

Voy a mirar otros casos altos por si acaso.

In [ ]:
df_6_m2_extremos_2 = df_fotocasa_6[df_fotocasa_6['num_m2'] > 1000]
df_6_m2_extremos_2

Hasta aquí veo que tenemos solo 37 filas con m2 por encima de 1.000, lo cual hace que sea más fácil retirarlos y no complicarnos más porque es difícil ir caso a caso analizando, y tampoco podemos establecer un patrón para corregir los m2.
PAso a analizar los valores que están por entre 500 y 1.000 metros cuadrados.

In [ ]:
# Primero retiro los anuncios con m2 > 1000. Voy a eliminar esos casos con .copy() y creo un nuevo df_fotocasa_web_scraping:
muchos_m2 = df_fotocasa_6['num_m2']>1000
df_6_sin_extremos = df_fotocasa_6[~muchos_m2].copy()
print(f"Filas eliminadas: {muchos_m2.sum()} | Filas restantes: {len(df_6_sin_extremos)}")

In [ ]:
# Y ahora voy a hacer ya un boxplot de nuevo para ver si puedo sacar algo más en calro
plt.figure(figsize=(15,5))
sns.boxplot(data=df_6_sin_extremos, x='num_m2')
plt.title("Distribución de los metros cuadrados")
plt.show()

In [ ]:
num_m2_2 = df_6_sin_extremos['num_m2']
plt.figure(figsize=(16,4))
plt.hist(num_m2_2, bins=40, edgecolor='black')
plt.title('Distribución de Nº de m2')
plt.xlabel('Nº de m2')
plt.ylabel('Número de inmuebles')
plt.show()

Una vez retirados los valores por encima de 1.000 m2, al hacer un boxplot de nuevo podemos ver ya mejor la distribución y lo que está dentro del IQR. Además, veo que hay muchos valores “díscolos” entre 600 m2 y 1.000 m2. Podría retirar esos casos también y ya haré una limpieza más exhaustiva una vez calculemos Precio/m2. Ahí sí que nos dirá si el valor tiene o no sentido, porque si el error es que habría que dividir los m2 por algún múltiplo de 10, entonces nos saldrá claro en Precio/m2.

In [ ]:
df_6_m2_extremos_3 = df_fotocasa_6[(df_fotocasa_6['num_m2'] < 1000) & (df_fotocasa_6['num_m2'] > 600)]
df_6_m2_extremos_3

Parece que tiene sentido dado los prrecios que encontramos (de varios millones de euros). Así que por ahora lo voy a dejar así, y pasaré a revisar el Precio/m2.

## 15.1 Guardar CSV (checkpoint 3)

In [ ]:
len(df_6_sin_extremos)

In [ ]:
# (2do) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_6_sin_extremos.to_csv('datos_fotocasa_limpio_3.csv', index=False)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_fotocasa_7 = df_6_sin_extremos.copy()
print(f"Filas del df_fotocasa_7: {len(df_fotocasa_7)}")

# 16. Extraemos el Precio y Calculamos Precio/m2

Voy a empezar haciendo repaso de las variables que tenemos hasta ahora

In [ ]:
df_fotocasa_7.info()

Antes de empezar a mirar correlaciones o calcular valores estadísticos, nos queda todavía por extraer el Precio que ahora mismo es un objetc con €.

In [ ]:
df_fotocasa_7.sample(6)

Usaremos el método que ya hemos usado para sacar valores numéricos en otras columnas, pero cambiando a [\d\.]+\s*€ para que capture el patrón completo (dígitos y puntos juntos seguidos de €). Luego quitamos los puntos de miles y el símbolo para convertirlo a entero con replace. Y dejaremos que los "A consultar" queden como nulos para que luego los retiremos.

In [ ]:
def extraer_precio(texto):
    if pd.isna(texto):
        return None
    match = re.search(r'[\d\.]+\s*€', texto)
    if match:
        precio_limpio = match.group().replace('.', '').replace('€', '').strip()
        return int(precio_limpio)
    return None  # Para cuando nos fata el precio

df_fotocasa_7['precio_eur'] = df_fotocasa_7['precio'].apply(extraer_precio)

In [ ]:
# comprobamos que ha funcionado:
print(f"NaN (A consultar u otros): {df_fotocasa_7['precio_eur'].isna().sum()}")
print(df_fotocasa_7['precio_eur'].describe().to_string())

In [ ]:
df_fotocasa_7.sample(6)

In [ ]:
# Ahora paso a retirar los valores nulos:
nulos_precio = df_fotocasa_7['precio_eur'].isnull()
df_7_precio = df_fotocasa_7[~nulos_precio].copy()
print(f"Filas eliminadas: {nulos_precio.sum()} | Filas restantes: {len(df_7_precio)}")

Hemos extraído exitósamente los valores numéricos, y hemos retirado los que nos han dado nulos porque no aportan ningún valor.

In [ ]:
df_7_precio

En el dataset de idealista, vimos que la principal correlación es con los m2. Antes de comprobar que es el caso o de calcular el Precio/m2, voy a mirar bien los valores en el precio. Primero un boxplot de solo el precio.


## 16.1 Limpiamos el precio

In [ ]:
# Hago el primer boxplot del precio
plt.figure(figsize=(15,5))
sns.boxplot(data=df_7_precio, x='precio_eur')
plt.title("Distribución del Precio")
plt.show()

In [ ]:
# Filtro por los más altos, asumiendo 1M pero sobre todo ordenando
precio_alto = df_7_precio[df_7_precio['precio_eur'] > 10000000].sort_values(by='precio_eur', ascending=False)
precio_alto

Está claro que el más alto es un error. Lo retiro y seguimos analizando y limpiando.

In [ ]:
# NOTA: en el cuaderno original esto se hacía con df_7_precio.drop([14483]), por índice de fila.
# Identificamos esa fila de forma robusta por su id_anuncio: 183984123 (precio de 999.999.999 €, un error claro).
df_7_corregido_altos = df_7_precio[df_7_precio['id_anuncio'] != 183984123].copy()

In [ ]:
# Y ahora voy a hacer otro boxplot de nuevo para ver si puedo sacar algo más en calro
plt.figure(figsize=(15,5))
sns.boxplot(data=df_7_corregido_altos, x='precio_eur')
plt.title("Distribución del Precio")
plt.show()

In [ ]:
precio_alto_2 = df_7_corregido_altos[df_7_corregido_altos['precio_eur'] > 8000000].sort_values(by='precio_eur', ascending=False)
precio_alto_2

Veo que hay claramente un grupo por encima de los 10M€ que se separa del resto, porque por debajo están más suvizados entre ellos (la diferencia entre uno y otro es menor), más escalonados.
Por ello, voy a retirar los que están por encima de 10M€ para poder continuar.

In [ ]:
# Pasamos a retirar los valores por encima de 10M€:
altos_precio = df_7_corregido_altos['precio_eur'] > 10000000
df_7_corregido_altos_2 = df_7_corregido_altos[~altos_precio].copy()
print(f"Filas eliminadas: {altos_precio.sum()} | Filas restantes: {len(df_7_corregido_altos_2)}")

In [ ]:
# Y ahora voy a hacer un 3er boxplot de nuevo para ver si puedo sacar algo más en calro
plt.figure(figsize=(15,5))
sns.boxplot(data=df_7_corregido_altos_2, x='precio_eur')
plt.title("Distribución del Precio")
plt.show()

En este punto, en vez de ir quitando valores según el boxplot, consideramos más lógico pasar a **evaluar conjuntamente Precio/m2** para identificar valores atípicos.

###Calculamos Precio/m2

Voy a empezar pintando un gráfico de regresión entre el precio y los m2 para observar cómo se relacionan ambas variables.

In [ ]:
df_fotocasa_8 = df_7_corregido_altos_2

In [ ]:
# Para poder pintar un gráfico mejor que pueda mostrarme más información facilmente, usaré plotly e instalo "statsmodels"
import plotly.express as px
!pip install statsmodels

In [ ]:
fig = px.scatter(
    df_fotocasa_8,
    x='num_m2',
    y='precio_eur',
    trendline='ols',          # línea de tendencia por regresión lineal
    title='Relación entre m² y Precio',
    labels={'num_m2': 'Metros cuadrados', 'precio_eur': 'Precio (€)'},
    opacity=0.4,              # transparencia para ver densidad donde se solapan puntos
)

fig.show()

En un vistazo del scatter entre las variables Precio y m2, veo que no solo puede haber inmuebles fuera de la línea de tendencia (y por tanto, posibles outliers), pero desde luego están más próximos a la línea de tendencia que en Idealista, por lo que puede que no haya muchos outliers. Pero seguramente esos puntos más alejados de la linea destaquen ahora al **calcular Precio/m2**.

In [ ]:
# Añadimos la columna Precio/m2
df_fotocasa_8['Precio/m2'] = df_fotocasa_8['precio_eur'] / df_fotocasa_8['num_m2']

In [ ]:
df_fotocasa_8.sample(4)

In [ ]:
# Hago un histrograma sencillo ahora solo para una comproabación
dist_precio_m2 = df_fotocasa_8['Precio/m2']
plt.figure(figsize=(16,4))
plt.hist(dist_precio_m2, bins=40, edgecolor='black')
plt.title('Distribución del Precio/m2')
plt.xlabel('Precio/m2')
plt.ylabel('Número de inmuebles')
plt.show()

In [ ]:
# Y hacemos el primer boxplot
plt.figure(figsize=(15,5))
sns.boxplot(data=df_fotocasa_8, x='Precio/m2')
plt.title("Distribución del Precio/m2")
plt.show()

La primera representación del Precio/m2 nos queda muy abierta a la derecha por los outliers que podemos observar en el boxplot. Particularmenete por esos 3 o 4 que son más díscolos o que están más alejados de los valores medios de la variable. Empezaré retriándolos y luego volvemos a graficar.

In [ ]:
# Filtro por los más altos para empezar limpiando
precio_m2_alto = df_fotocasa_8[df_fotocasa_8['Precio/m2'] > 32000].sort_values(by='Precio/m2', ascending=False)
precio_m2_alto

In [ ]:
# NOTA: en el cuaderno original esto se hacía con df_fotocasa_8.drop([9706, 14164, 8148, 2811]), por índice de fila.
# Identificamos estas 4 filas de forma robusta por su id_anuncio: 188941704, 186426673, 189484171 y 189265827
# (los 4 valores de Precio/m2 más altos, muy por encima del resto).
ids_precio_m2_altos = [188941704, 186426673, 189484171, 189265827]
df_8_corregido_altos = df_fotocasa_8[~df_fotocasa_8['id_anuncio'].isin(ids_precio_m2_altos)].copy()

In [ ]:
# Y volvemos a graficar:
dist_precio_m2_2 = df_8_corregido_altos['Precio/m2']
plt.figure(figsize=(16,4))
plt.hist(dist_precio_m2_2, bins=40, edgecolor='black')
plt.title('Distribución del Precio/m2')
plt.xlabel('Precio/m2')
plt.ylabel('Número de inmuebles')
plt.show()

In [ ]:
# Y hacemos el segundo boxplot
plt.figure(figsize=(15,5))
sns.boxplot(data=df_8_corregido_altos, x='Precio/m2')
plt.title("Distribución del Precio/m2")
plt.show()

Una vez retirados esos 4 valores más atípicos, ya tenemos una representación más normal en la distribución y los outliers más ordenados en el boxplot. De nuevo, no haría una gran limpieza de todos los que están en los valores superiores al límite del boxplot porque es lógico que haya valores extremos (por un m2 más caro) pero también porque no podemos permitirnos retirar muchos elementos de nuestra muestra.
Pero sí que voy a **revisar** los que estén **por encima de 20.000 € / m2** porque están muy alejados de los valores medios.

In [ ]:
# Filtro por los más altos para seguir limpiando
precio_m2_alto_2 = df_8_corregido_altos[df_8_corregido_altos['Precio/m2'] > 20000].sort_values(by='Precio/m2', ascending=False)
precio_m2_alto_2

## 16.3 Retirar duplicados (3ª pasada)

Al ir a revisar los valores que están por encima de 20.000 € por m2, he descubierto que **tenemos todavía** una serie de **duplicados** que antes no se han limpiado (cuando hice limpieza de los que tenían el mismo precio+publicacion+nombre+barrio+metros) porque parece que tienen un nombre un poco distinto, uno tiene planta el otro es nulo y sobre todo la fecha de publicación varía, pero otros detalles son iguales. Voy a tener que hacer **otra limpieza de duplicados**, retirando los que **tengan igual precio+metros+habitaciones**

Pero antes, voy a volver a ordenar las columnas y retirar los nulos de num_habitaciones.

In [ ]:
# Hago otra pasada para hacer una retirada de nulos en la columna habitaciones
df_8_corregido_altos = df_8_corregido_altos.dropna(subset=['num_habitaciones']).copy()
print(f"Filas restantes después de eliminar nulos en 'num_habitaciones': {len(df_8_corregido_altos)}")

In [ ]:
# Tercera pasada: eliminar los que comparten precio+publicacion+nombre+barrio+metros
# por si algún anuncio se scrapeó con distintos links porque se repiten entre las páginas o algún error parecido
df_duplicados_3 = df_8_corregido_altos[df_8_corregido_altos.duplicated(subset=['precio_eur', 'num_m2', 'num_habitaciones'], keep=False)].copy().sort_values(by='Precio/m2')
df_duplicados_3

In [ ]:
# Los retiramos
df_8_corregido_altos_duplicados = df_8_corregido_altos.drop_duplicates(
    subset=['precio_eur', 'num_m2', 'num_habitaciones'], keep='first'
).copy()

print(f"Antes: {len(df_8_corregido_altos)} | Después: {len(df_8_corregido_altos_duplicados)} | Retirados: {len(df_8_corregido_altos) - len(df_8_corregido_altos_duplicados)}")

De momento he retirado casi 300 filas solo con los nulos en la columna ‘num_habitaciones’. Ahora ya sí puedo pasar a hacer la otra retirada con precio+metros+habitaciones. Y al introducir la condición, veo que tenemos 5.616 filas duplicadas, lo cual es un golpe fuerte a nuestra muestra. Pero tendremos que retirar porque, si no, cualquier análisis tendrá fallos importantes:

**Antes: 15.395 | Después: 11.602 | Retirados: 3.793**

## 16.4 Volvemos con Precio/m2

In [ ]:
df_fotocasa_9 = df_8_corregido_altos_duplicados

In [ ]:
# Vuelvo a mirar los máss altos para seguir limpiando
precio_m2_alto_3 = df_fotocasa_9[df_fotocasa_9['Precio/m2'] > 25000].sort_values(by='Precio/m2', ascending=False)
precio_m2_alto_3

In [ ]:
# NOTA: en el cuaderno original esto se hacía con .drop([3197,4263,14951,14882,14951,3671,4077,6844,2582,10688]),
# por índice de fila (con el índice 14951 repetido dos veces por error de transcripción: son 9 filas, no 10).
# Identificamos las 9 filas de forma robusta por su id_anuncio. Ocho de ellas ya estaban anotadas como
# comentario en el cuaderno original; la novena (189235497) faltaba en el comentario pero sí se retiraba en
# el código — lo confirmamos ejecutando el pipeline real hasta este punto exacto.
ids_precio_m2_caros = [189507084, 188271567, 189270951, 188044311, 189306437,
                        183513583, 186171915, 189317465, 189235497]
df_9_corregido_caros = df_fotocasa_9[~df_fotocasa_9['id_anuncio'].isin(ids_precio_m2_caros)].copy()

Al ir a revisar los valores que están por encima de 20.000 € por m2 de nuevo, veo que sí que hay valores muy altos que quizás nos distorsionan la muestra, pero puede que haya valor en invertir en algunos inmuebles de lujo para luego poder alquilar. Así que voy a **retirar** solo los que se alejan más del resto (**igual y por encima de 25K €/m2**)y vuelvo a mirar la distribución.

In [ ]:
# Vuelvo a graficar:
dist_precio_m2_3 = df_9_corregido_caros['Precio/m2']
plt.figure(figsize=(16,4))
plt.hist(dist_precio_m2_3, bins=40, edgecolor='black')
plt.title('Distribución del Precio/m2')
plt.xlabel('Precio/m2')
plt.ylabel('Número de inmuebles')
plt.show()

In [ ]:
# Y hacemos el tercer boxplot
plt.figure(figsize=(15,5))
sns.boxplot(data=df_9_corregido_caros, x='Precio/m2')
plt.title("Distribución del Precio/m2")
plt.show()

Finalmente, dejaré de retirar los más caros porque ya por debajo de 25.000 € por m2 veo que evolución sin saltos y parece valores lógicos. Además que necesitaremos también muestreo de los más lujosos para algunos casos, y que siempre podremos retirarlos una vez hagamos el modelo de rentabilidad (ROI).

In [ ]:
len(df_9_corregido_caros)

## 16.5 Guardar CSV (checkpoint 4)

In [ ]:
# (5to) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_9_corregido_caros.to_csv('datos_fotocasa_limpio_4.csv', index=False)

In [ ]:
df_fotocasa_10 = df_9_corregido_caros
len(df_fotocasa_10)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

## 16.6 Vuelvo a reorganizar las columnas

Vuelvo a reorganizar las columnas para facilitar el trabajar con ellas

In [ ]:
df_fotocasa_10.columns

In [ ]:
# Introduzco el nuevo orden:
nuevo_orden_2 = ['id_anuncio', 'precio_eur', 'num_habitaciones', 'num_m2', 'Precio/m2', 'num_baños', 'num_planta', 'tipo_inmueble',
                 'ciudad', 'barrio', 'dias_publicado', 'extras', 'detalles_crudo', 'link', 'descripcion', 'nombre',  'precio', 'publicacion', 'metros', 'planta']

df_fotocasa_10 = df_fotocasa_10[nuevo_orden_2].copy()

# 17. Analizamos ahora Precio/hab

Al igual que hicimos con el dataset de Idealista, voy a calcular el Precio por número de habitaciones para observar si hay casos extraños que debamos limpiar tamibén.

In [ ]:
# Vamos a calcular ahora la variable Precio/hab para revisar si tenemos algun caso raro desde este punto de vista:
df_fotocasa_10['Precio/hab'] = df_fotocasa_10['precio_eur'] / df_fotocasa_10['num_habitaciones']
df_fotocasa_10.sample(4)

In [ ]:
# Vamos a volver a hacer un boxplot para así buscar outliers
plt.figure(figsize=(15,5))
sns.boxplot(data=df_fotocasa_10, x='Precio/hab')
plt.title("Distribución del Precio/Habitaciones")
plt.show()

A simple vista, tenemos 3 o 4 claros outliers que seguramente sean conflictivos. Voy a revisar y posiblemente retirar esos casos para a continuación pasar a dibujar el boxplot de nuevo y ver si hay que retirar más

In [ ]:
df_fotocasa_10.sort_values(by='Precio/hab', ascending=True).head(6)

In [ ]:
df_fotocasa_10.sort_values(by='precio_eur', ascending=True).head(6)

Al observar los Precio/hab m'as bajos, he visto que en general hay inmuebles con un precio demasiado bajo para ser realista. **Tendré que hacer luego una última pasada** a los valores de:
- Precio
- Metros cuadrados
- Precio por m2 y habitaciones

In [ ]:
df_fotocasa_10.sort_values(by='Precio/hab', ascending=False).head(7)

De momento retiro los 3 casos más bajos (5519,12461,5782) y los 5 casos más altos (4616, 2885, 2667, 2361, 3238)

In [ ]:
# NOTA: en el cuaderno original esto se hacía con .drop([4329, 9430, 4554, 8345, 3589, 2259, 2105, 1835, 2508]),
# por índice de fila. Identificamos estas 9 filas de forma robusta por su id_anuncio: son los 3 valores
# de Precio/hab más bajos y los 6 más altos (con saltos claros respecto al resto).
ids_precio_hab_outliers = [189424025, 185289041, 188973114, 189581038, 187474404,
                            184766886, 189032636, 189483564, 184689650]
df_10_corregido_hab = df_fotocasa_10[~df_fotocasa_10['id_anuncio'].isin(ids_precio_hab_outliers)].copy()

In [ ]:
# Volvemos a hacer el boxplot
plt.figure(figsize=(15,5))
sns.boxplot(data=df_10_corregido_hab, x='Precio/hab')
plt.title("Distribución del Precio/Habitaciones")
plt.show()

Seguimos teniendo varios valores muy alejados de los límites superiores del boxplot. Tendré que seguir revisando estos porque la variable habitaciones sí que es sensible a variaciones muy altas del precio, ya que esta variable será utilizada en el modelo de rentabilidad (ROI).

Antes de haecr ese análisis conjunto del IQR para precio y luego m2, habitaciones y los coeficientes, voy a calcular la primera correlación para observar qué tenemos de momento.

In [ ]:
# Toca volver a reorganizar columnas:
nuevo_orden_2 = ['id_anuncio', 'precio_eur', 'num_habitaciones', 'num_m2', 'Precio/m2', 'Precio/hab', 'num_baños', 'num_planta', 'tipo_inmueble',
                 'ciudad', 'barrio', 'dias_publicado', 'extras', 'detalles_crudo', 'link', 'descripcion', 'nombre',  'precio', 'publicacion', 'metros', 'planta']

df_fotocasa_10 = df_10_corregido_hab[nuevo_orden_2].copy()

## 17.1 Primera correlación

In [ ]:
df_fotocasa_10.columns

In [ ]:
# Calculo la primera matriz de correlaciones
columnas_correlacion = ['precio_eur', 'num_habitaciones', 'num_m2', 'Precio/m2','Precio/hab', 'num_baños', 'num_planta']
matriz_corr = df_fotocasa_10[columnas_correlacion].corr()
matriz_corr

Lo primero que he hecho es compararlo con las correlaciones que encontrábamos en Idealista y es prácticamente la misma. En lo que se refiere a la relación del precio con el resto de variables, las diferencias entre ambos son de menos de 0,02 para la mayoría. Por lo que podemos establecer estas relaciones como sólidas y no espurias, ya que se replican entre nuestras 2 bases de datos.

De nuevo, tenemos una clara correlación entre el precio y los m2, y sorprendentemente también el número de baños. De esto podremos quizás sacar algo para comparar con los datos de AirBnB.
Al tener la correlación de 0.82 entre el Precio/m2 y el Precio/hab, consideraré simplemente el precio y los m2 para la siguiente revisión que voy a hacer.


# 18. Limpieza de casos especiales (Ocupados, Ruinas, etc.)

En este punto, ya parece que hemos hecho toda la limpieza que se podía hacer (a falta de una segunda revisión de los valores de Precio, m2 y hab). Faltaría hacer una pasada para intentar identificar si hay pisos que están por reformar, ocupados, en regimen de propiedad especial, etc. Aunque aquí no encuentro una manera de sacar ese tipo de información porque solo tenemos el nombre (y quizás los extras) para averiguarlo.

Pero, después de hacer una revisión exhaustiva, no hay ninguna columna en la que se puedan encontrar esas palabras para definir el inmueble. No tenemos las descripciones y en el nombre no viene nada más que lo más básico. **Así que no podremos hacer esta limpieza** como hicimos en Idealista.

# 19. Analizamos distribución del Precio y los m², y sus estadísticos

In [ ]:
len(df_fotocasa_10)

In [ ]:
# Desactiva la notación científica y muestra 2 decimales
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df_fotocasa_10.describe()

In [ ]:
# Vamos a plotear el precio primero
# Configuramos el estilo
sns.set_theme(style="whitegrid")
# Creamos el histograma con ajuste previo de la escala (A MILES DE €)
plt.figure(figsize=(16, 8))
sns.histplot(df_fotocasa_10['precio_eur'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios de Venta', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')

plt.show()

Vamos que la distribución es la esperada, sin embargo está más suavizada (menos pico al principio) que en Idealista y además vemos que los precios son bastante más bajos porque la leyenda aparece hasta 8.000 (en miles de €). Parece que no tenemos inmuebles tan caros como en Idealista y que la distribución es más uniforme.

Voy ahora a sacar todos los **estadísticos** y además los valores del **IQR para el precio**.

In [ ]:
# Imprimo los principales estadísticos de todo el dataset:
print(f"Recuento de nuestro dataset: {df_fotocasa_10['precio_eur'].count()}")
print(f"Promedio del Precio en nuestro dataset: {df_fotocasa_10['precio_eur'].mean()}")
print(f"Valor Medio del Precio en nuestro dataset: {df_fotocasa_10['precio_eur'].median()}")
print(f"Valor Modal del Precio en nuestro dataset: {df_fotocasa_10['precio_eur'].mode()}")
print(f"Valor Mínimo del Precio en nuestro dataset: {df_fotocasa_10['precio_eur'].min()}")
print(f"Valor Máximo del Precio en nuestro dataset: {df_fotocasa_10['precio_eur'].max()}")
print(f"Desviación típica del Precio en nuestro dataset: {df_fotocasa_10['precio_eur'].std()}")

In [ ]:
#Calculamos cuartiles y límites para detección de outliers
Q1_p = df_fotocasa_10['precio_eur'].quantile(0.25)
Q2_p = df_fotocasa_10['precio_eur'].quantile(0.50)
Q3_p = df_fotocasa_10['precio_eur'].quantile(0.75)
IQR_p = Q3_p - Q1_p # Calculo el Rango intrecuartil
limite_inferior_p = Q1_p - 1.5 * IQR_p
limite_superior_p = Q3_p + 1.5 * IQR_p

print(f"Límite inferior: {limite_inferior_p}€")
print(f"1er cuartil del precio - Q1 (25%): {Q1_p}")
print(f"2do cuartil del precio - Q2 (50%): {Q2_p}")
print(f"Valor promedio del precio: {df_fotocasa_10['precio_eur'].mean()}")
print(f"3er cuartil del precio - Q3 (75%): {Q3_p}")
print(f"Límite superior: {limite_superior_p}€")
print(f"Rango intrecuartil (IQR):  {IQR_p} € de rengo")

In [ ]:
# Compruebo cuanto valores hay fuera de los límites superiores
df_10__limite_sup = df_fotocasa_10[df_fotocasa_10['precio_eur']> 2000000]
df_10__limite_sup

In [ ]:
# Compruebo cuanto valores hay fuera de los límites inferiores (el calculado no es realista para este análisis, por lo que escojo un valor aproximado del que debería ser el inferior)
df_10__limite_inf = df_fotocasa_10[df_fotocasa_10['precio_eur'] < 100000]
df_10__limite_inf

Recuento de nuestro dataset: 11.584 filas

Promedio del Precio en nuestro dataset: 801.388,11 €

Valor Medio del Precio en nuestro dataset: 475.000 €

Valor Modal del Precio en nuestro dataset: 350.000 €

Valor Mínimo del Precio en nuestro dataset: 39.000 €

Valor Máximo del Precio en nuestro dataset: 9.500.000 €

Desviación típica del Precio en nuestro dataset: 901.196,34 €

Límite inferior (IQR): - 697.949,62 €

1er cuartil del precio - Q1 (25%): 285.000 €

2do cuartil del precio - Q2 (50%): 475.000 €

Valor promedio del precio: 801.388,11 €

3er cuartil del precio - Q3 (75%): 940.299,75 €

Límite superior (IQR): **1.923.249,37 €**

Rango intercuartil (IQR):  655.299,75 € de rango

Nº de filas por encima del límite superior: **880 filas**

Nº de filas por debajo de 100.000 €: **124 filas**

Una opción segura para evitar que haya tanta dispersión sería retirar estos casi 1.000 inmuebles, pero no tenemos los suficientes en la muestra como para retirar lo que ahora mismo sería casi un 10%.
Voy a llevar a cabo el mismo ejercicio con el Precio/m2 y compruebo si merece más la pena combinar estas dos métricas para retirar valores.

In [ ]:
# Y ahora, vamos a plotear el Precio/m2 para comparar
sns.set_theme(style="whitegrid")
plt.figure(figsize=(20, 10))
sns.color_palette("mako", as_cmap=True)
sns.histplot(df_fotocasa_10['Precio/m2'], bins=60, kde=True)
plt.title('Distribución de Precios/m2 de Venta en Barcelona', fontsize=15)
plt.xlabel('Precio por metro cuadrado (en €)')
plt.ylabel('Número de Inmuebles')

plt.show()

Voy ahora a sacar todos los **estadísticos** y además los valores del **IQR para el Precio / m2**.

In [ ]:
# Imprimo los principales estadísticos de todo el dataset para el Precio/m2:
print(f"Recuento de nuestro dataset: {df_fotocasa_10['Precio/m2'].count()}")
print(f"Promedio del Precio/m2 en nuestro dataset: {df_fotocasa_10['Precio/m2'].mean()}")
print(f"Valor Medio del Precio/m2 en nuestro dataset: {df_fotocasa_10['Precio/m2'].median()}")
print(f"Valor Modal del Precio/m2 en nuestro dataset: {df_fotocasa_10['Precio/m2'].mode()}")
print(f"Valor Mínimo del Precio/m2 en nuestro dataset: {df_fotocasa_10['Precio/m2'].min()}")
print(f"Valor Máximo del Precio/m2 en nuestro dataset: {df_fotocasa_10['Precio/m2'].max()}")
print(f"Desviación típica del Precio/m2 en nuestro dataset: {df_fotocasa_10['Precio/m2'].std()}")

In [ ]:
#Calculamos cuartiles y límites para detección de outliers
Q1_p_m2 = df_fotocasa_10['Precio/m2'].quantile(0.25)
Q2_p_m2 = df_fotocasa_10['Precio/m2'].quantile(0.50)
Q3_p_m2 = df_fotocasa_10['Precio/m2'].quantile(0.75)
IQR_p_m2 = Q3_p_m2 - Q1_p_m2 # Calculo el Rango intrecuartil
limite_inferior_p_m2 = Q1_p_m2 - 1.5 * IQR_p_m2
limite_superior_p_m2 = Q3_p_m2 + 1.5 * IQR_p_m2

print(f"Límite inferior: {limite_inferior_p_m2}€")
print(f"1er cuartil del precio - Q1 (25%): {Q1_p_m2}")
print(f"2do cuartil del precio - Q2 (50%): {Q2_p_m2}")
print(f"Valor promedio del precio: {df_fotocasa_10['Precio/m2'].mean()}")
print(f"3er cuartil del precio - Q3 (75%): {Q3_p_m2}")
print(f"Límite superior: {limite_superior_p_m2}€")
print(f"Rango intrecuartil (IQR):  {IQR_p_m2} € de rengo")

In [ ]:
# Compruebo cuanto valores hay fuera de los límites superiores
df_10__limite_sup = df_fotocasa_10[df_fotocasa_10['Precio/m2']> 14000]
df_10__limite_sup

In [ ]:
# Compruebo cuanto valores hay fuera de los límites inferiores (el calculado no es realista para este análisis, por lo que escojo un valor aproximado del que debería ser el inferior)
df_10__limite_inf = df_fotocasa_10[df_fotocasa_10['Precio/m2'] < 1000]
df_10__limite_inf

Promedio del Precio en nuestro dataset: 6.127,02 €

Valor Medio del Precio en nuestro dataset: 5.277,78 €

Valor Modal del Precio en nuestro dataset: 5.000 €

Valor Mínimo del Precio en nuestro dataset: 451,8 €

Valor Máximo del Precio en nuestro dataset: 23.831,78 €

Desviación típica del Precio en nuestro dataset: 3.411,34 €

Límite inferior (IQR): - 2.405,11 €

1er cuartil del precio - Q1 (25%): 3.678,16 €

2do cuartil del precio - Q2 (50%): 475.000 €

Valor promedio del precio: 5.277,78 €

3er cuartil del precio - Q3 (75%): 7.733,67 €

Límite superior (IQR): 13.816,94 €

Rango intercuartil (IQR):  4.055,51 € de rango


Nº de filas por encima del límite superior: **419 filas**

Nº de filas por debajo de 1.000 €: **35 filas**

Esto ya es más lógico para limpiar valores extraños, podría proceder a retirar estos mejor. Aunque quizás lo más adecuado sea una mezcla de estos valores junto con los de m2 en sí. Voy a **revisar** rápido **la distribución de m2**.

## 19.1 Analizo m² de nuevo

In [ ]:
# Empiezo por un boxplot para así buscar outliers
plt.figure(figsize=(15,5))
sns.boxplot(data=df_fotocasa_10, x='num_m2')
plt.title("Distribución de los M2")
plt.show()

Veo que pasa lo mismo que con precio, Precio/m2 y Precio/hab; nos salen demasiados valores atípicos (outliers) por arriba. De momento voy a retirar aquellos que están por encima de 600 m2 por que no tendría sentido semejantes tamaños de inmueble.

In [ ]:
df_10_m2_altos = df_fotocasa_10[df_fotocasa_10['num_m2'] > 260]
df_10_m2_altos

Previamente he dejado mucho outliers en m2 porque faltaba análisis por hacer, pero veo que dejar los que están por encima de 600 es demasiado. Además que son solo 58.

**---- NO LOS VOY A RETIRAR DE MOMENTO PARA NO VICIAR LOS CÁLCULOS -----**

Y si miro por encima de 260 (aprox el límite superior del BBoxplot) son ya 707 filas. Está claro que vamos a tener que retirar casi 1.000 filas en cualquier caso. Pero no solo por arriba, sino también por abajo.

In [ ]:
# Retiro los que están por encima de 600m2 porque son solo 58 filas.
# AL FINAL NO LOS RETIRO AQUÍ, LO HARÉ DESPUÉS
# m2_altos = df_fotocasa_10['num_m2'] > 600
# df_10_m2_corregido = df_fotocasa_10[~m2_altos].copy()
# print(f"Filas eliminadas: {m2_altos.sum()} | Filas restantes: {len(df_10_m2_corregido)}")

In [ ]:
# NOTA: en el cuaderno original esta celda pintaba 'df_10_m2_corregido', una variable que en realidad
# nunca se llega a crear (el propio comentario de la celda anterior dice que al final no se retiran
# esos m2 altos). Usamos directamente df_fotocasa_10, que es exactamente lo que el original pretendía mostrar.
plt.figure(figsize=(15,5))
sns.boxplot(data=df_fotocasa_10, x='num_m2')
plt.title("Distribución de los M2")
plt.show()

In [ ]:
# Imprimo los principales estadísticos de todo el dataset para m2:
print(f"Recuento de nuestro dataset: {df_fotocasa_10['num_m2'].count()}")
print(f"Promedio del m2 en nuestro dataset: {df_fotocasa_10['num_m2'].mean()}")
print(f"Valor Medio del m2 en nuestro dataset: {df_fotocasa_10['num_m2'].median()}")
print(f"Valor Modal del m2 en nuestro dataset: {df_fotocasa_10['num_m2'].mode()}")
print(f"Valor Mínimo del m2 en nuestro dataset: {df_fotocasa_10['num_m2'].min()}")
print(f"Valor Máximo del m2 en nuestro dataset: {df_fotocasa_10['num_m2'].max()}")
print(f"Desviación típica del m2 en nuestro dataset: {df_fotocasa_10['num_m2'].std()}")

In [ ]:
#Calculamos cuartiles y límites para detección de outliers.
Q1_m2 = df_fotocasa_10['num_m2'].quantile(0.25)
Q2_m2 = df_fotocasa_10['num_m2'].quantile(0.50)
Q3_m2 = df_fotocasa_10['num_m2'].quantile(0.75)
IQR_m2 = Q3_m2 - Q1_m2 # Calculo el Rango intrecuartil
limite_inferior_m2 = Q1_m2 - 1.5 * IQR_m2
limite_superior_m2 = Q3_m2 + 1.5 * IQR_m2

print(f"Límite inferior: {limite_inferior_m2} m2")
print(f"1er cuartil del precio - Q1 (25%): {Q1_m2} m2")
print(f"2do cuartil del precio - Q2 (50%): {Q2_m2} m2")
print(f"Valor promedio del precio: {df_fotocasa_10['num_m2'].mean()} m2")
print(f"3er cuartil del precio - Q3 (75%): {Q3_m2} m2")
print(f"Límite superior: {limite_superior_m2} m2")
print(f"Rango intrecuartil (IQR):  {IQR_m2} m2 de rango")

In [ ]:
# Compruebo cuanto valores hay fuera de los límites superiores
df_10__limite_sup = df_fotocasa_10[df_fotocasa_10['num_m2']> 250]
df_10__limite_sup

In [ ]:
# Compruebo cuanto valores hay fuera de los límites inferiores (el calculado no es realista para este análisis, por lo que escojo un valor aproximado del que debería ser el inferior)
df_10__limite_inf = df_fotocasa_10[df_fotocasa_10['num_m2'] < 25]
df_10__limite_inf

Recuento de nuestro dataset: 11.584 filas

Promedio de los M2 en nuestro dataset: 121,1 m2

Valor Medio de los M2 en nuestro dataset: 95 m2

Valor Modal de los M2 en nuestro dataset: 60 y 80 m2

Valor Mínimo de los M2 en nuestro dataset: 10 m2

Valor Máximo de los M2 en nuestro dataset: 1.000 m2

Desviación típica de los M2 en nuestro dataset: 91,76 m2

Límite inferior (IQR): - 37,5 m2

1er cuartil del precio - Q1 (25%): 69 m2

2do cuartil del precio - Q2 (50%): 95 m2

Valor promedio del precio: 121,1 m2

3er cuartil del precio - Q3 (75%): 140 m2

Límite superior (IQR): 246,5 m2

Rango intercuartil (IQR):  71 m2 de rango

Nº de filas por encima del límite superior: **772 filas**

Nº de filas por debajo de 25 m2: **44 filas**

Volvemos a ver aquí lo mismo que en precio y en Precio/m2 (como es lógico, pues uno de ellos incluye m2 en el cálculo), que hay demasiada variabilidad o dispersión de los valores. Y tenemos muchos valores fuera de los límites que se considerarían razonables o representativos para nuestra muestra.

## 19.2 Analizamos la excesiva variabilidad


Hemos visto en los estadísticos de **Precio, de m2 y de Precio/m2** conjuntamente que, si bien la varibilidad entre zonas es lógica y esperable, tenemos **demasiada desviación típica** en general y, como ya he mencionado, muchos valores fuera de los límites que se considerarían razonables. Pero se intuye que ese variabilidad seguiría siendo muy grande aunque retiraramos las filas con valores extremos que he mecionado. Hay que hacer algo con esto.

Desde el punto de vista de analizar la población del mercado de imuebles a través de nuestra muestra, quizás es más complejo el retirar muchos de estos valores porque perdemos significatividad estadística. Aunque es verdad que reducimos la variabilidad y por ello sería más representativa una inferencia.
Pero desde el punto de vista de hacer una lista de inmuebles concretos para analizar su rentabilidad, sí que es relevante quitar todos los valores atípicos de nuestra lista (ya no sería muestra) de inmuebles.
Dado que esta es la principal dirección que tomaremos en el proyecto, procederé a limpiar con estos criterios todo nuestro dataset.

**PERO LO HAREMOS CONJUNTAMENTE CON EL DE IDEALISTA PARA UNIFICAR CRITERIOS**

In [ ]:
limite_superior_p_m2

In [ ]:
# Voy a representar la distribución de Precio/m2 con la media y la mediana para tener una vista de la situación antes de continuar.
# Calculo la meida y mediana para añadirlas:
mean_precio_m2   = df_fotocasa_10['Precio/m2'].mean()
median_precio_m2 = df_fotocasa_10['Precio/m2'].median()

plt.figure(figsize=(20, 8))

# Con histplot y kde=True, Python dibuja el histograma + la curva de densidad encima
sns.histplot(data=df_fotocasa_10, x='Precio/m2', kde=True, color='steelblue')
# Añado las linea verticales de Media y Mediana:
plt.axvline(mean_precio_m2,   color='red',   linestyle='--', label=f'Media P/m2 ({mean_precio_m2:.2f})')
plt.axvline(median_precio_m2, color='green', linestyle=':',  label=f'Mediana P/m2 ({median_precio_m2:.2f})')
plt.axvline(limite_superior_p_m2, color='black', linestyle=':',  label=f'Límite Sup. P/m2 ({limite_superior_p_m2:.2f})')
# Titulando el gráfico:
plt.title('Distribución de Precio/m2 con Media y Mediana')
plt.xlabel('Precio por metro cuadrado (€)')
plt.ylabel('Número de inmuebles')
plt.legend()
plt.show()

Como aún tengo que introducir el FILTRO DE BARRIOS TURÍSTICOS a los 2 datasets de Fotocasa e Idealista, de momento dejo los valores como están y paso a sacar distritos y a hacer recuento de barrios.

# 20. Barrios y Distritos del Dataset

Ahora ya hacemos un **recuento de** cuántos anuncios **de inmuebles** tenemos **por cada barrio** en cada una de las tres ciudades. Además, vamos a sacar la lista de barrios para constrastarla con los de Idealista y poder **sacar los distritos** aquí.

Con este ejercicio, ya podremos hacer un **filtro de qué barrios** son interesantes para el **mercado de alquiler turístico/vacacional**

## 20.1 Limpieza de Barrios

In [ ]:
# Hago el recuento del nº de anuncios/inmuebles por cada DISTRITO, diferenciando por cada ciudad
for ciudad, grupo in df_fotocasa_10.groupby('ciudad'):
    print(f"\n{'--'*20}")
    print(f"  {ciudad.upper()}")
    print(f"{'--'*20}")
    print(grupo['barrio'].value_counts().to_string())

In [ ]:
# Ahora hago el recuento sin diferenciar por ciudad:
print(f"Recuento Nº inmuebles en cada ciudad: {df_fotocasa_10['ciudad'].value_counts().sort_values(ascending=False)}")

Tendría que extraer los barrios concretos de cada celda porque está mezclado con la ciudad o el municipio. Porque se me han colado otros municipios en el scraping más allá de Madrid, Valencia y Barcelona.
Tengo que utilizar la coma pero hay ejemplos donde la estrucutra del barrio no sigue la norma.

In [ ]:
# Vamos a extraer los barrios y hacer una limipieza con los municipios usando .rsplit()
# Vamos a separar por la ÚLTIMA coma (por si el barrio tiene comas internas como "Sant Pere, Sta. Caterina...")
df_fotocasa_10['barrio_2']   = df_fotocasa_10['barrio'].str.rsplit(',', n=1).str[0].str.strip()
df_fotocasa_10['municipio']  = df_fotocasa_10['barrio'].str.rsplit(',', n=1).str[1].str.strip()

# Los casos sin coma y barrio solo quedarán con municipio NaN y así los podemos evaluar aparte
df_10_municipio_nulos = df_fotocasa_10[df_fotocasa_10['municipio'].isna()]

print(f"\nNulos en municipio: {df_10_municipio_nulos['municipio'].count()}")

In [ ]:
print(df_fotocasa_10[['barrio', 'barrio_2', 'municipio']].sample(6).to_string())

In [ ]:
# Ahora hago un recuento con los municipios para ver cómo han quedado:
print(f"Recuento de municipios: {df_fotocasa_10['municipio'].value_counts().sort_values(ascending=False)}")

Por la naturaleza propia del proyecto, y porque lo hice también en Idealista, retiro directamente todo lo que no son las capitales:

- Madrid Capital - 5.296
- Barcelona Capital - 4.397
- Valencia Capital - 1.613
- **Resto - 278 --> a eliminar**
- Total - 11.584


In [ ]:
# Paso a retirar los inmuebles fuera de las capitales
capitales = ['Madrid Capital', 'Barcelona Capital', 'Valencia Capital']
municipio_resto = ~df_fotocasa_10['municipio'].isin(capitales)

df_10_sin_resto = df_fotocasa_10[~municipio_resto].copy()

print(f"Filas eliminadas: {municipio_resto.sum()} | Filas restantes: {len(df_10_sin_resto)}")

In [ ]:
# Hago otro recuento:
print(f"Recuento de municipios: {df_10_sin_resto['municipio'].value_counts().sort_values(ascending=False)}")

In [ ]:
df_fotocasa_11 = df_10_sin_resto

## 20.2 Sacamos Distritos

Ahora que ya tenemos solo resultados de Madrid, Barcelona y Valencia capital, pasamos a sacar el distrito en función de los barrios. Voy a obtener las listas de barrios por distritos para crear un código similar al que utilizamos para sacar los barrios de los datos de Idealista.

De hecho, voy a reutilizar el mismo método de diccionario y función de mapeo pero invirtiéndolo para Fotocasa.

In [ ]:
# Empezamos por el diccionario de Distritos y barrios para luego crear la función mapear.

dicc_distritos = {

    # ── BARCELONA ────────────────────────────────────────────────────
    "Ciutat Vella":         ["barri gotic", "el raval", "barceloneta", "sant pere"],
    "Eixample":             ["dreta de l'eixample", "antiga esquerra", "nova esquerra",
                             "sagrada familia", "sant antoni", "fort pienc"],
    "Sants-Montjuïc":      ["sants-badal", "sants ", "hostafrancs", "bordeta",
                             "poble sec", "font de la guatlla", "marina del port",
                             "marina del prat vermell", "zona franca"],
    "Les Corts":            ["barri de les corts", "maternitat", "pedralbes"],
    "Sarrià-Sant Gervasi":  ["sarria", "sant gervasi", "bonanova", "putget",
                             "vallvidrera", "tibidabo", "les tres torres", "ciutat jardi"],
    "Gràcia":               ["vila de gracia", "camp d'en grassot", "gracia nova",
                             "el carmel", "la salut", "el coll", "vallcarca"],
    "Horta-Guinardó":       ["el guinardo", "baix guinardo", "can baro", "turó de la peira",
                             "horta", "sant genis", "montbau", "la teixonera",
                             "font d'en fargues", "el carmel"],
    "Nou Barris":           ["vilapicina", "torre llobeta", "prosperitat", "verdum",
                             "les roquetes", "trinitat nova", "la guineueta", "canyelles",
                             "les planes", "porta", "torre baro", "ciutat meridiana",
                             "vallbona", "can peguera"],
    "Sant Andreu":          ["la sagrera", "el congrés", "navas", "sant andreu de palomar",
                             "bon pastor", "baro de viver"],
    "Sant Martí":           ["el poblenou", "diagonal mar", "vila olimpica", "el clot",
                             "camp de l'arpa", "sant marti de provencals", "verneda",
                             "el besos", "el parc i la llacuna", "provencals"],

    # ── MADRID ───────────────────────────────────────────────────────
    "Centro":               ["sol", "palacio", "embajadores", "cortes - huertas",
                             "justicia - chueca", "universidad - malasana"],
    "Arganzuela":           ["imperial", "acacias", "chopera", "legazpi",
                             "delicias", "palos de moguer", "almendrales"],
    "Retiro":               ["jeronimos", "ibiza de madrid", "nino jesus",
                             "pacifico", "adelfas", "estrella"],
    "Salamanca":            ["recoletos", "goya", "fuente del berro",
                             "guindalera", "lista", "castellana"],
    "Chamartín":            ["el viso", "prosperidad", "ciudad universitaria",
                             "hispanoamerica", "nueva españa", "castilla"],
    "Tetuán":               ["bellas vistas", "cuatro caminos", "castillejos",
                             "almenara", "valdeacederas", "berruguete"],
    "Chamberí":             ["almagro", "trafalgar", "rios rosas",
                             "arapiles", "gaztambide", "vallehermoso", "arguelles"],
    "Fuencarral-El Pardo":  ["el pardo", "fuentelarreina", "penagrande", "el pilar",
                             "la paz", "valverde", "tres olivos", "mirasierra",
                             "las tablas", "sanchinarro", "montecarmelo",
                             "valdebebas", "virgen del cortijo"],
    "Moncloa-Aravaca":      ["aravaca", "ciudad universitaria", "valdezarza",
                             "valdemarín", "la florida", "casa de campo"],
    "Latina":               ["los carmenes", "puerta del angel", "aluche",
                             "lucero", "las aguilas", "comillas",
                             "puerta bonita", "buena vista"],
    "Carabanchel":          ["san isidro", "opanel", "pradolongo", "vista alegre",
                             "buenavista", "abrantes", "comillas", "pau de carabanchel"],
    "Usera":                ["moscardo", "pradolongo", "orcasitas", "almendrales",
                             "zofio", "pradillo"],
    "Puente de Vallecas":   ["entrevias", "san diego", "palomeras bajas",
                             "palomeras sureste", "portazgo", "numancia"],
    "Moratalaz":            ["pavones", "horcajo", "marroquina",
                             "media legua", "fontarron", "vinateros"],
    "Ciudad Lineal":        ["ventas", "pueblo nuevo", "quintana", "concepcion",
                             "san pascual", "san juan bautista", "colina",
                             "atalaya", "costillares"],
    "Hortaleza":            ["palomas", "conde orgaz", "piovera", "canillas",
                             "pinar del rey", "apostol santiago", "valdebebas",
                             "valdefuentes", "corralejos", "campo de las naciones"],
    "Villaverde":           ["villaverde alto", "san andres", "san cristobal",
                             "butarque", "los rosales", "los angeles"],
    "Villa de Vallecas":    ["casco historico de vallecas", "santa eugenia",
                             "ensanche de vallecas"],
    "Vicálvaro":            ["vicalvaro", "valdebernardo", "valderribas",
                             "el canonyal", "ambroz", "los ahijones",
                             "los berrocales", "los cerros", "valdecarros"],
    "San Blas-Canillejas":  ["simancas", "hellin", "amposta", "arcos",
                             "rosas - musas", "rejas", "canillejas", "salvador"],
    "Barajas":              ["alameda de osuna", "aeropuerto",
                             "casco historico de barajas", "timon", "corralejos"],

    # ── VALENCIA ─────────────────────────────────────────────────────
    "Ciutat Vella (VLC)":   ["el mercat", "sant francesc", "el carme",
                             "la xerea", "la seu"],
    "L'Eixample (VLC)":     ["russafa", "el pla del remei", "gran via"],
    "Extramurs":            ["el botanic", "la roqueta", "la petxina", "arrancapins"],
    "Campanar":             ["barrio de campanar", "nou campanar", "tendetes",
                             "el calvari", "sant pau"],
    "La Saïdia":            ["marxalenes", "morvedre", "trinitat", "tormos"],
    "El Pla del Real":      ["exposicio", "mestalla", "jaume roig", "creu del grau"],
    "L'Olivereta":          ["nou moles", "soternes", "tres forques",
                             "la llum", "sant isidre"],
    "Patraix":              ["barrio de patraix", "sant isidre", "vara de quart",
                             "safranar", "favara"],
    "Jesús":                ["la raiosa", "hort de senabre", "creu coberta",
                             "sant marcel", "cami reial"],
    "Quatre Carreres":      ["monteolivete", "en corts", "malilla", "na rovella",
                             "la torre", "fonteta de sant lluis", "ciencias"],
    "Poblats Marítims":     ["el cabanyal", "canyamelar", "el grau",
                             "natzaret", "la malva-rosa", "el perellonet",
                             "el saler", "el palmar"],
    "Camins al Grau":       ["aiora", "albors", "la creu del grau",
                             "cami fondo", "penya - roja", "beteró"],
    "Algirós":              ["l'amistat", "la bega baixa", "la carrasca",
                             "illa perduda", "ciutat de les ciencies"],
    "Benimaclet":           ["barrio de benimaclet", "cami de vera"],
    "Rascanya":             ["els orriols", "torrefiel", "sant llorenc"],
    "Benicalap":            ["barrio de benicalap", "nou benicalap", "ciutat fallera"],
    "Pobles del Nord":      ["benimàmet", "beniferri"],
    "Pobles del Sud":       ["faitanar", "pinedo", "el palmar", "la punta",
                             "el castellar", "el forn d'alcedo"],
}
# Un toque que he metido aquí: que se diferencien los distritos llamados casi igual en Barcelona y Valencia

In [ ]:
# Primero creamos la función que saca el texto y lo limpia para poder cuadrarlo con el diccionario
import unicodedata
def limpiar_texto(texto):
    if pd.isna(texto):          # Devuelve un string vacío si el nombre es NaN para que la función no falle
        return ""

    texto = texto.lower()       # Pasamos todo el texto objetivo a minúsculas para facilitar el match con el valor en el dicionario

                                # Usamos las funciones unicode data para quitar tildes y no tener que poner 2 opciones (con y sin tildo) en el diccionario.
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)  # Descomponemos las letras con tilde para retirarlas.
        if unicodedata.category(c) != 'Mn')         # Resultado: "Gràcia" → "gracia", "Ríos" → "rios"

    return texto

In [ ]:
# Ahora ya empleamos la función para ir mapeando:
def mapear_distrito(barrio):
    texto = limpiar_texto(barrio)  # reutilizamos la función que ya teníamos

    for distrito, barrios in dicc_distritos.items():
        for b in barrios:
            if b in texto:
                return distrito
    return "Otros"

In [ ]:
# Aplicamos la función mapear al df_fotocasa_web_scraping para crear la nueva columna
df_fotocasa_11['distrito'] = df_fotocasa_11['barrio_2'].apply(mapear_distrito)

In [ ]:
# Comprobamos:
print(df_fotocasa_11['distrito'].value_counts())
print(f"\nOtros: {(df_fotocasa_11['distrito'] == 'Otros').sum()}")

## 20.3 Guardar CSV (checkpoint 5)

In [ ]:
# (5to) Guardo el csv limpio por ir segmentando el Notebook y continuar el trabajo desde aquí:
df_fotocasa_11.to_csv('datos_fotocasa_limpio_6.csv', index=False)

In [ ]:
len(df_fotocasa_10)

*(En el cuaderno original, aquí se guardaba un CSV y se volvía a cargar con `files.upload()` para poder cerrar Google Colab y continuar la sesión más tarde sin perder el trabajo. En esta versión, ejecutable de principio a fin en una sola pasada, seguimos directamente con el mismo DataFrame que ya tenemos en memoria.)*

In [ ]:
df_fotocasa_11 = df_fotocasa_11.copy()
print(f"Filas del df_fotocasa_11: {len(df_fotocasa_11)}")
print(f"Columnas del df_fotocasa_11: {df_fotocasa_11.columns}")

## 20.4 Limpiamos Distritos

In [ ]:
# Miramos ahora lo que nos ha quedado en Otros:
df_11_distritos_otros = df_fotocasa_11[df_fotocasa_11['distrito'] == 'Otros']
df_11_distritos_otros

In [ ]:
df_11_distritos_otros['barrio_2'].unique()

En este punto, ya hemos hecho en paralelo una lista objetivo de barrios con potencial para alojamientos turísticos, así que ya puedo limpiar en función de los que nos interesan. Sin embargo, para no condicionar, **los meto todos para sacar el distrito y ya después filtraré los barrios turísticos**

In [ ]:
# 2da pasada para determinar el Distrito en Fotocasa:
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'La Vall d\'Hebron',        'distrito'] = 'Horta-Guinardó'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'La Clota',                 'distrito'] = 'Horta-Guinardó'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'El Congrés i els Indians', 'distrito'] = 'Sant Andreu'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Sants',                    'distrito'] = 'Sants-Montjuïc'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'El Turó de la Peira',      'distrito'] = 'Nou Barris'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Ciudad Jardín',            'distrito'] = 'Chamartín'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Cuatro vientos',           'distrito'] = 'Latina'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Nueva España',             'distrito'] = 'Chamartín'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Valdemarín',               'distrito'] = 'Moncloa-Aravaca'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Pilar',                    'distrito'] = 'Fuencarral-El Pardo'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Campamento',               'distrito'] = 'Latina'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Orcasur - 12 de Octubre',  'distrito'] = 'Usera'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'San Fermín',               'distrito'] = 'Usera'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'El Cañaveral',             'distrito'] = 'Vicálvaro'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Beteró',                   'distrito'] = 'Camins al Grau'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Massarrojos',              'distrito'] = 'Pobles del Nord'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Mont-Olivet',              'distrito'] = 'Quatre Carreres'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'La Fontsanta',             'distrito'] = 'L\'Olivereta'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Carpesa',                  'distrito'] = 'Pobles del Nord'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Benimàmet',                'distrito'] = 'Pobles del Nord'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Ciutat Universitària',     'distrito'] = 'Les Corts'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Benifaraig',               'distrito'] = 'Pobles del Nord'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Cases de Bàrcena',         'distrito'] = 'Pobles del Nord'
df_fotocasa_11.loc[df_fotocasa_11['barrio_2'] == 'Borbotó',                  'distrito'] = 'Pobles del Nord'

print(f"Otros restantes: {(df_fotocasa_11['distrito'] == 'Otros').sum()}")

Hemos conseguido arreglar todos los que estaban en otros. Paso a hacer un nuevo recuento de Barrios y Distritos.

In [ ]:
# Comprobamos:
print(df_fotocasa_11['distrito'].value_counts())

In [ ]:
len(df_fotocasa_11)

# 21. Guardar DataFrame final de Fotocasa

Voy a extraer varios dfs que ya son finales pero con distintos conjuntos de información:

- df_final_limpio -> tenemos todas las filas (11.306) y todas las columnas que han sido limpiadas en varias fases a lo largo de este cuaderno (llegando al df_fotocasa_11).
- df_final_limpio_simple -> igual que el anterior pero retirando las columnas que ya no nos aportan ninguna información o que no le vamos a dar ningún uso: ??
- ¿OTRO? Estadísticos o dividido por precios.

In [ ]:
# descargo el primer archivo (pendiente de hacer más trabajo):
df_fotocasa_11.to_csv('df_fotocasa_final_limpio.csv', index=False)

Finalmente, guardamos ahora el df_fotocasa_web_scraping final con las columnas inutiles eliminadas

In [ ]:
df_fotocasa_11.sample(3)

In [ ]:
df_fotocasa_11.columns

In [ ]:
# Voy a quitar 'extras', 'detalles_crudo', 'link', 'descripcion',  'nombre', 'precio', 'publicacion', 'metros', 'planta'
columnas_a_quitar = ['extras', 'detalles_crudo', 'link', 'descripcion',  'nombre', 'precio', 'publicacion', 'metros', 'planta']
df_limpio_columnas_simple = df_fotocasa_11.drop(columnas_a_quitar, axis=1)

In [ ]:
df_limpio_columnas_simple

Finalmente, guardamos ahora el df_fotocasa_web_scraping limpio y con las columnas inutiles eliminadas

In [ ]:
# Descargo el otros archivo simplificado (sin columnas inútiles):
df_limpio_columnas_simple.to_csv('df_fotocasa_final_limpio_simple.csv', index=False)

## 21.1 Recupero columna Extras para el DF simple

*(En el cuaderno original, aquí se volvía a cargar `df_fotocasa_final_limpio.csv` desde cero para poder quitar las columnas de nuevo pero conservando esta vez `extras`. Como seguimos con el mismo DataFrame en memoria, `df_fotocasa_11` todavía conserva todas las columnas —incluida `extras`— desde antes del primer recorte, así que no hace falta volver a cargar nada.)*

He cargado de nuevo el df_final con todas las columnas, y ahora voy a volver a retirarlas pero dejando extras para que los detalles que aquí hay puedan tener un homólogo con los de Idealista.

In [ ]:
# Voy a quitar 'extras', 'detalles_crudo', 'link', 'descripcion',  'nombre', 'precio', 'publicacion', 'metros', 'planta'
columnas_a_quitar = ['detalles_crudo', 'link', 'descripcion',  'nombre', 'precio', 'publicacion', 'metros', 'planta']
df_limpio_columnas_simple_2 = df_fotocasa_11.drop(columnas_a_quitar, axis=1)

In [ ]:
df_limpio_columnas_simple_2

Ahora ya sí guardamos el **df_fotocasa_web_scraping limpio y con las columnas inutiles eliminadas pero dejando extras**

In [ ]:
# Descargo el otros archivo simplificado (sin columnas inútiles):
df_limpio_columnas_simple_2.to_csv('df_fotocasa_final_limpio_simple_2 (check final Claude).csv', index=False)

# **ARMONIZAR DATASETS // ARMONIZAR DATASETS**

#**Armonización Idealista y Fotocasa**

# 22. Armonización de Barrios Oficiales

En este cuaderno voy a llevar a cabo el ejercicio de **ARMONIZACIÓN** ENTRE IDEALISTA Y FOTOCASA **en 2 fases**:

- Primero vamos a **armonizar** los nombres de los **barrios** entre ambos datasets para que podemos hacer comparaciones entre ellos de cara a hacer un **filtrado conjunto** de aquellos barrios que serán objetivo de inversión por su potencial para explotar un alojamiento **turístico o vacacional**. Primero denominaremos todos los barrios con el nombre oficial de las fuentes en la web. Después introduciremos el filtro a ambos.
- La segunda parte será hacer **una pasada de Outliers en ambos** con los mismos criterios en Idealista y Fotocasa. Ha habido momentos durante la limpieza donde habría retirado muchos valores, pero prefiero hacerlo conjuntamente **para poder tener un criterio unificado**.

## 22.1 Cargar datasets (Idealista y Fotocasa ya en memoria; Inside AirBnB se sube aquí)

*(Las librerías y los 3 datasets de partida ya se cargaron en la Sección 0. Aquí simplemente continuamos con `df_idealista_web_scraping`... `df_idealista_11` (ya convertido en `df_limpio_columnas_simple`), con `df_fotocasa_web_scraping`... `df_fotocasa_11` (ya convertido en `df_limpio_columnas_simple_2`), y con el CSV en bruto de Inside AirBnB subido en la Sección 0.4.)*

Cargo el csv de **FOTOCASA**

In [ ]:
df_fotocasa = df_limpio_columnas_simple_2.copy()
print(f"Filas del df_fotocasa: {len(df_fotocasa)}")
print(f"Columnas del df_fotocasa: {df_fotocasa.columns}")

Cargo el csv de **IDEALISTA**

In [ ]:
df_idealista = df_idealista_limpio_columnas_simple.copy()
print(f"Filas del df_idealista: {len(df_idealista)}")
print(f"Columnas del df_idealista: {df_idealista.columns}")

Cargo el csv de **INSIDE AIRBNB**

In [ ]:
df_inside_airbnb = pd.read_csv(archivo_inside_airbnb)
print(f"Filas del df_inside_airbnb: {len(df_inside_airbnb)}")
print(f"Columnas del df_inside_airbnb: {df_inside_airbnb.columns}")

## 22.2 Nombres Oficiales en Fotocasa

Vamos a empezar por cuadrar los nombres de los barrios en Fotocasa y meter en una columna llamada **"barrio_oficial"**

- El diccionario completo BARRIO_FC_A_OFICIAL con todas las claves exactas tal como vienen en el CSV (incluyendo espacios extra, tildes distintas, sufijos de ciudad).
- La función .map para hacer un mapeo en nuestro df con los que hemos previamente creado en el diccionario.
- Finalmente guardaremos el df para poder reutilizarlo.

In [ ]:
df_fotocasa.sample(4)

### 22.2.1 Diccionario Barrio Oficial Fotocasa

DICCIONARIO DE MAPEO

Clave → valor exacto de la columna 'barrio_2' en el CSV de Fotocasa

Valor → nombre oficial del barrio (nomenclatura municipal sacada de Wikipedia)

None → barrio sin equivalente oficial (PAU, agrupación propia, etc.)

In [ ]:

BARRIO_FC_A_OFICIAL = {
    # MADRID
    'Abrantes':                         'Abrantes',
    'Acacias':                          'Acacias',
    'Adelfas':                          'Adelfas',
    'Aeropuerto':                       'Aeropuerto',
    'Alameda de Osuna':                 'Alameda de Osuna',
    'Almagro':                          'Almagro',
    'Almendrales':                      'Almendrales',
    'Aluche':                           'Aluche',
    'Amposta':                          'Amposta',
    'Apóstol Santiago':                 'Apóstol Santiago',
    'Arapiles':                         'Arapiles',
    'Aravaca':                          'Aravaca',
    'Arcos':                            'Arcos',
    'Argüelles':                        'Argüelles',
    'Atalaya':                          'Atalaya',
    'Bellas Vistas':                    'Bellas Vistas',
    'Berruguete':                       'Berruguete',
    'Butarque':                         'Butarque',
    'Campamento':                       'Campamento',
    'Canillas':                         'Canillas',
    'Canillejas':                       'Canillejas',
    'Casa de Campo':                    'Casa de Campo',
    'Casco Histórico de Barajas':       'Casco Histórico de Barajas',
    'Casco Histórico de Vallecas':      'Casco Histórico de Vallecas',
    'Casco histórico de Vicálvaro':     'Casco Histórico de Vicálvaro',
    'Castellana':                       'Castellana',
    'Castilla':                         'Castilla',
    'Chopera':                          'Chopera',
    'Ciudad Jardín':                    'Ciudad Jardín',
    'Ciudad Universitaria':             'Ciudad Universitaria',
    'Colina':                           'Colina',
    'Comillas':                         'Comillas',
    'Concepción':                       'Concepción',
    'Costillares':                      'Costillares',
    'Delicias':                         'Delicias',
    'El Cañaveral':                     'El Cañaveral',
    'El Pardo':                         'El Pardo',
    'El Viso':                          'El Viso',
    'Entrevías':                        'Entrevías',
    'Estrella':                         'Estrella',
    'Fontarrón':                        'Fontarrón',
    'Fuente del Berro':                 'Fuente del Berro',
    'Fuentelarreina':                   'Fuentelarreina',
    'Gaztambide':                       'Gaztambide',
    'Goya':                             'Goya',
    'Guindalera':                       'Guindalera',
    'Hellín':                           'Hellín',
    'Horcajo':                          'Horcajo',
    'Imperial':                         'Imperial',
    'Jerónimos':                        'Jerónimos',
    'La Paz':                           'La Paz',
    'Las Águilas':                      'Las Águilas',
    'Legazpi':                          'Legazpi',
    'Lista':                            'Lista',
    'Los Cármenes':                     'Los Cármenes',
    'Los Rosales':                      'Los Rosales',
    'Los Ángeles':                      'Los Ángeles',
    'Lucero':                           'Lucero',
    'Marroquina':                       'Marroquina',
    'Media Legua':                      'Media Legua',
    'Mirasierra':                       'Mirasierra',
    'Moscardó':                         'Moscardó',
    'Niño Jesús':                       'Niño Jesús',
    'Nueva España':                     'Nueva España',
    'Numancia':                         'Numancia',
    'Opañel':                           'Opañel',
    'Orcasitas':                        'Orcasitas',
    'Pacífico':                         'Pacífico',
    'Palacio':                          'Palacio',
    'Palomas':                          'Palomas',
    'Palomeras Bajas':                  'Palomeras Bajas',
    'Palomeras Sureste':                'Palomeras Sureste',
    'Pavones':                          'Pavones',
    'Peñagrande':                       'Peñagrande',
    'Pilar':                            'Pilar',
    'Pinar del Rey':                    'Pinar del Rey',
    'Portazgo':                         'Portazgo',
    'Pradolongo':                       'Pradolongo',
    'Prosperidad':                      'Prosperidad',
    'Pueblo Nuevo':                     'Pueblo Nuevo',
    'Puerta Bonita':                    'Puerta Bonita',
    'Puerta del Ángel':                 'Puerta del Ángel',
    'Quintana':                         'Quintana',
    'Recoletos':                        'Recoletos',
    'Rejas':                            'Rejas',
    'Salvador':                         'Salvador',
    'San Cristóbal':                    'San Cristóbal',
    'San Diego':                        'San Diego',
    'San Fermín':                       'San Fermín',
    'San Isidro':                       'San Isidro',
    'San Juan Bautista':                'San Juan Bautista',
    'San Pascual':                      'San Pascual',
    'Santa Eugenia':                    'Santa Eugenia',
    'Simancas':                         'Simancas',
    'Sol':                              'Sol',
    'Timón':                            'Timón',
    'Trafalgar':                        'Trafalgar',
    'Valdeacederas':                    'Valdeacederas',
    'Valdemarín':                       'Valdemarín',
    'Valdezarza':                       'Valdezarza',
    'Vallehermoso':                     'Vallehermoso',
    'Ventas':                           'Ventas',
    'Vinateros':                        'Vinateros',
    'Vista Alegre':                     'Vista Alegre',

    # MADRID – requieren traducción
    'Almenara -Ventilla':               'Almenara',
    'Buena Vista':                      'Buenavista',
    'Castillejos - Cuzco':              'Castillejos',
    'Conde Orgaz - Piovera':            'Piovera',
    'Corralejos - Campo de las Naciones': 'Corralejos',
    'Cortes - Huertas':                 'Cortes',
    'Cuatro Caminos - Azca':            'Cuatro Caminos',
    'Cuatro vientos':                   'Cuatro Vientos',
    'Embajadores - Lavapiés':           'Embajadores',
    'Ensanche de Vallecas - La Gavia':  'Ensanche de Vallecas',
    'Hispanoamérica - Bernabéu':        'Hispanoamérica',
    'Ibiza de Madrid':                  'Ibiza',
    'Justicia - Chueca':                'Justicia',
    'La Florida - El Plantío':          'El Plantío',
    'Orcasur - 12 de Octubre':          'Orcasur',
    'Palos de Moguer':                  'Palos de la Frontera',
    'Ríos Rosas - Nuevos Ministerios':  'Ríos Rosas',
    'Rosas - Musas':                    'Rosas',
    'Tres Olivos - Valverde':           'Valverde',
    'Universidad - Malasaña':           'Universidad',
    'Valdebebas - Valdefuentes':        'Valdefuentes',
    'Valdebernardo - Valderribas':      'Valdebernardo',
    'Zofio':                            'Zofío',

    # MADRID – sin equivalente oficial (PAUs, agrupaciones)
    'Ambroz':                           None,
    'Las Tablas':                       None,
    'Los Ahijones':                     None,
    'Los Berrocales':                   None,
    'Los Cerros':                       None,
    'Montecarmelo':                     None,
    'PAU de Carabanchel':               None,
    'Sanchinarro':                      None,
    'Valdecarros':                      None,
    'Villaverde Alto':                  None,
    'Virgen del Cortijo - Manoteras':   None,

    # BARCELONA – coinciden exactamente
    'Baró de Viver':                    'Baró de Viver',
    'Can Baró':                         'Can Baró',
    'Can Peguera':                      'Can Peguera',
    'Canyelles':                        'Canyelles',
    'Ciutat Meridiana':                 'Ciutat Meridiana',
    "Dreta de l'Eixample":              "Dreta de l'Eixample",
    'El Baix Guinardó':                 'El Baix Guinardó',
    'El Bon Pastor':                    'El Bon Pastor',
    "El Camp d'en Grassot i Gràcia Nova": "El Camp d'en Grassot i Gràcia Nova",
    "El Camp de l'Arpa del Clot":       "El Camp de l'Arpa del Clot",
    'El Carmel':                        'El Carmel',
    'El Clot':                          'El Clot',
    'El Coll':                          'El Coll',
    'El Congrés i els Indians':         'El Congrés i els Indians',
    'El Guinardó':                      'El Guinardó',
    'El Parc i la Llacuna del Poblenou':'El Parc i la Llacuna del Poblenou',
    'El Poblenou':                      'El Poblenou',
    'El Raval':                         'El Raval',
    'El Turó de la Peira':              'El Turó de la Peira',
    'Horta':                            'Horta',
    'Hostafrancs':                      'Hostafrancs',
    'La Barceloneta':                   'La Barceloneta',
    'La Bordeta':                       'La Bordeta',
    'La Clota':                         'La Clota',
    "La Font d'en Fargues":             "La Font d'en Fargues",
    'La Font de la Guatlla':            'La Font de la Guatlla',
    'La Guineueta':                     'La Guineueta',
    'La Marina del Prat Vermell':       'La Marina del Prat Vermell',
    'La Maternitat i Sant Ramon':       'La Maternitat i Sant Ramon',
    'La Prosperitat':                   'La Prosperitat',
    'La Sagrera':                       'La Sagrera',
    'La Salut':                         'La Salut',
    'La Teixonera':                     'La Teixonera',
    'La Trinitat Nova':                 'La Trinitat Nova',
    "La Vall d'Hebron":                 "La Vall d'Hebron",
    'La Verneda i la Pau':              'La Verneda i la Pau',
    'La Vila Olímpica del Poblenou':    'La Vila Olímpica del Poblenou',
    'Les Roquetes':                     'Les Roquetes',
    'Les Tres Torres':                  'Les Tres Torres',
    'Montbau':                          'Montbau',
    'Navas':                            'Navas',
    'Pedralbes':                        'Pedralbes',
    'Porta':                            'Porta',
    'Provençals del Poblenou':          'Provençals del Poblenou',
    'Sagrada Família':                  'Sagrada Família',
    'Sant Andreu de Palomar':           'Sant Andreu de Palomar',
    'Sant Antoni':                      'Sant Antoni',
    'Sant Genís dels Agudells':         'Sant Genís dels Agudells',
    'Sant Martí de Provençals':         'Sant Martí de Provençals',
    'Sants':                            'Sants',
    'Sants-Badal':                      'Sants-Badal',
    'Sarrià':                           'Sarrià',
    'Torre Baró':                       'Torre Baró',
    'Vallcarca i els Penitents':        'Vallcarca i els Penitents',
    'Vilapicina i la Torre Llobeta':    'Vilapicina i la Torre Llobeta',
    'Vila de Gràcia':                   'Vila de Gràcia',

    # BARCELONA – requieren traducción
    'Barri Gòtic':                      'El Gòtic',
    'Barri de les Corts':               'Les Corts',
    'El Besós i el Maresme':            'El Besòs i el Maresme',
    'El Poble Sec - Parc de Montjuïc':  'El Poble-sec',
    'El Putget i el Farró':             'El Putxet i el Farró',
    'Fort Pienc':                       'El Fort Pienc',
    "L'Antiga Esquerra de l'Eixample":  "Antiga Esquerra de l'Eixample",
    'La Marina del Port':               'La Zona Franca - Port',
    "La Nova Esquerra de l'Eixample":   "Nova Esquerra de l'Eixample",
    'Sant Gervasi i la Bonanova':       'Sant Gervasi - la Bonanova',
    'Sant Gervasi- Galvany':            'Sant Gervasi - Galvany',
    'Sant Pere, Sta. Caterina i la Ribera': 'Sant Pere, Santa Caterina i la Ribera',
    'Trinitat Vella':                   'La Trinitat Vella',
    'Vallvidrera - Tibidabo - Les Planes': 'Vallvidrera, el Tibidabo i les Planes',
    'Verdum':                           'Verdun',

    # VALENCIA – coinciden exactamente
    'Aiora':                            'Aiora',
    'Albors':                           'Albors',
    "L'Amistat":                        "L'Amistat",
    'Arrancapins':                      'Arrancapins',
    'Benifaraig':                       'Benifaraig',
    'Beniferri':                        'Beniferri',
    'Benimàmet':                        'Benimàmet',
    'Beteró':                           'Beteró',
    'Borbotó':                          'Borbotó',
    'Camí Fondo':                       'Camí Fondo',
    'Camí Reial':                       'Camí Reial',
    'Camí de Vera':                     'Camí de Vera',
    'Carpesa':                          'Carpesa',
    'Cases de Bàrcena':                 'Cases de Bàrcena',
    'Ciutat Fallera':                   'Ciutat Fallera',
    'Ciutat Jardí':                     'Ciutat Jardí',
    'Ciutat Universitària':             'Ciutat Universitària',
    'El Botànic':                       'El Botànic',
    'El Cabanyal - El Canyamelar':      'El Cabanyal - El Canyamelar',
    'El Calvari':                       'El Calvari',
    'El Carme':                         'El Carme',
    "El Forn d'Alcedo":                 "El Forn d'Alcedo",
    'El Grau':                          'El Grau',
    'El Mercat':                        'El Mercat',
    'El Palmar':                        'El Palmar',
    'El Perellonet':                    'El Perellonet',
    'El Pilar':                         'El Pilar',
    'El Pla del Remei':                 'El Pla del Remei',
    'El Saler':                         'El Saler',
    'Els Orriols':                      'Els Orriols',
    'Exposició':                        'Exposició',
    'Faitanar':                         'Faitanar',
    'Fonteta de Sant Lluís':            'Fonteta de Sant Lluís',
    'Gran Via':                         'Gran Via',
    'Jaume Roig':                       'Jaume Roig',
    "L'Hort de Senabre":               "L'Hort de Senabre",
    "L'Illa Perduda":                   "L'Illa Perduda",
    'La Carrasca':                      'La Carrasca',
    'La Creu Coberta':                  'La Creu Coberta',
    'La Creu del Grau':                 'La Creu del Grau',
    'La Fontsanta':                     'La Fontsanta',
    'La Llum':                          'La Llum',
    'La Malva-rosa':                    'La Malva-rosa',
    'La Petxina':                       'La Petxina',
    'La Punta':                         'La Punta',
    'La Roqueta':                       'La Roqueta',
    'La Seu':                           'La Seu',
    'La Torre':                         'La Torre',
    'La Xerea':                         'La Xerea',
    'Malilla':                          'Malilla',
    'Marxalenes':                       'Marxalenes',
    'Massarrojos':                      'Massarrojos',
    'Mestalla':                         'Mestalla',
    'Mont-Olivet':                      'Mont-Olivet',
    'Morvedre':                         'Morvedre',
    'Natzaret':                         'Natzaret',
    'Nou Moles':                        'Nou Moles',
    'Pinedo':                           'Pinedo',
    'Russafa':                          'Russafa',
    'Safranar':                         'Safranar',
    'Sant Francesc':                    'Sant Francesc',
    'Sant Isidre':                      'Sant Isidre',
    'Sant Pau':                         'Sant Pau',
    'Soternes':                         'Soternes',
    'Tormos':                           'Tormos',
    'Torrefiel':                        'Torrefiel',
    'Tres Forques':                     'Tres Forques',
    'Trinitat':                         'Trinitat',
    'Vara de Quart':                    'Vara de Quart',

    # VALENCIA – requieren traducción
    'Barrio de Benicalap':              'Benicalap',
    'Barrio de Benimaclet':             'Benimaclet',
    'Barrio de Campanar':               'Campanar',
    'Barrio de Patraix':                'Patraix',
    'Ciutat de les Ciències i de les Arts - Justicia': 'Ciutat de les Arts i les Ciències',
    "El Castellar i l'Oliverar":        "El Castellar - L'Oliveral",
    'En Corts - Doctor Waksman':        'En Corts',
    'La Bega Baixa - Plaza Xúquer':     'La Bega Baixa',
    'La Raïosa':                        'La Raiosa',
    'Les Tendetes - Avenida Burjassot': 'Les Tendetes',
    'Na Rovella - Hermanos Maristas':   'Na Rovella',
    'Penya - Roja - Avda. Francia':     'Penya-Roja',
    'Sant Antoni':                      'Sant Antoni',
    'Sant Llorenç - Zona Alfahuir':     'Sant Llorenç',
    'Sant Marcel.lí':                   'Sant Marcel·lí',

    # VALENCIA – sin equivalente oficial
    'Favara':                           None,
    'Nou Benicalap':                    None,
    'Nou Campanar':                     None,
}


### 22.2.2 Función mapeo Barrio oficial (Fotocasa)

In [ ]:
df_fotocasa['barrio_oficial'] = df_fotocasa['barrio_2'].map(BARRIO_FC_A_OFICIAL)

In [ ]:
df_fotocasa.sample(10)

In [ ]:
df_fotocasa['barrio_oficial'].unique()

In [ ]:
# Reviso los que he metido en "None":
df_fotocasa_none = df_fotocasa[df_fotocasa['barrio_oficial'] == 'None']
df_fotocasa_none

In [ ]:
# Comprobamos:
print(df_fotocasa['barrio_oficial'].value_counts().to_string())
print(f"\nOtros: {(df_fotocasa['barrio_oficial'] == 'None').sum()}")

In [ ]:
# Los que pusimos en None en el diccionario, los retiramos como nulos:
nulos_barrio = df_fotocasa['barrio_oficial'].isna()
df_fotocasa = df_fotocasa[~nulos_barrio].copy()
print(f"Filas eliminadas: {nulos_barrio.sum()} | Filas restantes: {len(df_fotocasa)}")

Una vez mapeado el barrio oficial, retiramos los nulos que no se han podido mapear:

**Filas eliminadas: 242 | Filas restantes: 11.064**

In [ ]:
# Guardamos el csv para poder seguir trabajando después
df_fotocasa.to_csv('datos_fotocasa_limpio_barrio_oficial.csv', index=False)

## 22.3 Nombres Oficiales en Idealista

Vamos a empezar por cuadrar los nombres de los barrios en Idealista y meter en una columna llamada **"barrio_oficial"**

- El diccionario completo BARRIO_FC_A_OFICIAL con todas las claves exactas tal como vienen en el CSV (incluyendo espacios extra, tildes distintas, sufijos de ciudad).
- La función .map para hacer un mapeo en nuestro df con los que hemos previamente creado en el diccionario.
- Finalmente guardaremos el df para poder reutilizarlo.

In [ ]:
df_idealista.sample(5)

### 22.3.1 Diccionario Barrio Oficial Idealista

DICCIONARIO DE MAPEO

Clave → valor exacto de la columna 'barrio' en el CSV de Idealista

Valor → nombre oficial del barrio (nomenclatura municipal sacada de Wikipedia)

None → barrio sin equivalente oficial (PAU, agrupación propia, etc.)

In [ ]:

BARRIO_ID_A_OFICIAL = {

    # ── MADRID – coinciden exactamente ───────────────────────
    'Acacias':                          'Acacias',
    'Adelfas':                          'Adelfas',
    'Aeropuerto':                       'Aeropuerto',
    'Alameda de Osuna':                 'Alameda de Osuna',
    'Almagro':                          'Almagro',
    'Almenara':                         'Almenara',
    'Almendrales':                      'Almendrales',
    'Amposta':                          'Amposta',
    'Apóstol Santiago':                 'Apóstol Santiago',
    'Arapiles':                         'Arapiles',
    'Arcos':                            'Arcos',
    'Atalaya':                          'Atalaya',
    'Bellas Vistas':                    'Bellas Vistas',
    'Berruguete':                       'Berruguete',
    'Butarque':                         'Butarque',
    'Canillas':                         'Canillas',
    'Canillejas':                       'Canillejas',
    'Casco Histórico de Barajas':       'Casco Histórico de Barajas',
    'Casco Histórico de Vallecas':      'Casco Histórico de Vallecas',
    'Casco Histórico de Vicálvaro':     'Casco Histórico de Vicálvaro',
    'Castellana':                       'Castellana',
    'Castilla':                         'Castilla',
    'Castillejos':                      'Castillejos',
    'Chopera':                          'Chopera',
    'Ciudad Jardín':                    'Ciudad Jardín',
    'Colina':                           'Colina',
    'Concepción':                       'Concepción',
    'Corralejos':                       'Corralejos',
    'Cortes':                           'Cortes',
    'Costillares':                      'Costillares',
    'Cuatro Caminos':                   'Cuatro Caminos',
    'Delicias':                         'Delicias',
    'El Cañaveral':                     'El Cañaveral',
    'El Pardo':                         'El Pardo',
    'El Viso':                          'El Viso',
    'Embajadores':                      'Embajadores',
    'Entrevías':                        'Entrevías',
    'Estrella':                         'Estrella',
    'Fontarrón':                        'Fontarrón',
    'Fuente del Berro':                 'Fuente del Berro',
    'Fuentelarreina':                   'Fuentelarreina',
    'Gaztambide':                       'Gaztambide',
    'Goya':                             'Goya',
    'Guindalera':                       'Guindalera',
    'Hellín':                           'Hellín',
    'Hispanoamérica':                   'Hispanoamérica',
    'Horcajo':                          'Horcajo',
    'Ibiza':                            'Ibiza',
    'Imperial':                         'Imperial',
    'Jerónimos':                        'Jerónimos',
    'La Paz':                           'La Paz',
    'Legazpi':                          'Legazpi',
    'Lista':                            'Lista',
    'Los Rosales':                      'Los Rosales',
    'Los Ángeles':                      'Los Ángeles',
    'Marroquina':                       'Marroquina',
    'Media Legua':                      'Media Legua',
    'Mirasierra':                       'Mirasierra',
    'Moscardó':                         'Moscardó',
    'Niño Jesús':                       'Niño Jesús',
    'Numancia':                         'Numancia',
    'Orcasitas':                        'Orcasitas',
    'Pacífico':                         'Pacífico',
    'Palacio':                          'Palacio',
    'Palomas':                          'Palomas',
    'Palomeras Bajas':                  'Palomeras Bajas',
    'Palomeras Sureste':                'Palomeras Sureste',
    'Palos de la Frontera':             'Palos de la Frontera',
    'Pavones':                          'Pavones',
    'Peñagrande':                       'Peñagrande',
    'Pinar del Rey':                    'Pinar del Rey',
    'Piovera':                          'Piovera',
    'Portazgo':                         'Portazgo',
    'Pradolongo':                       'Pradolongo',
    'Prosperidad':                      'Prosperidad',
    'Pueblo Nuevo':                     'Pueblo Nuevo',
    'Quintana':                         'Quintana',
    'Recoletos':                        'Recoletos',
    'Rejas':                            'Rejas',
    'Ríos Rosas':                       'Ríos Rosas',
    'Rosas':                            'Rosas',
    'Salvador':                         'Salvador',
    'San Cristóbal':                    'San Cristóbal',
    'San Diego':                        'San Diego',
    'San Fermín':                       'San Fermín',
    'San Juan Bautista':                'San Juan Bautista',
    'San Pascual':                      'San Pascual',
    'Santa Eugenia':                    'Santa Eugenia',
    'Simancas':                         'Simancas',
    'Sol':                              'Sol',
    'Timón':                            'Timón',
    'Trafalgar':                        'Trafalgar',
    'Valdeacederas':                    'Valdeacederas',
    'Valdebernardo':                    'Valdebernardo',
    'Valdefuentes':                     'Valdefuentes',
    'Vallehermoso':                     'Vallehermoso',
    'Valverde':                         'Valverde',
    'Ventas':                           'Ventas',
    'Vinateros':                        'Vinateros',
    'Zofío':                            'Zofío',

    # ── MADRID – requieren traducción ────────────────────────
    '12 de Octubre - Orcasur':          'Orcasur',
    'Chueca - Justicia':                'Justicia',
    'Ensanche de Vallecas':             'Ensanche de Vallecas',  # sin "- La Gavia"
    'Malasaña - Universidad':           'Universidad',
    'Las Tablas':                       None,   # PAU Fuencarral
    'Los Ahijones':                     None,   # PAU Vicálvaro
    'Los Berrocales':                   None,   # PAU Vicálvaro
    'Los Cerros':                       None,   # PAU Vicálvaro
    'Montecarmelo':                     None,   # PAU Fuencarral
    'Ambroz':                           None,   # PAU Vicálvaro
    'Sanchinarro':                      None,   # zona Hortaleza sin oficial
    'Valdecarros':                      None,   # PAU Villa de Vallecas
    'Villaverde Alto':                  None,   # nombre popular
    'Virgen del Cortijo - Manoteras':   None,   # agrupación propia

    # ── BARCELONA – coinciden exactamente ────────────────────
    'Can Baró':                         'Can Baró',
    'Can Peguera - El Turó de la Peira':'Can Peguera',   # Idealista agrupa; oficial es Can Peguera
    'El Bon Pastor':                    'El Bon Pastor',
    'El Clot':                          'El Clot',
    'El Coll':                          'El Coll',
    'El Congrés i els Indians':         'El Congrés i els Indians',
    'El Fort Pienc':                    'El Fort Pienc',
    'El Guinardó':                      'El Guinardó',
    'El Gòtic':                         'El Gòtic',
    'El Poblenou':                      'El Poblenou',
    'El Poble Sec - Parc de Montjuïc':  'El Poble-sec',
    'El Putxet i el Farró':             'El Putxet i el Farró',
    'El Raval':                         'El Raval',
    'Horta':                            'Horta',
    'Hostafrancs':                      'Hostafrancs',
    'La Barceloneta':                   'La Barceloneta',
    'La Bordeta':                       'La Bordeta',
    'La Font de la Guatlla':            'La Font de la Guatlla',
    'La Marina del Prat Vermell':       'La Marina del Prat Vermell',
    'La Maternitat i Sant Ramon':       'La Maternitat i Sant Ramon',
    'La Prosperitat':                   'La Prosperitat',
    'La Sagrera':                       'La Sagrera',
    'La Salut':                         'La Salut',
    'La Teixonera':                     'La Teixonera',
    'La Verneda i la Pau':              'La Verneda i la Pau',
    'Les Corts (barrio)':               'Les Corts',
    'Les Roquetes':                     'Les Roquetes',
    'Les Tres Torres':                  'Les Tres Torres',
    'Navas':                            'Navas',
    'Pedralbes':                        'Pedralbes',
    'Porta':                            'Porta',
    'Sant Antoni':                      'Sant Antoni',
    'Sant Gervasi - Galvany':           'Sant Gervasi - Galvany',
    'Sant Gervasi - La Bonanova':       'Sant Gervasi - la Bonanova',
    'Sant Martí de Provençals':         'Sant Martí de Provençals',
    'Sants':                            'Sants',
    'Sants - Badal':                    'Sants-Badal',
    'Sarrià':                           'Sarrià',
    'Vallcarca i els Penitents':        'Vallcarca i els Penitents',
    'Vila de Gràcia':                   'Vila de Gràcia',
    'Verdun':                           'Verdun',

    # ── BARCELONA – requieren traducción ─────────────────────
    "L'Antiga Esquerra de l'Eixample":  "Antiga Esquerra de l'Eixample",
    'Ciutat Meridiana - Torre Baró - Vallbona': 'Ciutat Meridiana',   # Idealista agrupa 3 barrios
    "La Dreta de l'Eixample":           "Dreta de l'Eixample",
    "La Font d'En Fargues":             "La Font d'en Fargues",
    'La Marina del Port':               'La Zona Franca - Port',
    'La Sagrada Família':               'Sagrada Família',
    "La Nova Esquerra de l'Eixample":   "Nova Esquerra de l'Eixample",
    'Sant Andreu (barrio)':             'Sant Andreu de Palomar',
    'Sant Genís Dels Agudells - Montbau': 'Sant Genís dels Agudells',  # Idealista agrupa 2
    'Sant Pere - Santa Caterina i la Ribera': 'Sant Pere, Santa Caterina i la Ribera',
    'Vallvidrera - El Tibidabo i les Planes': 'Vallvidrera, el Tibidabo i les Planes',
    'Zona Franca - Port':               'La Zona Franca - Port',
    'El Besòs':                         'El Besòs i el Maresme',      # Idealista omite "i el Maresme"

    # ── VALENCIA – coinciden exactamente ─────────────────────
    'Aiora':                            'Aiora',
    'Albors':                           'Albors',
    "L'Amistat":                        "L'Amistat",
    'Arrancapins':                      'Arrancapins',
    'Benicalap':                        'Benicalap',
    'Beniferri':                        'Beniferri',
    'Benimaclet':                       'Benimaclet',
    'Beteró':                           'Beteró',
    'Campanar':                         'Campanar',
    'Camí Fondo':                       'Camí Fondo',
    'Camí Reial':                       'Camí Reial',
    'Camí de Vera':                     'Camí de Vera',
    'Ciutat Fallera':                   'Ciutat Fallera',
    'Ciutat Jardí':                     'Ciutat Jardí',
    "El Castellar - L'Oliveral":        "El Castellar - L'Oliveral",
    'El Calvari':                       'El Calvari',
    'El Grau':                          'El Grau',
    'El Mercat':                        'El Mercat',
    'El Pilar':                         'El Pilar',
    'El Pla del Remei':                 'El Pla del Remei',
    'El Botànic':                       'El Botànic',
    'En Corts':                         'En Corts',
    'Exposició':                        'Exposició',
    'Faitanar':                         'Faitanar',
    'Jaume Roig':                       'Jaume Roig',
    "L'Hort de Senabre":               "L'Hort de Senabre",
    "L'Illa Perduda":                   "L'Illa Perduda",
    'La Carrasca':                      'La Carrasca',
    'La Creu Coberta':                  'La Creu Coberta',
    'La Petxina':                       'La Petxina',
    'La Punta':                         'La Punta',
    'La Raiosa':                        'La Raiosa',
    'La Roqueta':                       'La Roqueta',
    'La Seu':                           'La Seu',
    'La Torre':                         'La Torre',
    'La Xerea':                         'La Xerea',
    'Les Tendetes':                     'Les Tendetes',
    'Malilla':                          'Malilla',
    'Mestalla':                         'Mestalla',
    'Mont-Olivet':                      'Mont-Olivet',
    'Na Rovella':                       'Na Rovella',
    'Natzaret':                         'Natzaret',
    'Nou Moles':                        'Nou Moles',
    'Patraix':                          'Patraix',
    'Russafa':                          'Russafa',
    'Safranar':                         'Safranar',
    'Sant Isidre':                      'Sant Isidre',
    'Sant Llorenç':                     'Sant Llorenç',
    'Sant Marcel·lí':                   'Sant Marcel·lí',
    'Sant Pau':                         'Sant Pau',
    'Soternes':                         'Soternes',
    'Torrefiel':                        'Torrefiel',
    'Tres Forques':                     'Tres Forques',
    'Trinitat':                         'Trinitat',
    'Vara de Quart':                    'Vara de Quart',

    # ── VALENCIA – requieren traducción ──────────────────────
    'Benimamet':                        'Benimàmet',        # falta tilde
    'City of Arts':                     'Ciutat de les Arts i les Ciències',
    'Creu del Grau':                    'La Creu del Grau',
    'El Cabanyal':                      'El Cabanyal - El Canyamelar',
    'El Carmen':                        'El Carme',         # castellano vs valenciano
    'Gran Vía':                         'Gran Via',         # tilde en Idealista
    'La Fonteta':                       'Fonteta de Sant Lluís',
    'La Vega Baixa':                    'La Bega Baixa',    # "Vega" vs "Bega"
    'Orriols':                          'Els Orriols',
    'Penya-roja':                       'Penya-Roja',       # mayúscula
    'Playa de la Malvarrosa':           'La Malva-rosa',
    'Sant Isidre (Olivereta)':          'Sant Isidre',      # Idealista distingue; oficial es único
    'Zona Franca - Port':               'La Zona Franca - Port',  # duplicado por si aparece en BCN/VLC

    # ── VALENCIA – sin equivalente oficial ───────────────────
    'Favara':                           None,   # sin barrio oficial propio

    # ── DESCARTADOS / CAJÓN DE SASTRE ────────────────────────
    'Otros':                            None,   # etiqueta genérica de Idealista
}


### 22.3.2 Función mapeo Barrio oficial (Idealista)

In [ ]:
df_idealista['barrio_oficial'] = df_idealista['barrio'].map(BARRIO_ID_A_OFICIAL)

In [ ]:
df_idealista.sample(10)

In [ ]:
df_idealista['barrio_oficial'].unique()

In [ ]:
# Reviso los que he metido en "None":
df_idealista_none = df_idealista[df_idealista['barrio_oficial'].isna()]
df_idealista_none

In [ ]:
# Comprobamos:
print(df_idealista['barrio_oficial'].value_counts().to_string())
print(f"\nOtros: {(df_idealista['barrio_oficial'].isna()).sum()}")

In [ ]:
# Los que pusimos en None en el diccionario, los retiramos como nulos:
nulos_barrio_ID = df_idealista['barrio_oficial'].isna()
df_idealista = df_idealista[~nulos_barrio_ID].copy()
print(f"Filas eliminadas: {nulos_barrio_ID.sum()} | Filas restantes: {len(df_idealista)}")

Una vez mapeado el barrio oficial, retiramos los nulos que no se han podido mapear:

**Filas eliminadas: 1.089 | Filas restantes: 24.499**

In [ ]:
# Guardamos el csv para poder seguir trabajando después
df_idealista.to_csv('datos_idealista_limpio_barrio_oficial.csv', index=False)

## 22.4 Nombres Oficiales en Inside AirBnB

Vamos a empezar por cuadrar los nombres de los barrios en Inside AirBnB y meter en una columna llamada **"barrio_oficial"**

- El diccionario completo BARRIO_AB_A_OFICIAL con todas las claves exactas tal como vienen en el CSV (incluyendo espacios extra, tildes distintas, sufijos de ciudad).
- La función .map para hacer un mapeo en nuestro df con los que hemos previamente creado en el diccionario.
- Finalmente guardaremos el df para poder reutilizarlo.

In [ ]:
df_inside_airbnb.sample(5)

### 22.4.1 Diccionario Barrio Oficial AirBnB

DICCIONARIO DE MAPEO

Clave → valor exacto de la columna 'barrio' en el CSV de Inside AirBnB

Valor → nombre oficial del barrio (nomenclatura municipal sacada de Wikipedia)

None → barrio sin equivalente oficial (PAU, agrupación propia, etc.)

In [ ]:


BARRIO_AB_A_OFICIAL = {

    # ── MADRID – coinciden exactamente ───────────────────────
    'Abrantes':                     'Abrantes',
    'Acacias':                      'Acacias',
    'Adelfas':                      'Adelfas',
    'Aeropuerto':                   'Aeropuerto',
    'Alameda de Osuna':             'Alameda de Osuna',
    'Almagro':                      'Almagro',
    'Almenara':                     'Almenara',
    'Almendrales':                  'Almendrales',
    'Aluche':                       'Aluche',
    'Amposta':                      'Amposta',
    'Arapiles':                     'Arapiles',
    'Aravaca':                      'Aravaca',
    'Arcos':                        'Arcos',
    'Argüelles':                    'Argüelles',
    'Atalaya':                      'Atalaya',
    'Bellas Vistas':                'Bellas Vistas',
    'Berruguete':                   'Berruguete',
    'Buenavista':                   'Buenavista',
    'Butarque':                     'Butarque',
    'Campamento':                   'Campamento',
    'Canillas':                     'Canillas',
    'Canillejas':                   'Canillejas',
    'Casa de Campo':                'Casa de Campo',
    'Casco Histórico de Barajas':   'Casco Histórico de Barajas',
    'Casco Histórico de Vallecas':  'Casco Histórico de Vallecas',
    'Casco Histórico de Vicálvaro': 'Casco Histórico de Vicálvaro',
    'Castellana':                   'Castellana',
    'Castilla':                     'Castilla',
    'Castillejos':                  'Castillejos',
    'Chopera':                      'Chopera',
    'Ciudad Jardín':                'Ciudad Jardín',
    'Ciudad Universitaria':         'Ciudad Universitaria',
    'Colina':                       'Colina',
    'Comillas':                     'Comillas',
    'Concepción':                   'Concepción',
    'Corralejos':                   'Corralejos',
    'Cortes':                       'Cortes',
    'Costillares':                  'Costillares',
    'Cuatro Caminos':               'Cuatro Caminos',
    'Cuatro Vientos':               'Cuatro Vientos',
    'Delicias':                     'Delicias',
    'El Pardo':                     'El Pardo',
    'El Plantío':                   'El Plantío',
    'El Viso':                      'El Viso',
    'Embajadores':                  'Embajadores',
    'Entrevías':                    'Entrevías',
    'Estrella':                     'Estrella',
    'Fontarrón':                    'Fontarrón',
    'Fuente del Berro':             'Fuente del Berro',
    'Gaztambide':                   'Gaztambide',
    'Goya':                         'Goya',
    'Guindalera':                   'Guindalera',
    'Hellín':                       'Hellín',
    'Hispanoamérica':               'Hispanoamérica',
    'Horcajo':                      'Horcajo',
    'Ibiza':                        'Ibiza',
    'Imperial':                     'Imperial',
    'Jerónimos':                    'Jerónimos',
    'Justicia':                     'Justicia',
    'La Paz':                       'La Paz',
    'Legazpi':                      'Legazpi',
    'Lista':                        'Lista',
    'Los Rosales':                  'Los Rosales',
    'Lucero':                       'Lucero',
    'Marroquina':                   'Marroquina',
    'Media Legua':                  'Media Legua',
    'Mirasierra':                   'Mirasierra',
    'Moscardó':                     'Moscardó',
    'Niño Jesús':                   'Niño Jesús',
    'Nueva España':                 'Nueva España',
    'Numancia':                     'Numancia',
    'Opañel':                       'Opañel',
    'Orcasitas':                    'Orcasitas',
    'Orcasur':                      'Orcasur',
    'Pacífico':                     'Pacífico',
    'Palacio':                      'Palacio',
    'Palomas':                      'Palomas',
    'Palomeras Bajas':              'Palomeras Bajas',
    'Palomeras Sureste':            'Palomeras Sureste',
    'Pavones':                      'Pavones',
    'Peñagrande':                   'Peñagrande',
    'Pilar':                        'Pilar',
    'Pinar del Rey':                'Pinar del Rey',
    'Piovera':                      'Piovera',
    'Portazgo':                     'Portazgo',
    'Pradolongo':                   'Pradolongo',
    'Prosperidad':                  'Prosperidad',
    'Pueblo Nuevo':                 'Pueblo Nuevo',
    'Puerta Bonita':                'Puerta Bonita',
    'Quintana':                     'Quintana',
    'Recoletos':                    'Recoletos',
    'Rejas':                        'Rejas',
    'Rosas':                        'Rosas',
    'Salvador':                     'Salvador',
    'San Diego':                    'San Diego',
    'San Fermín':                   'San Fermín',
    'San Isidro':                   'San Isidro',
    'San Juan Bautista':            'San Juan Bautista',
    'San Pascual':                  'San Pascual',
    'Santa Eugenia':                'Santa Eugenia',
    'Simancas':                     'Simancas',
    'Sol':                          'Sol',
    'Timón':                        'Timón',
    'Trafalgar':                    'Trafalgar',
    'Universidad':                  'Universidad',
    'Valdeacederas':                'Valdeacederas',
    'Valdefuentes':                 'Valdefuentes',
    'Valdemarín':                   'Valdemarín',
    'Valdezarza':                   'Valdezarza',
    'Vallehermoso':                 'Vallehermoso',
    'Valverde':                     'Valverde',
    'Ventas':                       'Ventas',
    'Vinateros':                    'Vinateros',
    'Vista Alegre':                 'Vista Alegre',
    'Zofío':                        'Zofío',

    # ── MADRID – requieren traducción ────────────────────────
    'Aguilas':                      'Las Águilas',      # falta "Las" y tilde
    'Apostol Santiago':             'Apóstol Santiago', # falta tilde
    'Atocha':                       'Palos de la Frontera',  # AirBnB usa nombre de la estación
    'Cármenes':                     'Los Cármenes',     # falta "Los"
    'El Goloso':                    'El Goloso',        # pedanía de Fuencarral, sin barrio oficial propio → None
    'Fuentelareina':                'Fuentelarreina',   # errata ortográfica
    'Los Angeles':                  'Los Ángeles',      # falta tilde
    'Palos de Moguer':              'Palos de la Frontera',  # nombre popular
    'Puerta del Angel':             'Puerta del Ángel', # falta tilde
    'Rios Rosas':                   'Ríos Rosas',       # falta tilde
    'San Andrés':                   'San Andrés',       # no aparece en oficial Madrid → None
    'San Cristobal':                'San Cristóbal',    # falta tilde

    # MADRID – sin equivalente oficial
    'Ambroz':                       None,
    'El Goloso':                    None,   # pedanía de Fuencarral
    'San Andrés':                   None,   # no es barrio oficial en Madrid capital

    # ── VALENCIA – en MAYÚSCULAS en AirBnB ───────────────────
    'AIORA':                        'Aiora',
    'ALBORS':                       'Albors',
    'ARRANCAPINS':                  'Arrancapins',
    'BENICALAP':                    'Benicalap',
    'BENIFERRI':                    'Beniferri',
    'BENIMACLET':                   'Benimaclet',
    'BENIMAMET':                    'Benimàmet',
    'BETERO':                       'Beteró',
    'BORBOTO':                      'Borbotó',
    'CABANYAL-CANYAMELAR':          'El Cabanyal - El Canyamelar',
    'CAMI DE VERA':                 'Camí de Vera',
    'CAMI FONDO':                   'Camí Fondo',
    'CAMI REAL':                    'Camí Reial',
    'CAMPANAR':                     'Campanar',
    "CASTELLAR-L'OLIVERAL":         "El Castellar - L'Oliveral",
    'CIUTAT DE LES ARTS I DE LES CIENCIES': 'Ciutat de les Arts i les Ciències',
    'CIUTAT FALLERA':               'Ciutat Fallera',
    'CIUTAT JARDI':                 'Ciutat Jardí',
    'CIUTAT UNIVERSITARIA':         'Ciutat Universitària',
    'EL BOTANIC':                   'El Botànic',
    'EL CALVARI':                   'El Calvari',
    'EL CARME':                     'El Carme',
    "EL FORN D'ALCEDO":             "El Forn d'Alcedo",
    'EL GRAU':                      'El Grau',
    'EL MERCAT':                    'El Mercat',
    'EL PALMAR':                    'El Palmar',
    'EL PERELLONET':                'El Perellonet',
    'EL PILAR':                     'El Pilar',
    'EL PLA DEL REMEI':             'El Pla del Remei',
    'EL SALER':                     'El Saler',
    'ELS ORRIOLS':                  'Els Orriols',
    'EN CORTS':                     'En Corts',
    'EXPOSICIO':                    'Exposició',
    'FAITANAR':                     'Faitanar',
    'FAVARA':                       None,   # sin barrio oficial propio
    'JAUME ROIG':                   'Jaume Roig',
    "L'AMISTAT":                    "L'Amistat",
    "L'HORT DE SENABRE":            "L'Hort de Senabre",
    "L'ILLA PERDUDA":               "L'Illa Perduda",
    'LA CARRASCA':                  'La Carrasca',
    'LA CREU COBERTA':              'La Creu Coberta',
    'LA CREU DEL GRAU':             'La Creu del Grau',
    'LA FONTETA S.LLUIS':           'Fonteta de Sant Lluís',
    'LA FONTSANTA':                 'La Fontsanta',
    'LA GRAN VIA':                  'Gran Via',
    'LA LLUM':                      'La Llum',
    'LA MALVA-ROSA':                'La Malva-rosa',
    'LA PETXINA':                   'La Petxina',
    'LA PUNTA':                     'La Punta',
    'LA RAIOSA':                    'La Raiosa',
    'LA ROQUETA':                   'La Roqueta',
    'LA SEU':                       'La Seu',
    'LA TORRE':                     'La Torre',
    'LA VEGA BAIXA':                'La Bega Baixa',
    'LA XEREA':                     'La Xerea',
    'LES TENDETES':                 'Les Tendetes',
    'MAHUELLA-TAULADELLA':          None,   # pedanía rural, sin barrio oficial
    'MALILLA':                      'Malilla',
    'MARXALENES':                   'Marxalenes',
    'MASSARROJOS':                  'Massarrojos',
    'MESTALLA':                     'Mestalla',
    'MONT-OLIVET':                  'Mont-Olivet',
    'MORVEDRE':                     'Morvedre',
    'NA ROVELLA':                   'Na Rovella',
    'NATZARET':                     'Natzaret',
    'NOU MOLES':                    'Nou Moles',
    'PATRAIX':                      'Patraix',
    'PENYA-ROJA':                   'Penya-Roja',
    'PINEDO':                       'Pinedo',
    'POBLE NOU':                    'Poble Nou',
    'RUSSAFA':                      'Russafa',
    'SAFRANAR':                     'Safranar',
    'SANT ANTONI':                  'Sant Antoni',
    'SANT FRANCESC':                'Sant Francesc',
    'SANT ISIDRE':                  'Sant Isidre',
    'SANT LLORENS':                 'Sant Llorenç',
    'SANT MARCEL.LI':               'Sant Marcel·lí',
    'SANT PAU':                     'Sant Pau',
    'SOTERNES':                     'Soternes',
    'TORMOS':                       'Tormos',
    'TORREFIEL':                    'Torrefiel',
    'TRES FORQUES':                 'Tres Forques',
    'TRINITAT':                     'Trinitat',
    'VARA DE QUART':                'Vara de Quart',
}



### 22.4.2 Función mapeo Barrio oficial (AirBnB)

In [ ]:
df_inside_airbnb['barrio_oficial'] = df_inside_airbnb['barrio'].map(BARRIO_AB_A_OFICIAL)

In [ ]:
df_inside_airbnb.sample(10)

In [ ]:
df_inside_airbnb['barrio_oficial'].unique()

In [ ]:
# Reviso los que he metido en "None":
df_airbnb_none = df_inside_airbnb[df_inside_airbnb['barrio_oficial'].isna()]
df_airbnb_none

In [ ]:
# Comprobamos:
print(df_inside_airbnb['barrio_oficial'].value_counts().to_string())
print(f"\nOtros: {(df_inside_airbnb['barrio_oficial'].isna()).sum()}")

In [ ]:
# Los que pusimos en None en el diccionario, los retiramos como nulos:
nulos_barrio_AB = df_inside_airbnb['barrio_oficial'].isna()
df_inside_airbnb = df_inside_airbnb[~nulos_barrio_AB].copy()
print(f"Filas eliminadas: {nulos_barrio_AB.sum()} | Filas restantes: {len(df_inside_airbnb)}")

Una vez mapeado el barrio oficial, retiramos los nulos que no se han podido mapear:

**Filas eliminadas: 93 | Filas restantes: 24.153**

In [ ]:
# Guardamos el csv para poder seguir trabajando después
df_inside_airbnb.to_csv('datos_airbnb_limpio_barrio_oficial.csv', index=False)

*(Nota de reordenación: en el cuaderno original, esta tabla de barrios se construía **después** de pasar Inside AirBnB a `id_barrio_oficial`, y para ese paso intermedio se recargaba una versión de la tabla ya guardada de una sesión anterior. Como aquí ejecutamos todo de un tirón sin sesiones previas, adelantamos la construcción de la tabla a este punto — el resultado es exactamente el mismo, solo cambia el orden de ejecución para que todo funcione en una sola pasada.)*

# 23. Tabla Índice de Barrios Oficiales y Turísticos

Vamos a crear ahora una tabla que usaremos como dimensión en Power BI pero también para filtrar en este cuaderno. Aquí recogeremos todos los **barrios con sus nombres oficiales y además sí son considerados turísticos** por nuestro propio análisis - Consultar el archivo "Tabla Barrios - armonizar turísticos".
Primero creamos la tabla en formato df de pandas **combinando los diccionarios** creados para mapear los barrios. Añadimos un Índice que usaremos a modo de código o ID.

## 23.1 Creamos la tabla

In [ ]:
# Unión de todos los barrios oficiales de las tres fuentes
todos_los_barrios = sorted(
    {v for v in BARRIO_FC_A_OFICIAL.values() if v is not None} |
    {v for v in BARRIO_ID_A_OFICIAL.values() if v is not None} |
    {v for v in BARRIO_AB_A_OFICIAL.values() if v is not None})
# Creamos el nuevo df para contener el índice de barrios para poder trabajar más adelante con ello
df_barrios_oficial = pd.DataFrame({
    'id_barrio_oficial': range(1, len(todos_los_barrios) + 1),
    'barrio_oficial':    todos_los_barrios
})

NameError: name 'BARRIO_FC_A_OFICIAL' is not defined

In [ ]:
df_barrios_oficial

## 23.2 Barrio Turístico (1/0)

Ahora ya pasamos a crear la columna que identifica si el barrio está dentro de nuestro objetivo por su potencial de alojamientos turísticos o vacacionales, usando el mismo método de toda esta seccion --> diccionario + mapear

In [ ]:
# Creamos el diccionario barrio_oficial en modo booleano → 1 turístico / 0 no turístico

BARRIO_TURISTICO = {
        'Acacias': 1, 'Aiora': 1, 'Almagro': 1, "Antiga Esquerra de l'Eixample": 1,
    'Arapiles': 1, 'Aravaca': 1, 'Arcos': 1, 'Argüelles': 1, 'Arrancapins': 1,
    'Atalaya': 1, 'Bellas Vistas': 1, 'Benicalap': 1, 'Beniferri': 1,
    'Benimaclet': 1, 'Benimàmet': 1, 'Berruguete': 1, 'Beteró': 1, 'Butarque': 1,
    'Campamento': 1, 'Campanar': 1, 'Camí Fondo': 1, 'Camí Reial': 1,
    'Camí de Vera': 1, 'Canillas': 1, 'Canillejas': 1, 'Casa de Campo': 1,
    'Casco Histórico de Barajas': 1, 'Castellana': 1, 'Castilla': 1,
    'Castillejos': 1, 'Chopera': 1, 'Ciudad Jardín': 1, 'Ciudad Universitaria': 1,
    'Ciutat Fallera': 1, 'Ciutat Jardí': 1, 'Ciutat Universitària': 1,
    'Ciutat de les Arts i les Ciències': 1, 'Colina': 1, 'Comillas': 1,
    'Concepción': 1, 'Corralejos': 1, 'Cortes': 1, 'Costillares': 1,
    'Cuatro Caminos': 1, 'Cuatro Vientos': 1, 'Delicias': 1,
    "Dreta de l'Eixample": 1, 'El Botànic': 1, 'El Cabanyal - El Canyamelar': 1,
    'El Calvari': 1, 'El Carme': 1, "El Castellar - L'Oliveral": 1,
    "El Forn d'Alcedo": 1, 'El Fort Pienc': 1, 'El Grau': 1, 'El Gòtic': 1,
    'El Mercat': 1, 'El Parc i la Llacuna del Poblenou': 1, 'El Pilar': 1,
    'El Pla del Remei': 1, 'El Poble-sec': 1, 'El Poblenou': 1, 'El Raval': 1,
    'El Viso': 1, 'Embajadores': 1, 'Estrella': 1, 'Exposició': 1,
    'Fuente del Berro': 1, 'Goya': 1, 'Gran Via': 1, 'Guindalera': 1,
    'Hostafrancs': 1, 'Ibiza': 1, 'Jaume Roig': 1, 'Jerónimos': 1,
    'Justicia': 1, 'La Barceloneta': 1, 'La Font de la Guatlla': 1,
    'La Malva-rosa': 1, 'La Roqueta': 1, 'La Seu': 1,
    'La Vila Olímpica del Poblenou': 1, 'La Xerea': 1, 'Lista': 1,
    'Mestalla': 1, 'Morvedre': 1, 'Niño Jesús': 1,
    "Nova Esquerra de l'Eixample": 1, 'Pacífico': 1, 'Palacio': 1,
    'Palos de la Frontera': 1, 'Recoletos': 1, 'Russafa': 1, 'Ríos Rosas': 1,
    'Sagrada Família': 1, 'Sant Antoni': 1, 'Sant Francesc': 1,
    'Sant Pere, Santa Caterina i la Ribera': 1, 'Sants': 1, 'Sol': 1,
    'Trafalgar': 1, 'Universidad': 1, 'Vallehermoso': 1,
    'Ciutat Universitària': 1, 'Exposició': 1, 'Jaume Roig': 1,}

df_barrios_oficial['barrio_turistico_objetivo'] = (
    df_barrios_oficial['barrio_oficial'].map(BARRIO_TURISTICO).fillna(0).astype(int))

In [ ]:
# Compruebo que los barrios están bien mapeados:
barrios_turist = df_barrios_oficial[df_barrios_oficial['barrio_turistico_objetivo'] == 1]
print(barrios_turist['barrio_oficial'].unique())

In [ ]:
df_barrios_oficial

Tenemos todos los barrios correctamente mapeados ahora. Descargo el df como csv para nuestro análisis.

In [ ]:
# Guardamos el csv para poder seguir trabajando después
df_barrios_oficial.to_csv('tabla_barrios_oficial_turísticos.csv', index=False)

*(Seguimos con `df_barrios_oficial` ya en memoria, sin recargarlo desde el CSV que se acaba de guardar.)*

## 23.3 Mejoramos ID y añadimos Ciudad, Distrito

Ahora vamos a añadir a nuestro dataset de "df_barrios_oficial_turístico" tanto la ciudad como el distrito también con el nombre oficial.
Que siga ordenado alfabéticamente por los barrios oficiales. Es clave este orden, porque **se me ha ocurrido un código o id de barrio aún mejor**: Para tener un código más útil, creo que estaría bien **que combinara un número para la ciudad, otro para el distrito y otro para el barrio**. El formato sería **C/DD/BBB** para comprender todos los números que tenemos.

Para las ciudades: Madrid el 1, Barcelona el 2 y Valencia el 3.

Luego los distritos ordenados alfabéticamente (Algirós es 01, Algirós es 02, …, Villaverde es 50).

Y finalmente el barrio que marca el orden alfabético general para el código. Todos los códigos o ids tienen que tener el mismo número de cifras, de ahí lo de 01, 02, etc.

- Ejemplo 1: Madrid, Carabanchel, Abrantes --> 108001
- Ejemplo 2: Barcelona, Carabanchel, La Teixonera --> 219158


## 23.4 Diccionarios de las zonas

REcurro al mismo método, ordenando la información de barrios, distritos y ciudades en diccionarios que luego pasaré a mapear en nuestro df

In [ ]:

# DICCIONARIO 1: Ciudades
codigo_ciudad = {
    'Madrid':    1,
    'Barcelona': 2,
    'Valencia':  3,
}

# DICCIONARIO 2: Distritos
codigo_distrito = {
    'Algirós':               '01',
    'Arganzuela':            '02',
    'Barajas':               '03',
    'Benicalap':             '04',
    'Benimaclet':            '05',
    'Camins al Grau':        '06',
    'Campanar':              '07',
    'Carabanchel':           '08',
    'Centro':                '09',
    'Chamartín':             '10',
    'Chamberí':              '11',
    'Ciudad Lineal':         '12',
    'Ciutat Vella':          '13',   # Barcelona
    'Ciutat Vella (Valencia)': '14', # Valencia
    'El Pla del Real':       '15',
    'Extramurs':             '16',
    'Fuencarral-El Pardo':   '17',
    'Gràcia':                '18',
    'Horta-Guinardó':        '19',
    'Hortaleza':             '20',
    'Jesús':                 '21',
    'La Saïdia':             '22',
    'Latina':                '23',
    "L'Eixample":            '24',   # Barcelona
    "L'Eixample (Valencia)": '25',   # Valencia
    'Les Corts':             '26',
    "L'Olivereta":           '27',
    'Moncloa-Aravaca':       '28',
    'Moratalaz':             '29',
    'Nou Barris':            '30',
    'Patraix':               '31',
    "Poblats de l'Oest":     '32',
    'Poblats del Nord':      '33',
    'Poblats del Sud':       '34',
    'Poblats Marítims':      '35',
    'Puente de Vallecas':    '36',
    'Quatre Carreres':       '37',
    'Rascanya':              '38',
    'Retiro':                '39',
    'Salamanca':             '40',
    'San Blas-Canillejas':   '41',
    'Sant Andreu':           '42',
    'Sant Martí':            '43',
    'Sants-Montjuïc':        '44',
    'Sarrià-Sant Gervasi':   '45',
    'Tetuán':                '46',
    'Usera':                 '47',
    'Vicálvaro':             '48',
    'Villa de Vallecas':     '49',
    'Villaverde':            '50',
}

# DICCIONARIO 3: Barrios --> ciudad, distrito
# Ordenado alfabéticamente. El código de barrio (BBB) se asigna automáticamente según la posición en este orden.
codigo_barrio_ = {
    'Abrantes':                                 ('Madrid',    'Carabanchel'),
    'Acacias':                                  ('Madrid',    'Arganzuela'),
    'Adelfas':                                  ('Madrid',    'Retiro'),
    'Aeropuerto':                               ('Madrid',    'Barajas'),
    'Aiora':                                    ('Valencia',  'Camins al Grau'),
    'Alameda de Osuna':                         ('Madrid',    'Barajas'),
    'Albors':                                   ('Valencia',  'Camins al Grau'),
    'Almagro':                                  ('Madrid',    'Chamberí'),
    'Almenara':                                 ('Madrid',    'Tetuán'),
    'Almendrales':                              ('Madrid',    'Usera'),
    'Aluche':                                   ('Madrid',    'Latina'),
    'Amposta':                                  ('Madrid',    'San Blas-Canillejas'),
    "Antiga Esquerra de l'Eixample":            ('Barcelona', "L'Eixample"),
    'Apóstol Santiago':                         ('Madrid',    'Hortaleza'),
    'Arapiles':                                 ('Madrid',    'Chamberí'),
    'Aravaca':                                  ('Madrid',    'Moncloa-Aravaca'),
    'Arcos':                                    ('Madrid',    'San Blas-Canillejas'),
    'Argüelles':                                ('Madrid',    'Moncloa-Aravaca'),
    'Arrancapins':                              ('Valencia',  'Extramurs'),
    'Atalaya':                                  ('Madrid',    'Ciudad Lineal'),
    'Baró de Viver':                            ('Barcelona', 'Sant Andreu'),
    'Bellas Vistas':                            ('Madrid',    'Tetuán'),
    'Benicalap':                                ('Valencia',  'Benicalap'),
    'Benifaraig':                               ('Valencia',  'Poblats del Nord'),
    'Beniferri':                                ("Valencia",  "Poblats de l'Oest"),
    'Benimaclet':                               ('Valencia',  'Benimaclet'),
    'Benimàmet':                                ("Valencia",  "Poblats de l'Oest"),
    'Berruguete':                               ('Madrid',    'Tetuán'),
    'Beteró':                                   ('Valencia',  'Poblats Marítims'),
    'Borbotó':                                  ('Valencia',  'Poblats del Nord'),
    'Buenavista':                               ('Madrid',    'Carabanchel'),
    'Butarque':                                 ('Madrid',    'Villaverde'),
    'Campamento':                               ('Madrid',    'Latina'),
    'Campanar':                                 ('Valencia',  'Campanar'),
    'Camí Fondo':                               ('Valencia',  'Camins al Grau'),
    'Camí Reial':                               ('Valencia',  'Jesús'),
    'Camí de Vera':                             ('Valencia',  'Benimaclet'),
    'Can Baró':                                 ('Barcelona', 'Gràcia'),
    'Can Peguera':                              ('Barcelona', 'Nou Barris'),
    'Canillas':                                 ('Madrid',    'Hortaleza'),
    'Canillejas':                               ('Madrid',    'San Blas-Canillejas'),
    'Canyelles':                                ('Barcelona', 'Nou Barris'),
    'Carpesa':                                  ('Valencia',  'Poblats del Nord'),
    'Casa de Campo':                            ('Madrid',    'Moncloa-Aravaca'),
    'Casco Histórico de Barajas':               ('Madrid',    'Barajas'),
    'Casco Histórico de Vallecas':              ('Madrid',    'Puente de Vallecas'),
    'Casco Histórico de Vicálvaro':             ('Madrid',    'Vicálvaro'),
    'Cases de Bàrcena':                         ('Valencia',  'Poblats del Nord'),
    'Castellana':                               ('Madrid',    'Salamanca'),
    'Castilla':                                 ('Madrid',    'Chamartín'),
    'Castillejos':                              ('Madrid',    'Tetuán'),
    'Chopera':                                  ('Madrid',    'Arganzuela'),
    'Ciudad Jardín':                            ('Madrid',    'Chamartín'),
    'Ciudad Universitaria':                     ('Madrid',    'Moncloa-Aravaca'),
    'Ciutat Fallera':                           ('Valencia',  'Benicalap'),
    'Ciutat Jardí':                             ('Valencia',  'Algirós'),
    'Ciutat Meridiana':                         ('Barcelona', 'Nou Barris'),
    'Ciutat Universitària':                     ('Valencia',  'Camins al Grau'),
    'Ciutat de les Arts i les Ciències':        ('Valencia',  'Quatre Carreres'),
    'Colina':                                   ('Madrid',    'Ciudad Lineal'),
    'Comillas':                                 ('Madrid',    'Carabanchel'),
    'Concepción':                               ('Madrid',    'Ciudad Lineal'),
    'Corralejos':                               ('Madrid',    'Barajas'),
    'Cortes':                                   ('Madrid',    'Centro'),
    'Costillares':                              ('Madrid',    'Ciudad Lineal'),
    'Cuatro Caminos':                           ('Madrid',    'Tetuán'),
    'Cuatro Vientos':                           ('Madrid',    'Latina'),
    'Delicias':                                 ('Madrid',    'Arganzuela'),
    "Dreta de l'Eixample":                      ('Barcelona', "L'Eixample"),
    'El Baix Guinardó':                         ('Barcelona', 'Horta-Guinardó'),
    'El Besòs i el Maresme':                    ('Barcelona', 'Sant Martí'),
    'El Bon Pastor':                            ('Barcelona', 'Sant Andreu'),
    'El Botànic':                               ('Valencia',  'Extramurs'),
    'El Cabanyal - El Canyamelar':              ('Valencia',  'Poblats Marítims'),
    'El Calvari':                               ('Valencia',  'Campanar'),
    "El Camp d'en Grassot i Gràcia Nova":       ('Barcelona', 'Gràcia'),
    "El Camp de l'Arpa del Clot":               ('Barcelona', 'Sant Martí'),
    'El Carme':                                 ('Valencia',  'Ciutat Vella (Valencia)'),
    'El Carmel':                                ('Barcelona', 'Horta-Guinardó'),
    "El Castellar - L'Oliveral":                ('Valencia',  'Poblats del Sud'),
    'El Cañaveral':                             ('Madrid',    'Vicálvaro'),
    'El Clot':                                  ('Barcelona', 'Sant Martí'),
    'El Coll':                                  ('Barcelona', 'Gràcia'),
    'El Congrés i els Indians':                 ('Barcelona', 'Sant Andreu'),
    "El Forn d'Alcedo":                         ('Valencia',  'Poblats del Sud'),
    'El Fort Pienc':                            ('Barcelona', "L'Eixample"),
    'El Grau':                                  ('Valencia',  'Poblats Marítims'),
    'El Guinardó':                              ('Barcelona', 'Horta-Guinardó'),
    'El Gòtic':                                 ('Barcelona', 'Ciutat Vella'),
    'El Mercat':                                ('Valencia',  'Ciutat Vella (Valencia)'),
    'El Palmar':                                ('Valencia',  'Poblats del Sud'),
    'El Parc i la Llacuna del Poblenou':        ('Barcelona', 'Sant Martí'),
    'El Pardo':                                 ('Madrid',    'Fuencarral-El Pardo'),
    'El Perellonet':                            ('Valencia',  'Poblats del Sud'),
    'El Pilar':                                 ('Valencia',  'Ciutat Vella (Valencia)'),
    'El Pla del Remei':                         ('Valencia',  "L'Eixample (Valencia)"),
    'El Plantío':                               ('Madrid',    'Moncloa-Aravaca'),
    'El Poble-sec':                             ('Barcelona', 'Sants-Montjuïc'),
    'El Poblenou':                              ('Barcelona', 'Sant Martí'),
    'El Putxet i el Farró':                     ('Barcelona', 'Sarrià-Sant Gervasi'),
    'El Raval':                                 ('Barcelona', 'Ciutat Vella'),
    'El Saler':                                 ('Valencia',  'Poblats del Sud'),
    'El Turó de la Peira':                      ('Barcelona', 'Nou Barris'),
    'El Viso':                                  ('Madrid',    'Chamartín'),
    'Els Orriols':                              ('Valencia',  'Rascanya'),
    'Embajadores':                              ('Madrid',    'Centro'),
    'En Corts':                                 ('Valencia',  'Quatre Carreres'),
    'Ensanche de Vallecas':                     ('Madrid',    'Villa de Vallecas'),
    'Entrevías':                                ('Madrid',    'Puente de Vallecas'),
    'Estrella':                                 ('Madrid',    'Retiro'),
    'Exposició':                                ('Valencia',  "L'Eixample (Valencia)"),
    'Faitanar':                                 ('Valencia',  'Poblats del Sud'),
    'Fontarrón':                                ('Madrid',    'Moratalaz'),
    'Fonteta de Sant Lluís':                    ('Valencia',  'Quatre Carreres'),
    'Fuente del Berro':                         ('Madrid',    'Salamanca'),
    'Fuentelarreina':                           ('Madrid',    'Fuencarral-El Pardo'),
    'Gaztambide':                               ('Madrid',    'Chamberí'),
    'Goya':                                     ('Madrid',    'Salamanca'),
    'Gran Via':                                 ('Valencia',  "L'Eixample (Valencia)"),
    'Guindalera':                               ('Madrid',    'Salamanca'),
    'Hellín':                                   ('Madrid',    'San Blas-Canillejas'),
    'Hispanoamérica':                           ('Madrid',    'Chamartín'),
    'Horcajo':                                  ('Madrid',    'Moratalaz'),
    'Horta':                                    ('Barcelona', 'Horta-Guinardó'),
    'Hostafrancs':                              ('Barcelona', 'Sants-Montjuïc'),
    'Ibiza':                                    ('Madrid',    'Retiro'),
    'Imperial':                                 ('Madrid',    'Arganzuela'),
    'Jaume Roig':                               ('Valencia',  "L'Eixample (Valencia)"),
    'Jerónimos':                                ('Madrid',    'Retiro'),
    'Justicia':                                 ('Madrid',    'Centro'),
    "L'Amistat":                                ('Valencia',  'Algirós'),
    "L'Hort de Senabre":                        ('Valencia',  'Jesús'),
    "L'Illa Perduda":                           ('Valencia',  'Algirós'),
    'La Barceloneta':                           ('Barcelona', 'Ciutat Vella'),
    'La Bega Baixa':                            ('Valencia',  'Quatre Carreres'),
    'La Bordeta':                               ('Barcelona', 'Sants-Montjuïc'),
    'La Carrasca':                              ('Valencia',  'Algirós'),
    'La Clota':                                 ('Barcelona', 'Horta-Guinardó'),
    'La Creu Coberta':                          ('Valencia',  'Jesús'),
    'La Creu del Grau':                         ('Valencia',  'Camins al Grau'),
    "La Font d'en Fargues":                     ('Barcelona', 'Horta-Guinardó'),
    'La Font de la Guatlla':                    ('Barcelona', 'Sants-Montjuïc'),
    'La Fontsanta':                             ("Valencia",  "L'Olivereta"),
    'La Guineueta':                             ('Barcelona', 'Nou Barris'),
    'La Llum':                                  ("Valencia",  "L'Olivereta"),
    'La Malva-rosa':                            ('Valencia',  'Poblats Marítims'),
    'La Marina del Prat Vermell':               ('Barcelona', 'Sants-Montjuïc'),
    'La Maternitat i Sant Ramon':               ('Barcelona', 'Les Corts'),
    'La Paz':                                   ('Madrid',    'Fuencarral-El Pardo'),
    'La Petxina':                               ('Valencia',  'Extramurs'),
    'La Prosperitat':                           ('Barcelona', 'Nou Barris'),
    'La Punta':                                 ('Valencia',  'Quatre Carreres'),
    'La Raiosa':                                ('Valencia',  'Jesús'),
    'La Roqueta':                               ('Valencia',  'Ciutat Vella (Valencia)'),
    'La Sagrera':                               ('Barcelona', 'Sant Andreu'),
    'La Salut':                                 ('Barcelona', 'Gràcia'),
    'La Seu':                                   ('Valencia',  'Ciutat Vella (Valencia)'),
    'La Teixonera':                             ('Barcelona', 'Horta-Guinardó'),
    'La Torre':                                 ('Valencia',  'Poblats del Sud'),
    'La Trinitat Nova':                         ('Barcelona', 'Nou Barris'),
    'La Trinitat Vella':                        ('Barcelona', 'Sant Andreu'),
    "La Vall d'Hebron":                         ('Barcelona', 'Horta-Guinardó'),
    'La Verneda i la Pau':                      ('Barcelona', 'Sant Martí'),
    'La Vila Olímpica del Poblenou':            ('Barcelona', 'Sant Martí'),
    'La Xerea':                                 ('Valencia',  'Ciutat Vella (Valencia)'),
    'La Zona Franca - Port':                    ('Barcelona', 'Sants-Montjuïc'),
    'Las Águilas':                              ('Madrid',    'Latina'),
    'Legazpi':                                  ('Madrid',    'Arganzuela'),
    'Les Corts':                                ('Barcelona', 'Les Corts'),
    'Les Roquetes':                             ('Barcelona', 'Nou Barris'),
    'Les Tendetes':                             ('Valencia',  'La Saïdia'),
    'Les Tres Torres':                          ('Barcelona', 'Sarrià-Sant Gervasi'),
    'Lista':                                    ('Madrid',    'Salamanca'),
    'Los Cármenes':                             ('Madrid',    'Latina'),
    'Los Rosales':                              ('Madrid',    'Villaverde'),
    'Los Ángeles':                              ('Madrid',    'Villaverde'),
    'Lucero':                                   ('Madrid',    'Latina'),
    'Malilla':                                  ('Valencia',  'Quatre Carreres'),
    'Marroquina':                               ('Madrid',    'Moratalaz'),
    'Marxalenes':                               ('Valencia',  'La Saïdia'),
    'Massarrojos':                              ('Valencia',  'Poblats del Nord'),
    'Media Legua':                              ('Madrid',    'Moratalaz'),
    'Mestalla':                                 ('Valencia',  'El Pla del Real'),
    'Mirasierra':                               ('Madrid',    'Fuencarral-El Pardo'),
    'Mont-Olivet':                              ('Valencia',  'Quatre Carreres'),
    'Montbau':                                  ('Barcelona', 'Horta-Guinardó'),
    'Morvedre':                                 ('Valencia',  'La Saïdia'),
    'Moscardó':                                 ('Madrid',    'Usera'),
    'Na Rovella':                               ('Valencia',  'Quatre Carreres'),
    'Natzaret':                                 ('Valencia',  'Poblats Marítims'),
    'Navas':                                    ('Barcelona', 'Sant Andreu'),
    'Niño Jesús':                               ('Madrid',    'Retiro'),
    'Nou Moles':                                ("Valencia",  "L'Olivereta"),
    "Nova Esquerra de l'Eixample":              ('Barcelona', "L'Eixample"),
    'Nueva España':                             ('Madrid',    'Chamartín'),
    'Numancia':                                 ('Madrid',    'Moratalaz'),
    'Opañel':                                   ('Madrid',    'Carabanchel'),
    'Orcasitas':                                ('Madrid',    'Usera'),
    'Orcasur':                                  ('Madrid',    'Usera'),
    'Pacífico':                                 ('Madrid',    'Retiro'),
    'Palacio':                                  ('Madrid',    'Centro'),
    'Palomas':                                  ('Madrid',    'Hortaleza'),
    'Palomeras Bajas':                          ('Madrid',    'Puente de Vallecas'),
    'Palomeras Sureste':                        ('Madrid',    'Moratalaz'),
    'Palos de la Frontera':                     ('Madrid',    'Arganzuela'),
    'Patraix':                                  ('Valencia',  'Patraix'),
    'Pavones':                                  ('Madrid',    'Moratalaz'),
    'Pedralbes':                                ('Barcelona', 'Les Corts'),
    'Penya-Roja':                               ('Valencia',  'Camins al Grau'),
    'Peñagrande':                               ('Madrid',    'Fuencarral-El Pardo'),
    'Pilar':                                    ('Madrid',    'Fuencarral-El Pardo'),
    'Pinar del Rey':                            ('Madrid',    'Hortaleza'),
    'Pinedo':                                   ('Valencia',  'Poblats del Sud'),
    'Piovera':                                  ('Madrid',    'Hortaleza'),
    'Poble Nou':                                ('Valencia',  'Poblats del Nord'),
    'Porta':                                    ('Barcelona', 'Nou Barris'),
    'Portazgo':                                 ('Madrid',    'Puente de Vallecas'),
    'Pradolongo':                               ('Madrid',    'Usera'),
    'Prosperidad':                              ('Madrid',    'Chamartín'),
    'Provençals del Poblenou':                  ('Barcelona', 'Sant Martí'),
    'Pueblo Nuevo':                             ('Madrid',    'Ciudad Lineal'),
    'Puerta Bonita':                            ('Madrid',    'Carabanchel'),
    'Puerta del Ángel':                         ('Madrid',    'Latina'),
    'Quintana':                                 ('Madrid',    'Ciudad Lineal'),
    'Recoletos':                                ('Madrid',    'Salamanca'),
    'Rejas':                                    ('Madrid',    'San Blas-Canillejas'),
    'Rosas':                                    ('Madrid',    'San Blas-Canillejas'),
    'Russafa':                                  ('Valencia',  "L'Eixample (Valencia)"),
    'Ríos Rosas':                               ('Madrid',    'Chamberí'),
    'Safranar':                                 ('Valencia',  'Patraix'),
    'Sagrada Família':                          ('Barcelona', "L'Eixample"),
    'Salvador':                                 ('Madrid',    'Ciudad Lineal'),
    'San Cristóbal':                            ('Madrid',    'Villaverde'),
    'San Diego':                                ('Madrid',    'Puente de Vallecas'),
    'San Fermín':                               ('Madrid',    'Usera'),
    'San Isidro':                               ('Madrid',    'Carabanchel'),
    'San Juan Bautista':                        ('Madrid',    'Ciudad Lineal'),
    'San Pascual':                              ('Madrid',    'Ciudad Lineal'),
    'Sant Andreu de Palomar':                   ('Barcelona', 'Sant Andreu'),
    'Sant Antoni':                              ('Barcelona', "L'Eixample"),
    'Sant Francesc':                            ('Valencia',  'Ciutat Vella (Valencia)'),
    'Sant Genís dels Agudells':                 ('Barcelona', 'Horta-Guinardó'),
    'Sant Gervasi - Galvany':                   ('Barcelona', 'Sarrià-Sant Gervasi'),
    'Sant Gervasi - la Bonanova':               ('Barcelona', 'Sarrià-Sant Gervasi'),
    'Sant Isidre':                              ('Valencia',  'Patraix'),
    'Sant Llorenç':                             ('Valencia',  'Rascanya'),
    'Sant Marcel·lí':                           ('Valencia',  'Jesús'),
    'Sant Martí de Provençals':                 ('Barcelona', 'Sant Martí'),
    'Sant Pau':                                 ("Valencia",  "L'Olivereta"),
    'Sant Pere, Santa Caterina i la Ribera':    ('Barcelona', 'Ciutat Vella'),
    'Santa Eugenia':                            ('Madrid',    'Villa de Vallecas'),
    'Sants':                                    ('Barcelona', 'Sants-Montjuïc'),
    'Sants-Badal':                              ('Barcelona', 'Sants-Montjuïc'),
    'Sarrià':                                   ('Barcelona', 'Sarrià-Sant Gervasi'),
    'Simancas':                                 ('Madrid',    'San Blas-Canillejas'),
    'Sol':                                      ('Madrid',    'Centro'),
    'Soternes':                                 ("Valencia",  "L'Olivereta"),
    'Timón':                                    ('Madrid',    'Moratalaz'),
    'Tormos':                                   ('Valencia',  'La Saïdia'),
    'Torre Baró':                               ('Barcelona', 'Nou Barris'),
    'Torrefiel':                                ('Valencia',  'Rascanya'),
    'Trafalgar':                                ('Madrid',    'Chamberí'),
    'Tres Forques':                             ("Valencia",  "L'Olivereta"),
    'Trinitat':                                 ('Valencia',  'La Saïdia'),
    'Universidad':                              ('Madrid',    'Centro'),
    'Valdeacederas':                            ('Madrid',    'Tetuán'),
    'Valdebernardo':                            ('Madrid',    'Vicálvaro'),
    'Valdefuentes':                             ('Madrid',    'Hortaleza'),
    'Valdemarín':                               ('Madrid',    'Moncloa-Aravaca'),
    'Valdezarza':                               ('Madrid',    'Moncloa-Aravaca'),
    'Vallcarca i els Penitents':                ('Barcelona', 'Gràcia'),
    'Vallehermoso':                             ('Madrid',    'Chamberí'),
    'Vallvidrera, el Tibidabo i les Planes':    ('Barcelona', 'Sarrià-Sant Gervasi'),
    'Valverde':                                 ('Madrid',    'Fuencarral-El Pardo'),
    'Vara de Quart':                            ('Valencia',  'Patraix'),
    'Ventas':                                   ('Madrid',    'Ciudad Lineal'),
    'Verdun':                                   ('Barcelona', 'Nou Barris'),
    'Vila de Gràcia':                           ('Barcelona', 'Gràcia'),
    'Vilapicina i la Torre Llobeta':            ('Barcelona', 'Nou Barris'),
    'Vinateros':                                ('Madrid',    'Moratalaz'),
    'Vista Alegre':                             ('Madrid',    'Carabanchel'),
    'Zofío':                                    ('Madrid',    'Usera'),
}


## 23.5 Función mapeo código mejorado

In [ ]:
# Añadir ciudad y distrito desde el diccionario
df_barrios_oficial['ciudad']           = df_barrios_oficial['barrio_oficial'].map(lambda b: codigo_barrio_.get(b, (None, None))[0])
df_barrios_oficial['distrito_oficial'] = df_barrios_oficial['barrio_oficial'].map(lambda b: codigo_barrio_.get(b, (None, None))[1])

# Construir id_barrio_oficial compuesto (C + DD + BBB)
# El orden alfabético de barrio_oficial ya está en el df, su posición da el código BBB
# Usams la función .zfill para ir introduciendo ls 3 cifras en la variable que nos dará la posición en la lista del barrio
barrio_posicion = {b: str(i).zfill(3) for i, b in enumerate(df_barrios_oficial['barrio_oficial'], 1)}

# Construimos la función que irá creando el código siguiendo las especificaciones aprtadas:
def build_id(row):
    if pd.isna(row['ciudad']): return None
    c   = str(codigo_ciudad[row['ciudad']])
    dd  = codigo_distrito[row['distrito_oficial']]
    bbb = barrio_posicion[row['barrio_oficial']]
    return c + dd + bbb

df_barrios_oficial['id_barrio_oficial'] = df_barrios_oficial.apply(build_id, axis=1) # aplicamos la función a la columna id_barrio_oficial

# Reordenar columnas
df_barrios_oficial = df_barrios_oficial[[
    'id_barrio_oficial', 'ciudad', 'distrito_oficial',
    'barrio_oficial', 'barrio_turistico_objetivo'
]]

print(df_barrios_oficial.head(10).to_string())
print(f"\nTotal: {len(df_barrios_oficial)} barrios")
print(f"Longitud IDs: {df_barrios_oficial['id_barrio_oficial'].dropna().str.len().unique()}")


In [ ]:
df_barrios_oficial.sample(10)

Éxito total. Ahora ya tendremos un csv con una tabla de dimensiones que nos servirá para relacionar entre sí las tablas de hechos de Idealista/Fotocasa con las de AirBnB.

In [ ]:
# Guardamos el csv para poder seguir trabajando después
df_barrios_oficial.to_csv('tabla_barrios_oficial_turísticos_id_mejorado (check final Claude).csv', index=False)

In [ ]:
# df_barrios es simplemente un alias de la tabla que acabamos de construir, para que coincida
# con el nombre que se usa en el resto del cuaderno original.
df_barrios = df_barrios_oficial.copy()

## 23.6 Pasamos Inside AirBnB a ID Barrio Oficial

Más adelante (o más abajo) he terminado convirtiendo todas las columnas de ciudad, distrito y barrio en idealista y fotocasa a un solo **código de identificación para cada ciudad/distrito/barrio** con 2 motivos:
- Simplificar las columnas de cada dataset: es más ligero y eficiente de gestionar con una sola columna para la localización del inmueble, aunque tengamos que comparar con la tabla de dimensiones de los barrios.
- En Python es más complejo de gestionar por tener que usar la tabla, pero ya está todo practicamente listo para trabajar lo que ya tenemos en Power BI.

Así, voy a **convertir esas columnas en 1 sola de id_barrio_oficial para el dataset de Inside AirBnB**.

In [ ]:
df_inside_airbnb.columns

Cargo el csv de **Tabla Barrios**

*(Ya tenemos `df_barrios` construido en la sección anterior — no hace falta volver a cargarlo desde un CSV.)*

In [ ]:
# Hacemos un merge de tipo left para añadir el código:
df_inside_airbnb = df_inside_airbnb.merge(df_barrios[['barrio_oficial', 'id_barrio_oficial']],
    on='barrio_oficial',
    how='left')
# Y comprobamos
print("Nulos en id_barrio_oficial:", df_inside_airbnb['id_barrio_oficial'].isna().sum())

In [ ]:
df_inside_airbnb.sample(6)

In [ ]:
# Eliminar columna barrio ya innecesaria
df_inside_airbnb_id_barrio = df_inside_airbnb.drop(columns=['barrio'])

In [ ]:
# Guardamos el csv ya listo
df_inside_airbnb_id_barrio.to_csv('datos_airbnb_limpio_id_barrio_oficial (check final Claude).csv', index=False)

# 24. Combinar Idealista y Fotocasa

Antes de seguir haciendo el filtro y el repaso estadístico, voy a llevar a cabo la **unión de los 2 datasets de Fotocasa y de Idealista** para así trabajar con ellos en conjunto.
Paso previo a unirlas, tendré que unificar las columnas de "**detalles extra**" de ambas:
- Idealista: Garaje, Ascensor, Exterior.
- Fotocasa: todo lo que está en "Extra" (Ascensor | Parking· | Calefacción | Aire Acondicionado).

Creamos una columna llamada "detalles_extra" donde pondremos todo junto para posterior uso si fuera necesario.

Cargo el csv de **FOTOCASA**

*(En el cuaderno original, aquí se recargaban Fotocasa, Idealista y la tabla de barrios desde los CSV ya guardados, para poder segmentar el trabajo. Como seguimos con el mismo DataFrame en memoria, no hace falta recargar nada — continuamos directamente.)*

Cargo el csv de **IDEALISTA**

Cargo el csv de **Tabla Barrios**

In [ ]:
#df_fotocasa.sample(6)
#df_idealista.sample(6)
df_barrios.sample(6)

## 24.1 Preparar Idealista

Primero tengo que juntar todos los detalles extra en una sol columna

In [ ]:
df_idealista.sample(6)

In [ ]:
# Definimos la función para ir extrayendo las "partes" de detalles:
def combinar_detalles_idealista(row):
    partes = []
    if str(row['garaje']).strip().lower() != 'no':
        partes.append(str(row['garaje']).strip())
    if str(row['ascensor']).strip().lower() == 'con ascensor':
        partes.append('Ascensor')
    if pd.notna(row['exterior']):
        partes.append(str(row['exterior']).strip().capitalize())
    return ' | '.join(partes) if partes else None

# aplicamos la función a la nueva columna nueva de detalles_extra
df_idealista['detalles_extra'] = df_idealista.apply(combinar_detalles_idealista, axis=1)

Ahora ya pasamos a renombrar columnas en Idealista


In [ ]:
df_idealista = df_idealista.rename(columns={
    'precio':           'precio_inmueble',
    'habitaciones':     'habitaciones',
    'metros_cuadrados': 'metros_cuadrados',
    'Precio/m2':        'precio_m2',
    'tipo_inmueble':    'tipo_inmueble',
    'id_anuncio':       'id_anuncio',
})

In [ ]:
df_idealista.sample(6)

In [ ]:
# Idealista ya tiene los tipos más lógicos; solo ajustamos id_anuncio a string/oject
#para que sea igual que en Fotocasa y pueda usarse como clave textual
df_idealista['id_anuncio']       = df_idealista['id_anuncio'].astype(str)
df_idealista['precio_inmueble']  = df_idealista['precio_inmueble'].astype(float)
df_idealista['habitaciones']     = pd.to_numeric(df_idealista['habitaciones'], errors='coerce')
df_idealista['metros_cuadrados'] = pd.to_numeric(df_idealista['metros_cuadrados'], errors='coerce')
df_idealista['precio_m2']        = pd.to_numeric(df_idealista['precio_m2'], errors='coerce')

Se me ha pasado eliminar las columnas que no son necesarias ya:
- Las que he metido en deatlles_extra
- La de Precio/Hab porque no la vamos a usar ahora (quizás en Power BI)

## 24.2 Preparar Fotocasa

Lo primero es dejar la columna detalles_extra igual que la de idealsita.

In [ ]:
# Limpiar primero los puntos·volados de la columna extras
df_fotocasa['extras'] = df_fotocasa['extras'].str.replace('·', '', regex=False).str.strip()


In [ ]:
# Ponemos los mismo nombres a las columnas
df_fotocasa = df_fotocasa.rename(columns={
    'precio_eur':        'precio_inmueble',
    'num_habitaciones':  'habitaciones',
    'num_m2':            'metros_cuadrados',
    'num_baños':         'baños',
    'Precio/m2':         'precio_m2',
    'tipo_inmueble':     'tipo_inmueble',
    'id_anuncio':        'id_anuncio',
    'dias_publicado':    'dias_publicado',
    'extras':            'detalles_extra',
})

In [ ]:
# Fotocasa tiene algunos campos como float donde Idealista tiene int;
# usamos float como tipo común para evitar NaN en enteros
df_fotocasa['id_anuncio']       = df_fotocasa['id_anuncio'].astype('Int64').astype(str)
df_fotocasa['precio_inmueble']  = pd.to_numeric(df_fotocasa['precio_inmueble'], errors='coerce')
df_fotocasa['precio_m2']        = pd.to_numeric(df_fotocasa['precio_m2'], errors='coerce')
df_fotocasa['habitaciones']     = pd.to_numeric(df_fotocasa['habitaciones'], errors='coerce')
df_fotocasa['baños']            = pd.to_numeric(df_fotocasa['baños'], errors='coerce')
df_fotocasa['metros_cuadrados'] = pd.to_numeric(df_fotocasa['metros_cuadrados'], errors='coerce')
df_fotocasa['dias_publicado']   = pd.to_numeric(df_fotocasa['dias_publicado'], errors='coerce').astype('Int64')

Comparamos ambas

In [ ]:
df_fotocasa.sample(6)

In [ ]:
df_fotocasa.info()

In [ ]:
df_idealista.sample(6)

In [ ]:
df_idealista.info()

## 24.3 Añadimos id_barrio

Vamos a meter ahora la columna clave de id_barrio_oficial porque así podemos quitar ya las columnas de este ámbito.

In [ ]:
# Fotocasa
df_fotocasa = df_fotocasa.merge(
    df_barrios[['barrio_oficial', 'id_barrio_oficial']],
    on='barrio_oficial', how='left')

df_fotocasa['base_datos'] = 'Fotocasa'

# Idealista
df_idealista = df_idealista.merge(
    df_barrios[['barrio_oficial', 'id_barrio_oficial']],
    on='barrio_oficial', how='left')

df_idealista['base_datos'] = 'Idealista'

In [ ]:
df_fotocasa.sample(6)

In [ ]:
df_idealista.sample(6)

## 24.4 Retirar, renombrar y ordenar columnas

Ahora ya sí **vamos a eliminar columnas que no van al df combinado**


In [ ]:
# Fotocasa
df_fotocasa_2 = df_fotocasa.drop(columns=[
    'Precio/hab', 'barrio', 'barrio_2',
    'municipio', 'distrito', 'barrio_oficial'])

# Idealista
df_idealista_2 = df_idealista.drop(columns=[
    'zona', 'nombre', 'garaje', 'ascensor', 'exterior',
    'Precio/hab', 'barrio', 'barrio_oficial'])

Hago otra pasada para armonizar el tipo de dato

In [ ]:
# habitaciones: int64 en Idealista, float64 en Fotocasa → float64 en ambos
df_idealista_2['habitaciones']     = df_idealista_2['habitaciones'].astype(float)
df_idealista_2['metros_cuadrados'] = df_idealista_2['metros_cuadrados'].astype(float)

# metros_cuadrados: int64 en Fotocasa → float64 para consistencia
df_fotocasa_2['metros_cuadrados']  = df_fotocasa_2['metros_cuadrados'].astype(float)

Ya podemos pasar a ordenar columnas igual en ambos

In [ ]:
COLUMNAS_ORDEN = [
    'base_datos', 'id_anuncio', 'precio_inmueble', 'precio_m2',
    'tipo_inmueble', 'habitaciones', 'baños', 'metros_cuadrados',
    'num_planta', 'id_barrio_oficial', 'dias_publicado', 'detalles_extra'
]

# Idealista no tiene baños ni dias_publicado → NaN
df_idealista_2['baños']          = None
df_idealista_2['dias_publicado'] = None

df_fotocasa_combinar  = df_fotocasa_2[COLUMNAS_ORDEN]
df_idealista_combinar = df_idealista_2[COLUMNAS_ORDEN]

In [ ]:
df_fotocasa_combinar

In [ ]:
df_idealista_combinar

## 24.5 Combinar y Comprobar

Ya tenemos todo lo necesario apra que los dfs sean apropiados para juntarlos: ambos tienen exactamente las mismas 12 columnas en el mismo orden y los mismos tipos. El único detalle es que en Idealista "baños" y "dias_publicado" aparecen como None (object) en lugar de NaN numéric (nulos vamos), pero eso se resuelve solo al **hacer el concat**.

**Ya podemos combinarlo**

In [ ]:
df_compra_inmuebles_combinado = pd.concat(
    [df_fotocasa_combinar, df_idealista_combinar],
    ignore_index=True)

In [ ]:
df_compra_inmuebles_combinado

Vamos a hacer una serie de comprobaciones para estar seguros de que tenemos lo que necesitamos

In [ ]:
# 1. Dimensiones: debe ser la suma de ambos (11064 + 24499 = 35563)
print("Filas totales:", len(df_compra_inmuebles_combinado))
print("Columnas:", df_compra_inmuebles_combinado.shape[1])

# 2. Tipos de dato por columna
print("\nTipos de dato:")
print(df_compra_inmuebles_combinado.dtypes)

# 3. Filas por fuente
print("\nRegistros por fuente:")
print(df_compra_inmuebles_combinado['base_datos'].value_counts())

# 4. Nulos por columna (baños y dias_publicado tendrán los de Idealista)
print("\nNulos por columna:")
print(df_compra_inmuebles_combinado.isnull().sum())


**Resultado:**

**Filas totales: 35.563**

**Columnas: 12**

Registros por fuente:
base_datos

**Idealista    24.499**

**Fotocasa     11.064**

**Nulos por columna:**

- baños: 24.627 nulos = 24.499 de Idealista (no tiene la columna) + 128 que ya faltaban en Fotocasa.
- num_planta: 6.000 nulos = combinación de los que faltaban en ambas fuentes.
- dias_publicado: 24.499 nulos = exactamente todos los de Idealista.
- detalles_extra: 7.786 nulos = los que no tenían extras en ninguna de las dos fuentes.

In [ ]:
# Guardamos el csv para poder seguir trabajando después (aunque ya casi es el definitivo)
df_compra_inmuebles_combinado.to_csv('datos_compra_inmuebles_combinado (IDL + FTC).csv', index=False)

# 25. Outliers en Idealista/Fotocasa y segmentación de precios

En esta sección vamos a llevar a cabo el análisis en conjunto de todo el dataset resultante de combinar Idealista y Fotocasa, ya que como se indicó en varios momentos al evaluar **valores atípicos (outliers)**, la retirada total parecía excesiva y, de llevarse a cabo, habría que alinear los criterios en todo el dataset del mercado de compra-venta de inmuebles. Por ello procedemos a:
- Hacer una evaluación de los estadísticos del precio, los m2 y sobre todo el Precio/m2 como valor más relevante para analizar.
- Posiblemente retirar todos aquellos que sean extremos y/o poco relevantes, pero atendiendo a que la variabilidad del precio y los m2 representa la gran variedad de inmuebles en el mercado.
- Añadir una nueva variable con el precio esperado en compra del inmueble (**aplicar estimaciones de precio en notaria vs mercado**)
- Finalmente, hacer una segmentación lógica del Precio/m2 para atender a distintos mercados (Entrada, Común, Alto, Lujo) que nos permita luego discrimar estos mercados en el cálculo del ROI para cada uno de ellos.

*(Seguimos con `df_compra_inmuebles_combinado` ya en memoria, sin recargar el CSV que se acaba de guardar.)*

In [ ]:
df_inmuebles = df_compra_inmuebles_combinado.copy()
print(f"Filas del df_inmuebles: {len(df_inmuebles)}")

In [ ]:
df_inmuebles

Ahora vamos a evaluar las varibales clave (Precio y m2) de manera conjunta. Primero, para tener la perspectiva, saco TODOS los estadísticos para estas variables.

En vez de hacer prints como hacía en las partes anterioress, voy a crear una función para juntar todos los cálculos relevantes que nos pueden ayudar a evaluar las variables.
Además de las habituales de percentiles, media, mediana, IQR, etc; voy a emplear estos otros estadísticos:
- **Coeficiente de Variación (CV)**: Es la desviación típica expresada como porcentaje de la media. Nos da una forma de comparar, entre varias variables, cuál de ellas tiene más dispersión respecto al valor esperado.
- **Skewness (Asimetría)**: mide lo simétrica que es la distribución de los valores de nuestras variables, y si esta está inclinada hacia uno u otro lado. Una Skewness de 0 sería una distribución normal (campana de gauss) perfectamente simétrica con Promedio=Mediana=Moda. Ya intuyo, por el análisis previo, que habrá una asimetría positiva (cola de la distribución alargada hacia la derecha, los valoresmás altos).
- **Curtosis**: aquí ya es básicamente valorar esa asimetría (skewness), pues una curtosis más o menos alta nos dirá si esos valores en las colas tendrán más o menos peso en distorsionar la muestra. Nos ayuda a distinguir si los extremos que vemos son anomalías estadísticas (errores, datos corruptos) o segmentos de mercado legítimos como el lujo o las microviviendas.

In [ ]:
# DEfino un diccionario con las variables de interés
vars_analisis = {
    "precio_inmueble": "Precio (€)",
    "metros_cuadrados": "Metros Cuadrados (m²)",
    "precio_m2": "Precio / m² (€/m²)"
}

# Estadísticos y cálcuos personalizados
def estadisticos_variable(series, nombre):
    q1  = series.quantile(0.25)
    q3  = series.quantile(0.75)
    IQR = q3 - q1
    return pd.Series({
        "Variable"    : nombre,
        "Count"       : int(series.count()),
        "Nulos"       : int(series.isna().sum()),
        "Media"       : round(series.mean(), 2),
        "Mediana"     : round(series.median(), 2),
        "Moda"        : round(series.mode()[0], 2) if not series.mode().empty else np.nan,
        "Std"         : round(series.std(), 2),
        "CV (%)"      : round((series.std() / series.mean()) * 100, 2),
        "Min"         : round(series.min(), 2),
        "P5"          : round(series.quantile(0.05), 2),
        "P10"         : round(series.quantile(0.10), 2),
        "P25 (Q1)"    : round(q1, 2),
        "P50 (Med)"   : round(series.quantile(0.50), 2),
        "P75 (Q3)"    : round(q3, 2),
        "P85"         : round(series.quantile(0.85), 2),
        "P90"         : round(series.quantile(0.90), 2),
        "P95"         : round(series.quantile(0.95), 2),
        "P99"         : round(series.quantile(0.99), 2),
        "Max"         : round(series.max(), 2),
        "IQR"         : round(IQR, 2),
        "Límite inf (Q1 - 1.5·IQR)" : round(q1 - 1.5 * IQR, 2),
        "Límite sup (Q3 + 1.5·IQR)" : round(q3 + 1.5 * IQR, 2),
        "Skewness"    : round(series.skew(), 4),
        "Kurtosis"    : round(series.kurt(), 4),
    })

# Harenos una tabla que resuma todos estos cálculos para nuestras 3 variables
filas = []
for col, nombre in vars_analisis.items():
    serie = df_inmuebles[col].dropna()
    filas.append(estadisticos_variable(serie, nombre))

df_stats = pd.DataFrame(filas).set_index("Variable").T

# Imprimimos el resultado con un formato legible
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("ESTADÍSTICOS DESCRIPTIVOS — Dataset combinado (Idealista + Fotocasa)")
print("=" * 65)
print(df_stats.to_string())

Aquí podemos interpretar que el precio tiene mucha más variación que los m2, hasta el punto de que el CV es mayor que 100%. De aquí extraigo, de momento, que **el precio es principal elemento de exceso de variabilidad**.
Pero que los m2 también tienen mucha varibilidad (cv 75%). Así que hasta ahora saco en claro que tengo que limpiar sobre todo el precio pero también los m2.

Como ya intuía, porque ya vimos en los gráficos, tenemos Skewness positiva y alta en ambas variables, precio y m2, ya que las colas son pronunciadas hacia la derecha (valores altos). En cambio,  es un poco significativo que Precio/m2 no tiene tanta skewness (asimetría). Esto era evidente ya viendo los gráficos. Pero, la curtosis nos dice que la distribución en ambos casos, aunque sobre todo precio, es leptocúrtica. Las colas son muy pesadas y hay una probabilidad extremadamente alta de valores atípicos extremos.  De esto derivo que tengo que hacer una retirada importante de valores extremos por lo alto para que podamos tener una muestra depurada.

Ahora, **la cuestión es si aplicar el IQR y retirar valores fuera de los límites es lo adecuado**. El IQR es un método robusto y útil en distribuciones simétricas o moderadamente asimétricas. El problema es que con una kurtosis de 36 en el precio, los límites que genera son conservadores en el extremo inferior y demasiado agresivos en el extremo superior. El límite superior IQR para precio es 1.925.000€, pero tenemos un P90 de 1.836.000€, lo que significa que **el IQR eliminaría** prácticamente todo el decil superior del mercado. Eso incluiría viviendas que no son errores sino **el segmento Alto y Lujo** que queremos conservar.

Lo que tiene más sentido es **una limpieza por percentiles con criterio de negocio**.



In [ ]:
df_inmuebles

In [ ]:
# Voy a representar el precio con varios valores, pero retiro los que están por encima de 2 millones
# ya que está cerca del Límite sup (Q3 + 1.5·IQR) para que se pueda visualizar
df_inmuebles_precio = df_inmuebles[df_inmuebles['precio_inmueble'] < 2000000]

In [ ]:
# Voy a representar la distribución de Precio con el límite de 100.000€, la media y la mediana para tener una vista de la situación antes de continuar.
# Calculo la meida y mediana para añadirlas:
limite_precio_inf   = 100000
mean_precio   = df_inmuebles_precio['precio_inmueble'].mean()
median_precio = df_inmuebles_precio['precio_inmueble'].median()

plt.figure(figsize=(20, 8))

# Con histplot y kde=True, Python dibuja el histograma + la curva de densidad encima
sns.histplot(data=df_inmuebles_precio, x='precio_inmueble', kde=True, color='steelblue')
# Añado las linea verticales de Media y Mediana:
plt.axvline(limite_precio_inf,   color='red',   linestyle='--', label=f'100.000 €')
plt.axvline(mean_precio, color='green', linestyle=':',  label=f'Media Precio ({mean_precio:.2f})')
plt.axvline(median_precio, color='black', linestyle=':',  label=f'Mediana Precio ({median_precio:.2f})')
# Titulando el gráfico:
plt.title('Distribución del Precio < 2M€ con Límite, Media y Mediana')
plt.xlabel('Precio (€)')
plt.ylabel('Número de inmuebles')
plt.legend()
plt.show()

## 25.1 Límites inferiores precio y m2

**Empezamos por el extremo inferior:**

En las tres ciudades del proyecto (Madrid, Barcelona, Valencia), un precio **por debajo de 100.000€** para una vivienda en venta es prácticamente inviable salvo casos muy excepcionales (locales reconvertidos, trasteros mal clasificados, o errores de entrada de datos). El P5 está en 179.999€, así que retiraríamos aproximadamente el 3-4% inferior, que casi con seguridad son registros con problemas. **Es un límite limpio.**

E**n cuanto a metros cuadrados:**

Si el modelo de negocio es alquiler vacacional, un estudio de 35-40 m² bien ubicado en el centro de Barcelona es perfectamente válido e incluso rentable. Si lo bajamos a 35 m² como límite inferior, eliminaríamos lo que pueden ser errores o viviendas mal categorizadas (aunque ya hicimos esta limpieza en su momento). Pero casi con seguridad, por debajo son poco relevantes para WhiteHosting. Así que lo más lógico sería **poner el límite en 35 m²** para no perder esos estudios urbanos que tienen sentido en el contexto del proyecto.

In [ ]:
# Retiramos los límtes inferiores
filtro_precio_min = df_inmuebles["precio_inmueble"] > 100_000
filtro_m2_min     = df_inmuebles["metros_cuadrados"] >= 35

df_inmuebles_2 = df_inmuebles[filtro_precio_min & filtro_m2_min].copy()

# Resumen retirada
total_original = len(df_inmuebles)
total_clean    = len(df_inmuebles_2)
retirados      = total_original - total_clean

print(f"Registros originales : {total_original:,}")
print(f"Registros retirados  : {retirados:,}  ({retirados/total_original*100:.2f}%)")
print(f"Registros resultantes: {total_clean:,}  ({total_clean/total_original*100:.2f}%)")

print("\nDesglose de retirados por filtro:")
print(f"Precio < 100.000€ : {(~filtro_precio_min).sum():,}")
print(f"Metros cuadrados < 35 : {(~filtro_m2_min).sum():,}")

Impacto de retirada de límetes inferiores:


Registros **retirados**  : **801  (2.25%)**

Registros resultantes: 34,762  (97.75%)

Desglose de retirados por filtro:
- Por precio < 100.000€  : 217
- Por m² < 35            : 631

**Continuamos por el extremo inferior:**

Lo más intuitivo a estas alturas, porque ha sido la dinámica en todo el análisis Idealista&fotocasa, es enfocarnos en Precio/m2 porque ahí sabemos el valor real de la propiedad. Pero también habría que retirar viviendas excesivamente grandes (en m2) o excesivamente caras (en precio) porque no tendría sentido ser adquiridas y explotadas en Booking/AirBnB por WhiteHosting. Queremos un análisis genérico que tendrá viviendas baratas y de lujo, pero tampoco en exceso. Especialmente con las conclusiones que sacamos antes donde la **extrema asimetría por lo alto nos distorsiona el resto de la distribución y por ende los estadísticos principales**.
Además que en Power BI haremos representaciones de los estadísticos de nuestra muestra, y si hay valores extremos que nos trastocan la media, la desviación típica y otros, no tendrán tanto valor.

Extremos superiores de Precio

Como el IQR no será fiable para hacer el corte, vamos a fijarnos en los percentiles:

- P90: 1.836.000€
- P95: 2.590.000€
- P99: 4.900.000€
- Max: 19.900.000€

Una vivienda de 19,9M€ o incluso de 6,7M€ no es un activo que WhiteHosting vaya a adquirir para explotar en Airbnb/Booking, independientemente de que exista en el mercado. El modelo de negocio es comprar volumen de unidades residenciales, de varios segmentos de precio, pero no grandes activos singulares. Además, esos valores extremos con una kurtosis de 36 van a destruir cualquier visualización de media o desviación en Power BI.

Un buen planteamiento sería **poner el techo entre P95 y P99. Es decir, entre 2.590.000€ y 4.900.000€**. Esto significa que conservamos viviendas de lujo real (entre 1M€ y 4M€ hay mercado legítimo en Madrid y Barcelona) pero eliminamos el 5% superior que son casos singulares o activos que no encajan en el modelo. Pero antes vamos a graficar y observar la distribución.

In [ ]:
# Vamos a graficar la parte alta para comprobar (REPENSADO TRAS BOXPLOT ABAJO):
df_inmuebles_precio_alto = df_inmuebles_2[(df_inmuebles_2['precio_inmueble'] > 2000000) & (df_inmuebles_2['precio_inmueble'] < 10000000)]

In [ ]:
# Calculo los percentiles de nuevo (con el df sin alterar o filtrar):
print(df_inmuebles['precio_inmueble'].quantile(0.95).round(2))
print(df_inmuebles['precio_inmueble'].quantile(0.97).round(2))
print(df_inmuebles['precio_inmueble'].quantile(0.99).round(2))
print(df_inmuebles_precio_alto['precio_inmueble'].mean())

In [ ]:
# Voy a representar la distribución de Precios por encima de 2M€ (REHACER: por debajo de 10M€),
# pintando los percentiles para tener una vista de la situación antes de continuar.
# Calculo los percentiles para añadirlos:
p95   = df_inmuebles['precio_inmueble'].quantile(0.95).round(2)
p97   = df_inmuebles['precio_inmueble'].quantile(0.97).round(2)
media_precio_alto = df_inmuebles_precio_alto['precio_inmueble'].mean()
p99   = df_inmuebles['precio_inmueble'].quantile(0.99).round(2)

plt.figure(figsize=(20, 8))

# Con histplot y kde=True, Python dibuja el histograma + la curva de densidad encima
sns.histplot(data=df_inmuebles_precio_alto, x='precio_inmueble', kde=True, color='steelblue')
# Añado las linea verticales de Media y Mediana:
plt.axvline(p95, color='black', linestyle='--', label=f'Percentile 95% ({p95:.2f})')
plt.axvline(p97, color='red', linestyle='--', label=f'Percentile 97% ({p97:.2f})')
plt.axvline(media_precio_alto, color='green', linestyle=':',  label=f'Media Precios altos ({media_precio_alto:.2f})')
plt.axvline(p99, color='black', linestyle='--', label=f'Percentile 99% ({p99:.2f})')

# Titulando el gráfico:
plt.title('Distribución de Precios Altos con percentiles')
plt.xlabel('Precio (€)')
plt.ylabel('Número de inmuebles')
plt.legend()
plt.show()

Vale, como no puedo ver con todo el detalle deseado la gráfica y, como ya planteo retirar muchos de los valores extremos, hago un boxplot rápido para quitar los que están por encima de 10M€ para volver a hacer la gráfica

In [ ]:
# Hago un boxplot rápido:
plt.figure(figsize=(15,5))
sns.boxplot(data=df_inmuebles_precio_alto, x='precio_inmueble')
plt.title("Distribución del Precio (tramos altos)")
plt.show()

Vuelvo atrás y quito los valores por encima de 10M€ para **rehacer la gráfica**.

Una vez hecho el cambio, ya se pude ver que **el techo podría estar alrededor del percentil 97% (3.300.000€)**. Voy a sacar la lista de los que están por encima de 3.5M€.

In [ ]:
# Filtramos por un potencial techo de 3.5M€:
df_inmuebles_precio_techo = df_inmuebles_2[df_inmuebles_2['precio_inmueble'] > 3500000]
df_inmuebles_precio_techo

## 25.2 Límites superiores precio y m2

Parece lógico el escoger este techo de 3.5M€. Procedo pues, pero antes vamos a revisar los metros cuadrados.

**Extremos superiores de Metros Cuadrados**

Miramos los mismo valores:

- P90: 215 m²
- P95: 284 m²
- P99: 500 m²
- Max: 1.000 m²

Una vivienda de 1.000 m² o incluso de 500 m² no es un alojamiento turístico razonable. Son chalets, mansiones o propiedades singulares que distorsionan sin aportar. Una vivienda de 200-250 m² ya es un piso grande o un ático premium perfectamente que operaría en el mercado. Por encima de eso empieza el territorio que no encaja con el modelo de negocio.
El techo adecuado estaría **entre el P95 y el P99 como con el precio, es decir unos 300 m²**, lo cual cubre desde estudios hasta áticos y viviendas familiares grandes.

In [ ]:
# Voy a graficar los m2 con varios puntos estadísticos para hacernos una idea,
# pintando los percentiles para tener una vista de la situación antes de continuar.
# Calculo los percentiles para añadirlos:
media_m2 = df_inmuebles_2['metros_cuadrados'].mean()
p95   = df_inmuebles_2['metros_cuadrados'].quantile(0.95).round(2)
p96   = df_inmuebles_2['metros_cuadrados'].quantile(0.96).round(2)
p99   = df_inmuebles_2['metros_cuadrados'].quantile(0.99).round(2)

plt.figure(figsize=(20, 8))

# Con histplot y kde=True, Python dibuja el histograma + la curva de densidad encima
sns.histplot(data=df_inmuebles_2, x='metros_cuadrados', kde=True, color='steelblue')
# Añado las linea verticales de Media y Mediana:
plt.axvline(media_m2, color='green', linestyle=':',  label=f'Media m2 ({media_m2:.2f})')
plt.axvline(p95, color='black', linestyle='--', label=f'Percentil 95% ({p95:.2f})')
plt.axvline(p97, color='red', linestyle='--', label=f'Percentil 97% ({p96:.2f})')
plt.axvline(p99, color='black', linestyle='--', label=f'Percentil 99% ({p99:.2f})')

# Titulando el gráfico:
plt.title('Distribución de Metros Cuadrados con percentiles')
plt.xlabel('Metros Cuadrados')
plt.ylabel('Número de inmuebles')
plt.legend()
plt.show()

In [ ]:
# Filtramos por un potencial techo de 300 m2:
df_inmuebles_m2_techo = df_inmuebles_2[df_inmuebles_2['metros_cuadrados'] > 300]
df_inmuebles_m2_techo

Parece lógico y razonable proceder con la retirada de los que están por encima de 300 m2.

Así que **ya tenemos los límites superiores: 3.5M € y 300 m2**.
Procedemos a retirarlos

In [ ]:
# Retiramos los límites superiores en precio y m2
filtro_precio_max = df_inmuebles_2["precio_inmueble"] < 3_500_000
filtro_m2_max     = df_inmuebles_2["metros_cuadrados"] < 300

df_inmuebles_3 = df_inmuebles_2[filtro_precio_max & filtro_m2_max].copy()

# Resumen retirada
total_original_2 = len(df_inmuebles_2)
total_clean_2    = len(df_inmuebles_3)
retirados_2      = total_original_2 - total_clean_2

print(f"Registros originales : {total_original_2:,}")
print(f"Registros retirados  : {retirados_2:,}  ({retirados_2/total_original_2*100:.2f}%)")
print(f"Registros resultantes: {total_clean_2:,}  ({total_clean_2/total_original_2*100:.2f}%)")

print("\nDesglose de retirados por filtro:")
print(f"Precio > 3.500.000 € : {(~filtro_precio_max).sum():,}")
print(f"Metros cuadrados > 300 : {(~filtro_m2_max).sum():,}")

RESULTADO:

- Registros originales : 34,762
- Registros retirados  : 1,901  (5.47%)
- Registros resultantes: **32,861**  (94.53%)

**Desglose** de retirados por filtro:

- Precio > 3.500.000 € : 971
- Metros cuadrados > 300 : 1,581

## 25.3 Vistazo final a Precio/m2

Para finalizar la revisión de valores extremos, echamos un vistazo a cómo quedan los valores del Precio/m2.
Buscando cuáles son los valores mínimos del Precio por metro cuadrado en las ciudades del proyecto, veo en varias fuentes que no baja de los 2.100 €. Y son barrios en las zonas menos recomendables de las afueras de la ciudad, por lo que no serán relevantes una vez que filtremos por los barrios turísticos objetivo. Así, **consideramos retirar aquellos inmuebles con un precio por metro cuadrado menor a 2.500 € (por debajo del Percentil 5%)**.

In [ ]:
# Filtro por menos de 2.500 € por m2:
df_inmuebles_3_bajos = df_inmuebles_3[df_inmuebles_3['precio_m2'] < 2500]
df_inmuebles_3_bajos

**Extremos superiores de Precio/m²**

Tenemos estos datos relevante:

- P95: 13.000 €/m²
- Límite superior: 14.000 €/m²
- P99: 17.523 €/m²
- Max: 27.660 €/m²

Como comentamos antes, este es el filtro más informativo. El espacio entre el límite superior en 14.000 €/m² y el P99 en 17.523 €/m² ya está en territorio de lujo extremo real en España.
De hecho, consultando varias fuentes en internet, encontramos que los m2 más caros de España (precios de Notario, que son más bajos) están en:
- Playa de la concha, San Sebastián: 9.000 €/m²
- Recoletos, Madrid: 14.000 €/m²
- Urbanización Los Verdales (Marbella), Málaga: 18.000 €/m²

Desde luego que no tendría sentido incluir inmuebles con €/m² por encima de 17.000 €/m². Pero incluso sería recomendable retirar aquellos inmuebles por encima de 15.000 €/m². Además es cercano al límite superior y en el boxplot vemos que nos quitaría gran parte de los outliers. Pero no estamos renunciando al lujo, porque ya por encima de 10.000 €/m² estaríamos hablando de precios muy altos donde una vivienda de tamaño medio-alto (140 m2) son casi 1.5M€ de precio de compra.

In [ ]:
# Hago un boxplot rápido:
plt.figure(figsize=(15,5))
sns.boxplot(data=df_inmuebles_3, x='precio_m2')
plt.title("Distribución del Precio por m2")
plt.show()

In [ ]:
df_inmuebles_3 = df_inmuebles_3[df_inmuebles_3['precio_m2'] < 18000]

In [ ]:
# Voy a graficar el Precio por m2 con los límites planteados, la media y la mediana,
limite_inferior = 2500
mediana_precio_m2 = df_inmuebles_3['precio_m2'].median()
media_precio_m2 = df_inmuebles_3['precio_m2'].mean()
limite_superior = 15000

plt.figure(figsize=(20, 8))

# Con histplot y kde=True, Python dibuja el histograma + la curva de densidad encima
sns.histplot(data=df_inmuebles_3, x='precio_m2', kde=True, color='steelblue')
# Añado las linea verticales de Media y Mediana:
plt.axvline(limite_inferior, color='red', linestyle='--',  label=f'Límite inferior ({limite_inferior:.2f})')
plt.axvline(mediana_precio_m2, color='black', linestyle=':', label=f'Mediana ({mediana_precio_m2:.2f})')
plt.axvline(media_precio_m2, color='green', linestyle=':', label=f'Media ({media_precio_m2:.2f})')
plt.axvline(limite_superior, color='red', linestyle='--',  label=f'Límite superior ({limite_superior:.2f})')

# Titulando el gráfico:
plt.title('Distribución de Precio por Metro Cuadrado')
plt.xlabel('Precio en € por metros cuadrado')
plt.ylabel('Número de inmuebles')
plt.legend()
plt.show()

Parece lógico el hacer la retirada de los inmuebles fuera de esos límites. Procedemos a retirarlos.

In [ ]:
# Retiramos los límites inferiores y superiores del Precio/m2
filtro_precio_m2_min = df_inmuebles_3["precio_m2"] >= 2_500   # MANTENER los que están POR ENCIMA
filtro_precio_m2_max = df_inmuebles_3["precio_m2"] <= 15_000  # MANTENER los que están POR DEBAJO

df_inmuebles_4 = df_inmuebles_3[filtro_precio_m2_min & filtro_precio_m2_max].copy()
# Resumen retirada
total_original_3 = len(df_inmuebles_3)
total_clean_3    = len(df_inmuebles_4)
retirados_3      = total_original_3 - total_clean_3

print(f"Registros originales : {total_original_3:,}")
print(f"Registros retirados  : {retirados_3:,}  ({retirados_3/total_original_3*100:.2f}%)")
print(f"Registros resultantes: {total_clean_3:,}  ({total_clean_3/total_original_3*100:.2f}%)")

print("\nDesglose de retirados por filtro:")
print(f"Precio/m2 < 2.500 €  : {(df_inmuebles_3['precio_m2'] < 2_500).sum():,}")
print(f"Precio/m2 > 15.000 € : {(df_inmuebles_3['precio_m2'] > 15_000).sum():,}")

**Resultado:**

- Registros originales : 32,738
- Registros retirados  : 1,845  (5.64%)
- **Registros resultantes: 30,893  (94.36%)**

Desglose de retirados por filtro:
- Precio/m2 < 2.500 €  : 1,487
- Precio/m2 > 15.000 € : 358

Volvemos a sacar los estadísticos para comprobar cómo quedan

## 25.4 Balance final

In [ ]:
# DEfino un diccionario con las variables de interés
vars_analisis = {
    "precio_inmueble": "Precio (€)",
    "metros_cuadrados": "Metros Cuadrados (m²)",
    "precio_m2": "Precio / m² (€/m²)"
}

# Estadísticos y cálcuos personalizados
def estadisticos_variable(series, nombre):
    q1  = series.quantile(0.25)
    q3  = series.quantile(0.75)
    IQR = q3 - q1
    return pd.Series({
        "Variable"    : nombre,
        "Count"       : int(series.count()),
        "Nulos"       : int(series.isna().sum()),
        "Media"       : round(series.mean(), 2),
        "Mediana"     : round(series.median(), 2),
        "Moda"        : round(series.mode()[0], 2) if not series.mode().empty else np.nan,
        "Std"         : round(series.std(), 2),
        "CV (%)"      : round((series.std() / series.mean()) * 100, 2),
        "Min"         : round(series.min(), 2),
        "P5"          : round(series.quantile(0.05), 2),
        "P10"         : round(series.quantile(0.10), 2),
        "P25 (Q1)"    : round(q1, 2),
        "P50 (Med)"   : round(series.quantile(0.50), 2),
        "P75 (Q3)"    : round(q3, 2),
        "P85"         : round(series.quantile(0.85), 2),
        "P90"         : round(series.quantile(0.90), 2),
        "P95"         : round(series.quantile(0.95), 2),
        "P99"         : round(series.quantile(0.99), 2),
        "Max"         : round(series.max(), 2),
        "IQR"         : round(IQR, 2),
        "Límite inf (Q1 - 1.5·IQR)" : round(q1 - 1.5 * IQR, 2),
        "Límite sup (Q3 + 1.5·IQR)" : round(q3 + 1.5 * IQR, 2),
        "Skewness"    : round(series.skew(), 4),
        "Kurtosis"    : round(series.kurt(), 4),
    })

# Harenos una tabla que resuma todos estos cálculos para nuestras 3 variables
filas = []
for col, nombre in vars_analisis.items():
    serie = df_inmuebles_4[col].dropna()
    filas.append(estadisticos_variable(serie, nombre))

df_stats = pd.DataFrame(filas).set_index("Variable").T

# Imprimimos el resultado con un formato legible
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("ESTADÍSTICOS DESCRIPTIVOS — Dataset combinado (Idealista + Fotocasa) - LIMPIO DE OUTLIERS")
print("=" * 75)
print(df_stats.to_string())

**Resumen consolidado de limpieza de outliers**

Dataset original: 35,563 registros

Filtros inferiores:
- Precio < 100.000 €     →   217 retirados
- Metros² < 35 m²        →   631 retirados
- Total retirados         →   801  (2.25%)
- Registros resultantes   → 34,762 (97.75%)

Filtros Superiores:
- Precio > 3.500.000 €   →   971 retirados
- Metros² > 300 m²       → 1,581 retirados
- Total retirados         → 1,901  (5.47%)
- Registros resultantes   → 32,861 (94.53%)

Filtrs Precio/m²
- Precio/m² < 2.500 €    → 1,487 retirados
- Precio/m² > 15.000 €   →   358 retirados
- Total retirados         → 1,845  (5.64%)
- Registros resultantes   → 30,893 (94.36%)

**Resultado Final: 30,893 registros válidos (86.87% del dataset original)**

Hemos tenido que retirar más del 13% de nuestra muestra, pero hemos conseguido importantes mejoras:

Hemos conseguido bajar los CVs de precio y m2 por debajo de 100% y 50% respectivamente, llevando al de Precio/m2 por debajo de 45%. Siguen siendo altos pero ya han mejorado mucho, y no hay que olvidar que segmentaremos por precios populares y lujo.
Pero, desde luego, ha mejorado mucho la asimetría de la distribución y cómo afecta a nuestra distribución (skweness y curtosis).


In [ ]:
# Guardamos el csv para poder seguir trabajando después (aunque ya casi es el definitivo)
df_inmuebles_4.to_csv('datos_compra_inmuebles_combinado (IDL + FTC) y sin outliers.csv', index=False)

# 26. Segmentación por precios

Una vez llevada a cabo la limpieza de los valores atípicos para mejorar nuestra muestra, vamos a llevar a cabo una segmentación de nuestro dataset para evaluar los inmuebles y su rentabilidad en función de si son comparables por su precio. Es problabe que en el análisis final en Power BI tengamos que hacer comparaciones dentro de segmentos de precio (como bajos, medios, lujo). Esta segmentación la vamos a considerar para **Precio o Precio/m2**

*(Nota de orden: en el cuaderno original, `df_inmuebles_5` se recargaba aquí desde el CSV recién guardado — es decir, una copia de `df_inmuebles_4` **anterior** a que existiera la columna `segmento_precio` (que se añade más abajo, a `df_inmuebles_4`). Como más adelante sí hace falta que `df_inmuebles_5` tenga `segmento_precio` (se usa en el cruce de segmentaciones), aquí seguimos trabajando directamente sobre `df_inmuebles_4` y creamos `df_inmuebles_5` como copia suya un poco más abajo, justo cuando se necesita por primera vez — así el resultado final es idéntico al que produce el cuaderno original.)*

In [ ]:
# Vamos a plotear el precio primero
# Configuramos el estilo
sns.set_theme(style="whitegrid")
# Creamos el histograma con ajuste previo de la escala (A MILES DE €)
plt.figure(figsize=(16, 8))
sns.histplot(df_inmuebles_4['precio_inmueble'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios de Venta', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')

plt.show()

In [ ]:
# Y ahora el PRECIO/M2
# Configuramos el estilo
sns.set_theme(style="whitegrid")
# Creamos el histograma con ajuste previo de la escala (A MILES DE €)
plt.figure(figsize=(16, 8))
sns.histplot(df_inmuebles_4['precio_m2'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios por m2', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')

plt.show()

In [ ]:
df_inmuebles_4

## 26.1 Segmento Precio

In [ ]:
# Posible segmentación: Precio de entrada hasta 500.000 €, precio medio hasta 1.000.000, precio alto hasta 1.500.000 y lujo por encima de ese 1.500.000
#                                                   +++
# Posible segmentación: Entrada hasta 6.000 €, medio hasta 8.000, precio alto hasta 10.000 y lujo por encima de esos 10.000

In [ ]:
# Segmentación por precio
def segmentar_precio(precio):
    if precio <= 500_000:
        return "Entrada"
    elif precio <= 1_000_000:
        return "Medio"
    elif precio <= 1_500_000:
        return "Alto"
    else:
        return "Lujo"

df_inmuebles_4["segmento_precio"] = df_inmuebles_4["precio_inmueble"].apply(segmentar_precio)

In [ ]:
# Recuento por segmento
resumen_segmentos = df_inmuebles_4.groupby("segmento_precio").agg(
    num_inmuebles    = ("precio_inmueble", "count"),
    precio_min       = ("precio_inmueble", "min"),
    precio_mediana   = ("precio_inmueble", "median"),
    precio_max       = ("precio_inmueble", "max"),
    precio_m2_medio  = ("precio_m2",       "mean"),
    m2_medio         = ("metros_cuadrados","mean"),
    precio_std         = ("precio_inmueble", "std")
).round(0)

# Añadimos el % sobre el total
resumen_segmentos["pct_total"] = (
    resumen_segmentos["num_inmuebles"] / resumen_segmentos["num_inmuebles"].sum() * 100
).round(2)

# Ordenamos lógicamente de menor a mayor segmento
orden = ["Entrada", "Medio", "Alto", "Lujo"]
resumen_segmentos = resumen_segmentos.reindex(orden)

print(resumen_segmentos.to_string())

In [ ]:
df_inmuebles_4

In [ ]:
df_inmuebles_entrada = df_inmuebles_4[df_inmuebles_4["segmento_precio"] == "Entrada"]
df_inmuebles_entrada

In [ ]:
# Compruebo la distribución del Precio/m2 para el segmento ENTRADA
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 8))
sns.histplot(df_inmuebles_entrada['precio_m2'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios por m2 en segmento ENTRADA', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')

plt.show()

In [ ]:
df_inmuebles_medio = df_inmuebles_4[df_inmuebles_4["segmento_precio"] == "Medio"]
df_inmuebles_medio

In [ ]:
# Compruebo la distribución del Precio/m2 para el segmento MEDIO
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 8))
sns.histplot(df_inmuebles_medio['precio_m2'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios por m2 en segmento MEDIO', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')

plt.show()

Parecía que lo más lógico es hacer una segmentación directamente por el precio, ya que al final es lo que tendremos en cuenta para el cálculo de rentabilidad y para el informe de WhitHosting. Sin embargo, observamos que **dentro de cada segmento de precio, la variabilidad de los precios por m2 es demasiado alta**. Dentro del segmento Entrada el precio/m² va de 2.500 a 13.429 €/m², con una std de 1.514. En Medio va de 2.500 a 15.000 €/m², con una std de 2.186. Es decir, dentro de cada segmento conviven realidades de mercado muy distintas en cuanto al valor real del inmueble por metro.

Es por ello que tendríamos que considerar hacer la segmentación Precio/m2 en su lugar. Vamos a comprar:

**Segmentación por Precio absoluto — la actual**

Pros: Responde directamente a la pregunta de inversión de WhiteHosting: cuánto capital inmoviliza cada activo. Para distribir 300 millones, lo que importa primero es el importe de compra. Es intuitiva para cualquier audiencia y los cortes en 500K, 1M y 1.5M son los más lógicos en una diapositiva. Los segmentos tienen volúmenes equilibrados respecto a los percentiles (51% / 29% / 10% / 9%).

Contras: Mezcla inmuebles de valor intrínseco muy distinto dentro del mismo segmento. Un estudio de 40 m2 en el centro de Madrid a 400.000 € y un piso de 120 m2 en la periferia de Valencia a 380.000 € son ambos "Entrada" pero no son comparables en ningún análisis de rentabilidad. La std interna de precio/m2 es demasiado alta para que las medias por segmento sean etadisticamente representativas.

**Segmentación por Precio/m² — la alternativa**

Los percentiles del dataset limpio sugieren unos cortes naturales: P25 en 4.098, P50 en 5.585, P75 en 7.767, P90 en 10.256. Algo como hasta 4.500 €/m² Entrada, hasta 7.000 €/m² Medio, hasta 10.000 €/m² Alto, por encima Lujo.

Pros: Segmenta por el valor real del inmueble independientemente de su tamaño. Es la métrica más homogénea dentro de cada grupo, lo que hace que las medias y desviaciones por segmento sean mucho más representativas. Es también la métrica más usada en análisis inmobiliario profesional para comparar mercados. Y **para el cálculo de ROI es más limpia** porque el rendimiento por metro en Airbnb está más correlacionado con el precio/m2 de compra que con el precio absoluto.

Contras: Es menos intuitiva para comunicar a una audiencia no técnica. Y los segmentos resultantes tendrían volúmenes menos controlados, dependiendo de dónde pongams los cortes.

**Conclusión - usaremos ambas**

No es una decisión de uno u otro, sino de para qué usas cada una. Lo más sólido analíticamente para vuestro proyecto sería tener las dos columnas con nombres distintos y usarlas para cosas distintas.

**En Power BI las dejamos como dos dimensiones independientes** y podremos cruzarlas para hacer una análisis más rico, lo que además genera visualizacioens interesantes. Aunque no descarto que utilicemos solo la de Precio/m2.

## 26.2 Segmento Precio/m2

In [ ]:
df_inmuebles_5 = df_inmuebles_4.copy()
df_inmuebles_5

In [ ]:
# Segmentación por precio / m2 acorde a lo mencionado:
# Hasta 4.500 €/m² Entrada, hasta 7.000 €/m² Medio, hasta 10.000 €/m² Alto, por encima Lujo.
# Lo voy a llamar "segmento_valor" para
def segmentar_valor(precio_m2):
    if precio_m2 <= 4_500:
        return "Entrada"
    elif precio_m2 <= 7_000:
        return "Medio"
    elif precio_m2 <= 10_000:
        return "Alto"
    else:
        return "Lujo"

df_inmuebles_5["segmento_valor"] = df_inmuebles_5["precio_m2"].apply(segmentar_valor)

In [ ]:
df_inmuebles_5 = df_inmuebles_5.rename(columns={"segmento_valor": "segmento_precio_m2"})

In [ ]:
# Resumen segmento_precio_m2
resumen_valor = df_inmuebles_5.groupby("segmento_precio_m2").agg(
    num_inmuebles  = ("precio_m2",       "count"),
    pm2_min        = ("precio_m2",       "min"),
    pm2_mediana    = ("precio_m2",       "median"),
    pm2_max        = ("precio_m2",       "max"),
    precio_medio   = ("precio_inmueble", "mean"),
    m2_medio       = ("metros_cuadrados","mean"),
    pm2_std        = ("precio_m2",       "std")
).round(0)
resumen_valor["pct_total"] = (resumen_valor["num_inmuebles"] / resumen_valor["num_inmuebles"].sum() * 100).round(2)
resumen_valor = resumen_valor.reindex(["Entrada", "Medio", "Alto", "Lujo"])
print(resumen_valor.to_string())

In [ ]:
df_inmuebles_5

In [ ]:
# Cruce entre ambas segmentaciones
print("\n --- CRUCE segmento_precio vs segmento_precio_m2 --- \n")
cruce = pd.crosstab(
    df_inmuebles_5["segmento_precio"],
    df_inmuebles_5["segmento_precio_m2"],
    margins=True
).reindex(["Entrada", "Medio", "Alto", "Lujo", "All"])
print(cruce.to_string())

In [ ]:
# Hago la comprobación que hice antes pero al revés
df_inmuebles_5_medio = df_inmuebles_5[df_inmuebles_5["segmento_precio_m2"] == "Medio"]
df_inmuebles_5_medio

In [ ]:
# Compruebo la distribución del Precio/m2 para el segmento MEDIO
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 8))
sns.histplot(df_inmuebles_5_medio['precio_inmueble'] / 1000, bins=100, kde=True, color='skyblue')
plt.title('Distribución de Precios en segmento MEDIO por m2', fontsize=15)
plt.xlabel('Precio (en miles de €)')
plt.ylabel('Número de Inmuebles')

plt.show()

In [ ]:
# Sacamos ahora sí el DF FINAL DE INMUEBLES DE IDEALISTA Y DE FOTOCASA
df_inmuebles_5.to_csv('datos_compra_inmuebles_listo (check final Claude).csv', index=False)